# SPCE5025 Final – Trent Douglas

## Problem Description

You are an orbit analyst for UltraHuge Aerospace, Inc. Your task is to plan a two-burn Hohmann Transfer sequence for the **ChaserSat** to rendezvous with the **TargetSat**.

> **Note:** The Chaser vehicle is behind the Target and closing. Manage signs so that times come out positive and phase angle changes are consistent with a closing satellite.

**Reference Frame:** True of Date  
**Epoch:** 01 May 2012 00:00:00


## Initial State Vectors

| Component | Target Vehicle | Chaser Vehicle | Units |
|-----------|---------------|----------------|-------|
| X         | -7202235.416  | -6491589.449   | m     |
| Y         |   783685.330  |  2613001.096   | m     |
| Z         |  2839574.411  |  3352419.284   | m     |
| XD        |  -1594.528305 | -3259.157555   | m/s   |
| YD        |  -6621.500276 | -6217.288683   | m/s   |
| ZD        |  -2197.570004 | -1448.482747   | m/s   |


## Part 1: Initial Computations

1. Compute the Keplerian elements for each satellite.
2. Compute the Keplerian orbit period for each satellite.
3. Compute the semi-major axis difference between the target and chaser orbits.
4. Use the semi-major axis difference to estimate the ΔV required to raise the Chaser to the target orbit.
5. Compute the angular difference (phase angle) between the Chaser and Target at the initial epoch.
6. Using the Keplerian periods from Problem 2, compute the phase angle rate between the two satellites.

In [80]:
from standards import *
#initial conditions:
EPOCH = datetime(year=2012, month=5, day=1, hour=0, minute=0, second=0)
POS_TARGET = Vector3(-7202235.416, 783685.330, 2839574.411)
VEL_TARGET = Vector3(-1594.528305, -6621.500276, -2197.570004)
POS_CHASER = Vector3(-6491589.449, 2613001.096, 3352419.284)
VEL_CHASER = Vector3(-3259.157555, -6217.288683, -1448.482747)

#1. Compute the Keplerian elements for each satellite.
KEP_TARGET = KeplerianElements(POS_TARGET, VEL_TARGET)
KEP_CHASER = KeplerianElements(POS_CHASER, VEL_CHASER)
print("Target Kep elements:")
print(f"SMA:  {KEP_TARGET.a:.4f} m")
print(f"ECC:  {KEP_TARGET.ecc:.6f}")
print(f"INC:  {KEP_TARGET.inc_deg:.4f} deg")
print(f"RAAN: {KEP_TARGET.raan_deg:.4f} deg")
print(f"ARGP: {KEP_TARGET.argp_deg:.4f} deg")
print(f"TA:   {KEP_TARGET.ta_deg:.4f} deg\n")
print("Chaser Kep elements:")
print(f"SMA:  {KEP_CHASER.a:.4f} m")
print(f"ECC:  {KEP_CHASER.ecc:.6f}")
print(f"INC:  {KEP_CHASER.inc_deg:.4f} deg")
print(f"RAAN: {KEP_CHASER.raan_deg:.4f} deg")
print(f"ARGP: {KEP_CHASER.argp_deg:.4f} deg")
print(f"TA:   {KEP_CHASER.ta_deg:.4f} deg\n")

#2. Compute the Keplerian orbit period for each satellite.
TP_TARGET = 2*pi*sqrt((KEP_TARGET.a**3)/KeplerianElements.mu_earth)
TP_CHASER = 2*pi*sqrt((KEP_CHASER.a**3)/KeplerianElements.mu_earth)
print(f"Target Period {TP_TARGET:.4f} s")
print(f"Chaser Period {TP_CHASER:.4f} s\n")

#3. Compute the semi-major axis difference between the target and chaser orbits.
SMA_DIFF = KEP_TARGET.a - KEP_CHASER.a
print(f"SMA diff between target and chaser: {SMA_DIFF:.4f} m\n")

#4. Use the semi-major axis difference to estimate the ΔV required to raise the Chaser to the target orbit.
DELTA_V_TO_RAISE_CHASER_TO_TARGET = SMA_DIFF*(pi/(TP_CHASER))
print(f"Delta v to raise chaser to target: {DELTA_V_TO_RAISE_CHASER_TO_TARGET:.4f} m/s\n")

#5. Compute the angular difference (phase angle) between the Chaser and Target at the initial epoch.
PHASE_DIFF = (KEP_TARGET.argp_deg + KEP_TARGET.ta_deg) - (KEP_CHASER.argp_deg + KEP_CHASER.ta_deg)
print(f"Anglular separation between target and chaser: {PHASE_DIFF:.4f} deg\n")

#6. Using the Keplerian periods from Problem 2, compute the phase angle rate between the two satellites.
PHASE_RATE_RAD_PER_SEC = 2*pi*(1/(TP_TARGET) - 1/(TP_CHASER))
PHASE_RATE_DEG_PER_DAY = degrees(PHASE_RATE_RAD_PER_SEC)*60*60*24
print(f"Phase Angle Rate: {PHASE_RATE_DEG_PER_DAY:.4f} deg/day")

Target Kep elements:
SMA:  7780000.0008 m
ECC:  0.001000
INC:  28.5000 deg
RAAN: 40.0000 deg
ARGP: 30.0000 deg
TA:   100.1128 deg

Chaser Kep elements:
SMA:  7759999.9996 m
ECC:  0.001000
INC:  28.5000 deg
RAAN: 40.0000 deg
ARGP: 30.0000 deg
TA:   85.1142 deg

Target Period 6829.3658 s
Chaser Period 6803.0484 s

SMA diff between target and chaser: 20000.0012 m

Delta v to raise chaser to target: 9.2358 m/s

Anglular separation between target and chaser: 14.9987 deg

Phase Angle Rate: -17.6187 deg/day


## Part 2: Hohmann Transfer Burn Planning

> **Assume e = 0 for the initial and final orbits (circular).**

7. Compute the semi-major axis of the intermediate orbit.
8. Compute the ΔV for each of the two burns in the Hohmann transfer sequence.
9. Compute the period of the intermediate orbit.
10. Compute the phase angle rate between the intermediate orbit and the target orbit.
11. How much will the phase angle change in the half-orbit between the burns? (Use absolute value.)
12. Subtract the answer in Problem 11 from the initial phase angle (Problem 5). This is the angular distance the Chaser must travel before the first burn.
13. Using the initial phase angle rate (Problem 6), compute how long it will take the Chaser to travel the distance from Problem 12. (Take absolute value if negative.)
14. State the date/time of the **first burn** (epoch + time from Problem 13).
15. State the date/time of the **second burn**.

In [81]:
# 7. Compute the semi-major axis of the intermediate orbit.
SMA_INT = (KEP_CHASER.a + KEP_TARGET.a)/2
print(f"Intermediate SMA: {SMA_INT:.4f} m\n")

# 8. Compute the ΔV for each of the two burns in the Hohmann transfer sequence.
DELTA_V_BURN_1 = sqrt(KeplerianElements.mu_earth*(2/KEP_CHASER.a - 1/SMA_INT)) - sqrt(KeplerianElements.mu_earth/KEP_CHASER.a)
DELTA_V_BURN_2 = sqrt(KeplerianElements.mu_earth/KEP_TARGET.a) - sqrt(KeplerianElements.mu_earth*(2/KEP_TARGET.a - 1/SMA_INT))
print(f"Delta v for first burn: {DELTA_V_BURN_1:.4f} m/s")
print(f"Delta v for second burn: {DELTA_V_BURN_2:.4f} m/s")
print(f"Total delta v: {DELTA_V_BURN_1 + DELTA_V_BURN_2:.4f} m/s\n")

# 9. Compute the period of the intermediate orbit.
INT_TP = compute_period(SMA_INT, KeplerianElements.mu_earth)
print(f"Intermediate orbit period: {INT_TP:.4f} s\n")

# 10. Compute the phase angle rate between the intermediate orbit and the target orbit.
PHASE_ANGLE_RATE_INT_TO_TARGET_RAD_PER_SEC = 2*pi*(1/(TP_TARGET) - 1/(INT_TP))
PHASE_ANGLE_RATE_INT_TO_TARGET_DEG_PER_DAY = degrees(PHASE_ANGLE_RATE_INT_TO_TARGET_RAD_PER_SEC)*60*60*24
print(f"Phase angle rate between intermediate and target orbits: {PHASE_ANGLE_RATE_INT_TO_TARGET_DEG_PER_DAY:.4f} deg/day\n")

# 11. How much will the phase angle change in the half-orbit between the burns? (Use absolute value.)
HALF_INT_ORBIT_PHASE_CHANGE = degrees(PHASE_ANGLE_RATE_INT_TO_TARGET_RAD_PER_SEC*INT_TP)/2
print(f"Phase angle change between burns: {HALF_INT_ORBIT_PHASE_CHANGE:.4f} deg\n")

# 12. Subtract the answer in Problem 11 from the initial phase angle (Problem 5). This is the angular distance the Chaser must travel before the first burn.
ANGULAR_DISTANCE_CHASER_BEFORE_BURN_1 = PHASE_DIFF+HALF_INT_ORBIT_PHASE_CHANGE
print(f"Angular distance Chaser must travel before first burn: {ANGULAR_DISTANCE_CHASER_BEFORE_BURN_1:.4f} deg\n")

# 13. Using the initial phase angle rate (Problem 6), compute how long it will take the Chaser to travel the distance from Problem 12. (Take absolute value if negative.)
SECONDS_TO_TRAVEL_TO_BURN_1 = abs(1/(PHASE_RATE_RAD_PER_SEC/radians(ANGULAR_DISTANCE_CHASER_BEFORE_BURN_1)))
print(f"Seconds to travel to point of first burn: {SECONDS_TO_TRAVEL_TO_BURN_1:.4f} s\n")

# 14. State the date/time of the **first burn** (epoch + time from Problem 13).
BURN_TIME_1 = EPOCH + timedelta(seconds=SECONDS_TO_TRAVEL_TO_BURN_1)
print(f"Burn 1 time: {str(BURN_TIME_1)}\n")

# 15. State the date/time of the **second burn**.
BURN_TIME_2 = BURN_TIME_1 + timedelta(seconds=(INT_TP/2))
print(f"Burn 2 time: {str(BURN_TIME_2)}\n")

Intermediate SMA: 7770000.0002 m

Delta v for first burn: 4.6105 m/s
Delta v for second burn: 4.6075 m/s
Total delta v: 9.2180 m/s

Intermediate orbit period: 6816.2029 s

Phase angle rate between intermediate and target orbits: -8.7952 deg/day

Phase angle change between burns: -0.3469 deg

Angular distance Chaser must travel before first burn: 14.6517 deg

Seconds to travel to point of first burn: 71850.1180 s

Burn 1 time: 2012-05-01 19:57:30.117992

Burn 2 time: 2012-05-01 20:54:18.219434



## Part 3: Ground Site Computations

| Site | Latitude (deg) | Longitude (deg) | Altitude (m) |
|------|---------------|-----------------|--------------|
| DGSA | -7.2700       | 72.3700         | -68.4        |
| VTSA | 34.8233       | 239.4983        | 269.4        |

16. Create a Chaser ephemeris spanning **5/1/2012 – 5/2/2012** using:
    - RK4 integrator, step size = 60 seconds
    - Central body-only gravity
17. Is the Chaser in view (elevation > 0°) of **DGSA or VTSA** at the **first burn**? Show computed elevations.
18. Is the Chaser in view (elevation > 0°) of **DGSA or VTSA** at the **second burn**? Show computed elevations.

In [82]:
#16
import copy
DGSA_LAT = radians(-7.27)
DGSA_LON = radians(72.37)
DGSA_ALTD = radians(-68.4)
VTSA_LAT = radians(34.8233)
VTSA_LON = radians(239.4983)
VTSA_ALTD = radians(269.4)
R_ECEF_TOPOCENTRIC_DGSA = np.array([
    [-sin(DGSA_LON), cos(DGSA_LON), 0],
    [-sin(DGSA_LAT)*cos(DGSA_LON), -sin(DGSA_LAT)*sin(DGSA_LON), cos(DGSA_LAT)],
    [ cos(DGSA_LAT)*cos(DGSA_LON), cos(DGSA_LAT)*sin(DGSA_LON), sin(DGSA_LAT)]
])
R_ECEF_TOPOCENTRIC_VTSA = np.array([
    [-sin(VTSA_LON), cos(VTSA_LON), 0],
    [-sin(VTSA_LAT)*cos(VTSA_LON), -sin(VTSA_LAT)*sin(VTSA_LON), cos(VTSA_LAT)],
    [ cos(VTSA_LAT)*cos(VTSA_LON), cos(VTSA_LAT)*sin(VTSA_LON), sin(VTSA_LAT)]
])
Step_Size = 60
Step_Num = 1440
start_time = EPOCH
Earth_rotation = 72.921151467e-6 #rad/sec
Earth_gravitational_parameter = 3.986004418e14 #m^3/s^2
Earth_radius = 6378137 #m
f = 1/298.257223563
Earth_Eccentricity = sqrt(2*f-f**2)
class Step:
    def __init__(self, T, V1_RK_X, V1_RK_Y, V1_RK_Z, V1_RK_XD, V1_RK_YD, V1_RK_ZD,
                 V2_RK_X, V2_RK_Y, V2_RK_Z, V2_RK_XD, V2_RK_YD, V2_RK_ZD, V1_AZ_DGSA, 
                 V1_EL_DGSA, V2_AZ_DGSA, V2_EL_DGSA, V1_AZ_VTSA, V1_EL_VTSA, V2_AZ_VTSA, V2_EL_VTSA, SepA):
        self.T = T
        self.V1_RK_X = V1_RK_X
        self.V1_RK_Y = V1_RK_Y
        self.V1_RK_Z = V1_RK_Z
        self.V1_RK_XD = V1_RK_XD
        self.V1_RK_YD = V1_RK_YD
        self.V1_RK_ZD = V1_RK_ZD
        self.V2_RK_X = V2_RK_X
        self.V2_RK_Y = V2_RK_Y
        self.V2_RK_Z = V2_RK_Z
        self.V2_RK_XD = V2_RK_XD
        self.V2_RK_YD = V2_RK_YD
        self.V2_RK_ZD = V2_RK_ZD
        self.V1_AZ_DGSA = V1_AZ_DGSA
        self.V1_EL_DGSA = V1_EL_DGSA
        self.V2_AZ_DGSA = V2_AZ_DGSA
        self.V2_EL_DGSA = V2_EL_DGSA
        self.V1_AZ_VTSA = V1_AZ_VTSA
        self.V1_EL_VTSA = V1_EL_VTSA
        self.V2_AZ_VTSA = V2_AZ_VTSA
        self.V2_EL_VTSA = V2_EL_VTSA
        self.SepA = SepA

steps_central_body:list[Step] = []
V1_y_0 = Vector6(POS_TARGET.x, POS_TARGET.y, POS_TARGET.z, VEL_TARGET.x, VEL_TARGET.y, VEL_TARGET.z)
V2_y_0 = Vector6(POS_CHASER.x, POS_CHASER.y, POS_CHASER.z, VEL_CHASER.x, VEL_CHASER.y, VEL_CHASER.z)
h = Step_Size
w_ = Vector3(0, 0, 72.921151467e-6)
V1_v_r_ = VEL_TARGET - (w_.cross(POS_TARGET))
V2_v_r_ = VEL_CHASER - (w_.cross(POS_CHASER))
V1_temp_pos = copy.deepcopy(POS_TARGET)
V2_temp_pos = copy.deepcopy(POS_CHASER)
V1_temp_vel = copy.deepcopy(VEL_TARGET)
V2_temp_vel = copy.deepcopy(VEL_CHASER)
timestamp = start_time

# calculate ECEF position vectors
V1_gah, V1_pos_ECEF_np = rotate_TOD_to_ECEF(start_time, POS_TARGET, Earth_rotation)
V2_gah, V2_pos_ECEF_np = rotate_TOD_to_ECEF(start_time, POS_CHASER, Earth_rotation)
V1_pos_ECEF = Vector3(V1_pos_ECEF_np[0][0], V1_pos_ECEF_np[1][0], V1_pos_ECEF_np[2][0])
V2_pos_ECEF = Vector3(V2_pos_ECEF_np[0][0], V2_pos_ECEF_np[1][0], V2_pos_ECEF_np[2][0])

# calculate az/el DGSA:
V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V1_pos_ECEF_np
V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V2_pos_ECEF_np
Sen_ECEF = convert_lat_lon_h_to_ecef(DGSA_LAT, DGSA_LON, DGSA_ALTD, Earth_radius)
Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_DGSA @ Sen_ECEF.get_np_vector()
sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
V1_Az_DGSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
V1_El_DGSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
V2_Az_DGSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
V2_El_DGSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

# calculate az/el VTSA:
V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V1_pos_ECEF_np
V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V2_pos_ECEF_np
Sen_ECEF = convert_lat_lon_h_to_ecef(VTSA_LAT, VTSA_LON, VTSA_ALTD, Earth_radius)
Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_VTSA @ Sen_ECEF.get_np_vector()
sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
V1_Az_VTSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
V1_El_VTSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
V2_Az_VTSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
V2_El_VTSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

# calculate SepA
sen_to_v1_vector3 = Vector3(sen_to_v1[0][0], sen_to_v1[1][0], sen_to_v1[2][0])
sen_to_v2_vector3 = Vector3(sen_to_v2[0][0], sen_to_v2[1][0], sen_to_v2[2][0])
SepA = degrees(acos(sen_to_v1_vector3.dot(sen_to_v2_vector3)/(sen_to_v1_vector3.magnitude()*sen_to_v2_vector3.magnitude())))

steps_central_body.append(Step(0, V1_y_0.x, V1_y_0.y, V1_y_0.z, V1_temp_vel.x, V1_temp_vel.y, V1_temp_vel.z,
V2_y_0.x, V2_y_0.y, V2_y_0.z, V2_temp_vel.x, V2_temp_vel.y, V2_temp_vel.z, V1_Az_DGSA, V1_El_DGSA, V2_Az_DGSA, V2_El_DGSA,
V1_Az_VTSA, V1_El_VTSA, V2_Az_VTSA, V2_El_VTSA, SepA))

for i in range(Step_Num):
    # calculate TOD position vectors
    t = h*(i+1)
    timestamp = timestamp + timedelta(seconds=h)
    V1_step = compute_rk_2_body_step(0, Earth_gravitational_parameter, V1_temp_pos, V1_temp_vel, h, V1_y_0, timestamp, 0, 0, 0)
    V2_step = compute_rk_2_body_step(0, Earth_gravitational_parameter, V2_temp_pos, V2_temp_vel, h, V2_y_0, timestamp, 0, 0, 0)
    
     # calculate ECEF position vectors
    V1_gah, V1_pos_ECEF_np = rotate_TOD_to_ECEF(timestamp, Vector3(V1_step.x, V1_step.y, V1_step.z), Earth_rotation)
    V2_gah, V2_pos_ECEF_np = rotate_TOD_to_ECEF(timestamp, Vector3(V2_step.x, V2_step.y, V2_step.z), Earth_rotation)
    SEN_gah, SEN_pos_ECEF_np = rotate_TOD_to_ECEF(timestamp, Vector3(V2_step.x, V2_step.y, V2_step.z), Earth_rotation)
    V1_pos_ECEF = Vector3(V1_pos_ECEF_np[0][0], V1_pos_ECEF_np[1][0], V1_pos_ECEF_np[2][0])
    V2_pos_ECEF = Vector3(V2_pos_ECEF_np[0][0], V2_pos_ECEF_np[1][0], V2_pos_ECEF_np[2][0])
    
    # calculate az/el DGSA:
    V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
    V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
    V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V1_pos_ECEF_np
    V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_DGSA @ V2_pos_ECEF_np
    Sen_ECEF = convert_lat_lon_h_to_ecef(DGSA_LAT, DGSA_LON, DGSA_ALTD, Earth_radius)
    Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_DGSA @ Sen_ECEF.get_np_vector()
    sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    V1_Az_DGSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
    V1_El_DGSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
    V2_Az_DGSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
    V2_El_DGSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

    # calculate az/el VTSA:
    V1_lat, V1_lon, V1_height = calculate_lat_lon_h(Earth_Eccentricity, V1_pos_ECEF, Earth_radius)
    V2_lat, V2_lon, V2_height = calculate_lat_lon_h(Earth_Eccentricity, V2_pos_ECEF, Earth_radius)
    V1_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V1_pos_ECEF_np
    V2_TOCOCENTRIC_Pos_sat = R_ECEF_TOPOCENTRIC_VTSA @ V2_pos_ECEF_np
    Sen_ECEF = convert_lat_lon_h_to_ecef(VTSA_LAT, VTSA_LON, VTSA_ALTD, Earth_radius)
    Sen_TOCOCENTRIC = R_ECEF_TOPOCENTRIC_VTSA @ Sen_ECEF.get_np_vector()
    sen_to_v1 = V1_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    sen_to_v2 = V2_TOCOCENTRIC_Pos_sat - Sen_TOCOCENTRIC
    V1_Az_VTSA = degrees(atan2(sen_to_v1[0][0], sen_to_v1[1][0]))% 360.0
    V1_El_VTSA = degrees(atan(sen_to_v1[2][0]/sqrt(sen_to_v1[0][0]**2 + sen_to_v1[1][0]**2)))
    V2_Az_VTSA = degrees(atan2(sen_to_v2[0][0], sen_to_v2[1][0]))% 360.0
    V2_El_VTSA = degrees(atan(sen_to_v2[2][0]/sqrt(sen_to_v2[0][0]**2 + sen_to_v2[1][0]**2)))

    # calculate sepA
    sen_to_v1_vector3 = Vector3(sen_to_v1[0][0], sen_to_v1[1][0], sen_to_v1[2][0])
    sen_to_v2_vector3 = Vector3(sen_to_v2[0][0], sen_to_v2[1][0], sen_to_v2[2][0])
    SepA = degrees(acos(sen_to_v1_vector3.dot(sen_to_v2_vector3)/(sen_to_v1_vector3.magnitude()*sen_to_v2_vector3.magnitude())))

    # append data to array and move on
    steps_central_body.append(Step(t, V1_step.x, V1_step.y, V1_step.z, V1_step.xd, V1_step.yd, V1_step.zd,
        V2_step.x, V2_step.y, V2_step.z, V2_step.xd, V2_step.yd, V2_step.zd, V1_Az_DGSA, V1_El_DGSA, V2_Az_DGSA, V2_El_DGSA,
        V1_Az_VTSA, V1_El_VTSA, V2_Az_VTSA, V2_El_VTSA, SepA))
    V1_y_0 = V1_step
    V2_y_0 = V2_step
    V1_temp_pos = Vector3(V1_step.x, V1_step.y, V1_step.z)
    V1_temp_vel = Vector3(V1_step.xd, V1_step.yd, V1_step.zd)
    V2_temp_pos = Vector3(V2_step.x, V2_step.y, V2_step.z)
    V2_temp_vel = Vector3(V2_step.xd, V2_step.yd, V2_step.zd)

from IPython.display import Markdown, display

header = "| Time (s) | T (seconds) | CHASER_X (m) | CHASER_Y (m) | CHASER_Z (m) | CHASER_XD (m/s) | CHASER_YD (m/s) | CHASER_ZD (m/s) | CHASER_AZ_DGSA | CHASER_EL_DGSA | CHASER_AZ_VTSA | CHASER_EL_VTSA |"
separator = "|------|-----------------|----------|----------|----------|--------------|--------------|--------------|-----------|-----------|---------|---------|"

rows = []
for i, s in enumerate(steps_central_body):
    row = (
        f"| {EPOCH+timedelta(seconds=s.T)} | {s.T} | {s.V2_RK_X:.2f} | {s.V2_RK_Y:.2f} | {s.V2_RK_Z:.2f} | {s.V2_RK_XD:.2f} | {s.V2_RK_YD:.2f} | {s.V2_RK_ZD:.2f} | {s.V2_AZ_DGSA:.3f} | {s.V2_EL_DGSA:.3f} | {s.V2_AZ_VTSA:.3f} | {s.V2_EL_VTSA:.3f} |"
    )
    rows.append(row)

table_md = "\n".join([header, separator] + rows)

display(Markdown(f"{table_md}"))

# 2012-05-01 19:57:00	71820	7309069.83	-113394.58	-2598067.35	963.14	6672.13	2438.99	118.413	-57.692	221.317	-24.598
# 2012-05-01 19:58:00	71880	7355599.24	286902.08	-2447811.69	587.44	6667.68	2568.26	118.367	-59.358	219.563	-22.937

# 2012-05-01 20:54:00	75240	-7333463.16	-37639.88	2543761.26	-834.22	-6669.11	-2482.72	293.438	-26.995	51.600	-48.890
# 2012-05-01 20:55:00	75300	-7372244.02	-437523.91	2390972.94	-458.15	-6656.95	-2608.92	293.009	-25.085	50.345	-50.441

#17 and 18
print("At the first burn  (2012-05-01 19:57:30.117992), DGSA EL is between -57.692 deg and -59.358 deg and the VTSA EL is between -24.598 deg and -22.937 deg so NEITHER sensor has visibility of chaser")
print("At the second burn (2012-05-01 20:54:18.219434), DGSA EL is between -26.995 deg and -25.085 deg and the VTSA EL is between -48.890 deg and -50.441 deg so NEITHER sensor has visibility of chaser")

| Time (s) | T (seconds) | CHASER_X (m) | CHASER_Y (m) | CHASER_Z (m) | CHASER_XD (m/s) | CHASER_YD (m/s) | CHASER_ZD (m/s) | CHASER_AZ_DGSA | CHASER_EL_DGSA | CHASER_AZ_VTSA | CHASER_EL_VTSA |
|------|-----------------|----------|----------|----------|--------------|--------------|--------------|-----------|-----------|---------|---------|
| 2012-05-01 00:00:00 | 0 | -6491589.45 | 2613001.10 | 3352419.28 | -3259.16 | -6217.29 | -1448.48 | 298.137 | -63.635 | 83.088 | -14.083 |
| 2012-05-01 00:01:00 | 60 | -6677072.06 | 2236142.91 | 3260407.73 | -2922.02 | -6341.43 | -1617.78 | 296.323 | -62.170 | 82.385 | -16.225 |
| 2012-05-01 00:02:00 | 120 | -6842053.91 | 1852419.00 | 3158385.66 | -2575.97 | -6446.09 | -1782.08 | 294.629 | -60.690 | 81.745 | -18.308 |
| 2012-05-01 00:03:00 | 180 | -6986031.94 | 1463008.49 | 3046667.93 | -2222.08 | -6530.94 | -1940.89 | 293.037 | -59.197 | 81.158 | -20.341 |
| 2012-05-01 00:04:00 | 240 | -7108567.73 | 1069107.52 | 2925599.04 | -1861.42 | -6595.73 | -2093.71 | 291.532 | -57.691 | 80.615 | -22.330 |
| 2012-05-01 00:05:00 | 300 | -7209288.78 | 671925.64 | 2795552.07 | -1495.10 | -6640.28 | -2240.08 | 290.101 | -56.174 | 80.111 | -24.280 |
| 2012-05-01 00:06:00 | 360 | -7287889.61 | 272682.07 | 2656927.50 | -1124.26 | -6664.44 | -2379.55 | 288.733 | -54.646 | 79.639 | -26.195 |
| 2012-05-01 00:07:00 | 420 | -7344132.67 | -127398.07 | 2510152.02 | -750.04 | -6668.16 | -2511.71 | 287.417 | -53.108 | 79.194 | -28.081 |
| 2012-05-01 00:08:00 | 480 | -7377849.04 | -527087.48 | 2355677.11 | -373.57 | -6651.42 | -2636.14 | 286.146 | -51.560 | 78.773 | -29.940 |
| 2012-05-01 00:09:00 | 540 | -7388938.83 | -925160.43 | 2193977.74 | 3.99 | -6614.29 | -2752.46 | 284.913 | -50.002 | 78.371 | -31.775 |
| 2012-05-01 00:10:00 | 600 | -7377371.52 | -1320396.54 | 2025550.84 | 381.48 | -6556.88 | -2860.33 | 283.710 | -48.435 | 77.986 | -33.589 |
| 2012-05-01 00:11:00 | 660 | -7343185.99 | -1711584.49 | 1850913.82 | 757.74 | -6479.39 | -2959.41 | 282.531 | -46.859 | 77.614 | -35.383 |
| 2012-05-01 00:12:00 | 720 | -7286490.31 | -2097525.70 | 1670602.90 | 1131.62 | -6382.04 | -3049.41 | 281.371 | -45.272 | 77.252 | -37.160 |
| 2012-05-01 00:13:00 | 780 | -7207461.46 | -2477037.99 | 1485171.56 | 1501.99 | -6265.14 | -3130.05 | 280.226 | -43.676 | 76.898 | -38.921 |
| 2012-05-01 00:14:00 | 840 | -7106344.65 | -2848959.20 | 1295188.75 | 1867.70 | -6129.07 | -3201.09 | 279.089 | -42.070 | 76.549 | -40.668 |
| 2012-05-01 00:15:00 | 900 | -6983452.63 | -3212150.67 | 1101237.22 | 2227.65 | -5974.23 | -3262.31 | 277.956 | -40.453 | 76.203 | -42.402 |
| 2012-05-01 00:16:00 | 960 | -6839164.63 | -3565500.74 | 903911.70 | 2580.72 | -5801.10 | -3313.53 | 276.824 | -38.825 | 75.857 | -44.125 |
| 2012-05-01 00:17:00 | 1020 | -6673925.22 | -3907928.12 | 703817.09 | 2925.85 | -5610.23 | -3354.59 | 275.686 | -37.185 | 75.509 | -45.836 |
| 2012-05-01 00:18:00 | 1080 | -6488242.88 | -4238385.13 | 501566.60 | 3261.98 | -5402.19 | -3385.37 | 274.540 | -35.533 | 75.156 | -47.537 |
| 2012-05-01 00:19:00 | 1140 | -6282688.48 | -4555860.92 | 297779.91 | 3588.08 | -5177.63 | -3405.78 | 273.379 | -33.867 | 74.795 | -49.229 |
| 2012-05-01 00:20:00 | 1200 | -6057893.46 | -4859384.50 | 93081.27 | 3903.17 | -4937.24 | -3415.76 | 272.200 | -32.187 | 74.424 | -50.912 |
| 2012-05-01 00:21:00 | 1260 | -5814547.94 | -5148027.67 | -111902.42 | 4206.28 | -4681.75 | -3415.28 | 270.996 | -30.491 | 74.038 | -52.587 |
| 2012-05-01 00:22:00 | 1320 | -5553398.55 | -5420907.85 | -316543.47 | 4496.48 | -4411.94 | -3404.35 | 269.764 | -28.778 | 73.636 | -54.255 |
| 2012-05-01 00:23:00 | 1380 | -5275246.20 | -5677190.74 | -520215.37 | 4772.90 | -4128.65 | -3382.99 | 268.495 | -27.047 | 73.211 | -55.915 |
| 2012-05-01 00:24:00 | 1440 | -4980943.58 | -5916092.81 | -722294.62 | 5034.69 | -3832.73 | -3351.27 | 267.185 | -25.297 | 72.759 | -57.569 |
| 2012-05-01 00:25:00 | 1500 | -4671392.58 | -6136883.70 | -922162.68 | 5281.05 | -3525.09 | -3309.30 | 265.824 | -23.524 | 72.275 | -59.216 |
| 2012-05-01 00:26:00 | 1560 | -4347541.54 | -6338888.42 | -1119207.84 | 5511.23 | -3206.68 | -3257.20 | 264.406 | -21.727 | 71.751 | -60.857 |
| 2012-05-01 00:27:00 | 1620 | -4010382.38 | -6521489.37 | -1312827.02 | 5724.54 | -2878.47 | -3195.13 | 262.919 | -19.904 | 71.180 | -62.492 |
| 2012-05-01 00:28:00 | 1680 | -3660947.53 | -6684128.21 | -1502427.68 | 5920.32 | -2541.45 | -3123.28 | 261.355 | -18.053 | 70.550 | -64.120 |
| 2012-05-01 00:29:00 | 1740 | -3300306.87 | -6826307.57 | -1687429.55 | 6097.97 | -2196.66 | -3041.87 | 259.698 | -16.169 | 69.850 | -65.742 |
| 2012-05-01 00:30:00 | 1800 | -2929564.41 | -6947592.52 | -1867266.43 | 6256.96 | -1845.14 | -2951.16 | 257.936 | -14.250 | 69.063 | -67.358 |
| 2012-05-01 00:31:00 | 1860 | -2549854.97 | -7047611.90 | -2041387.87 | 6396.79 | -1487.98 | -2851.41 | 256.050 | -12.294 | 68.167 | -68.966 |
| 2012-05-01 00:32:00 | 1920 | -2162340.77 | -7126059.45 | -2209260.89 | 6517.05 | -1126.27 | -2742.93 | 254.020 | -10.295 | 67.136 | -70.566 |
| 2012-05-01 00:33:00 | 1980 | -1768207.83 | -7182694.74 | -2370371.55 | 6617.36 | -761.09 | -2626.05 | 251.822 | -8.252 | 65.933 | -72.157 |
| 2012-05-01 00:34:00 | 2040 | -1368662.43 | -7217343.90 | -2524226.53 | 6697.42 | -393.58 | -2501.14 | 249.427 | -6.160 | 64.507 | -73.737 |
| 2012-05-01 00:35:00 | 2100 | -964927.44 | -7229900.13 | -2670354.63 | 6756.98 | -24.85 | -2368.56 | 246.801 | -4.018 | 62.788 | -75.304 |
| 2012-05-01 00:36:00 | 2160 | -558238.62 | -7220324.09 | -2808308.20 | 6795.86 | 343.97 | -2228.72 | 243.903 | -1.826 | 60.672 | -76.854 |
| 2012-05-01 00:37:00 | 2220 | -149840.83 | -7188643.97 | -2937664.49 | 6813.93 | 711.76 | -2082.05 | 240.686 | 0.415 | 58.008 | -78.380 |
| 2012-05-01 00:38:00 | 2280 | 259015.69 | -7134955.46 | -3058026.98 | 6811.14 | 1077.40 | -1929.00 | 237.094 | 2.696 | 54.561 | -79.872 |
| 2012-05-01 00:39:00 | 2340 | 667079.15 | -7059421.45 | -3169026.52 | 6787.50 | 1439.76 | -1770.04 | 233.065 | 5.003 | 49.958 | -81.316 |
| 2012-05-01 00:40:00 | 2400 | 1073100.01 | -6962271.58 | -3270322.54 | 6743.07 | 1797.75 | -1605.63 | 228.530 | 7.309 | 43.593 | -82.681 |
| 2012-05-01 00:41:00 | 2460 | 1475834.75 | -6843801.52 | -3361604.02 | 6677.99 | 2150.25 | -1436.30 | 223.420 | 9.570 | 34.500 | -83.917 |
| 2012-05-01 00:42:00 | 2520 | 1874049.69 | -6704372.14 | -3442590.49 | 6592.45 | 2496.21 | -1262.55 | 217.677 | 11.721 | 21.331 | -84.928 |
| 2012-05-01 00:43:00 | 2580 | 2266524.71 | -6544408.42 | -3513032.89 | 6486.71 | 2834.56 | -1084.92 | 211.267 | 13.673 | 3.163 | -85.557 |
| 2012-05-01 00:44:00 | 2640 | 2652056.99 | -6364398.15 | -3572714.31 | 6361.08 | 3164.26 | -903.95 | 204.205 | 15.314 | 341.909 | -85.638 |
| 2012-05-01 00:45:00 | 2700 | 3029464.65 | -6164890.54 | -3621450.71 | 6215.96 | 3484.30 | -720.18 | 196.574 | 16.521 | 322.545 | -85.142 |
| 2012-05-01 00:46:00 | 2760 | 3397590.41 | -5946494.50 | -3659091.44 | 6051.77 | 3793.71 | -534.19 | 188.543 | 17.186 | 308.060 | -84.216 |
| 2012-05-01 00:47:00 | 2820 | 3755305.05 | -5709876.90 | -3685519.78 | 5869.01 | 4091.53 | -346.53 | 180.350 | 17.239 | 298.018 | -83.028 |
| 2012-05-01 00:48:00 | 2880 | 4101510.92 | -5455760.47 | -3700653.24 | 5668.24 | 4376.86 | -157.79 | 172.268 | 16.674 | 291.042 | -81.690 |
| 2012-05-01 00:49:00 | 2940 | 4435145.31 | -5184921.71 | -3704443.89 | 5450.06 | 4648.80 | 31.47 | 164.546 | 15.550 | 286.049 | -80.263 |
| 2012-05-01 00:50:00 | 3000 | 4755183.65 | -4898188.50 | -3696878.51 | 5215.15 | 4906.54 | 220.65 | 157.366 | 13.971 | 282.350 | -78.780 |
| 2012-05-01 00:51:00 | 3060 | 5060642.73 | -4596437.62 | -3677978.62 | 4964.21 | 5149.26 | 409.19 | 150.828 | 12.060 | 279.520 | -77.259 |
| 2012-05-01 00:52:00 | 3120 | 5350583.68 | -4280592.11 | -3647800.47 | 4698.01 | 5376.24 | 596.50 | 144.957 | 9.931 | 277.295 | -75.713 |
| 2012-05-01 00:53:00 | 3180 | 5624114.89 | -3951618.47 | -3606434.86 | 4417.36 | 5586.75 | 782.01 | 139.727 | 7.679 | 275.505 | -74.148 |
| 2012-05-01 00:54:00 | 3240 | 5880394.73 | -3610523.71 | -3554006.93 | 4123.11 | 5780.17 | 965.15 | 135.082 | 5.373 | 274.034 | -72.568 |
| 2012-05-01 00:55:00 | 3300 | 6118634.22 | -3258352.33 | -3490675.75 | 3816.17 | 5955.88 | 1145.36 | 130.954 | 3.060 | 272.806 | -70.977 |
| 2012-05-01 00:56:00 | 3360 | 6338099.41 | -2896183.13 | -3416633.86 | 3497.46 | 6113.34 | 1322.08 | 127.275 | 0.770 | 271.765 | -69.376 |
| 2012-05-01 00:57:00 | 3420 | 6538113.72 | -2525125.92 | -3332106.73 | 3167.97 | 6252.07 | 1494.77 | 123.981 | -1.481 | 270.871 | -67.766 |
| 2012-05-01 00:58:00 | 3480 | 6718060.00 | -2146318.15 | -3237352.09 | 2828.70 | 6371.63 | 1662.91 | 121.014 | -3.684 | 270.095 | -66.148 |
| 2012-05-01 00:59:00 | 3540 | 6877382.51 | -1760921.44 | -3132659.11 | 2480.68 | 6471.64 | 1825.97 | 118.326 | -5.837 | 269.415 | -64.524 |
| 2012-05-01 01:00:00 | 3600 | 7015588.65 | -1370118.01 | -3018347.61 | 2125.00 | 6551.81 | 1983.45 | 115.874 | -7.938 | 268.812 | -62.892 |
| 2012-05-01 01:01:00 | 3660 | 7132250.48 | -975107.10 | -2894767.02 | 1762.73 | 6611.86 | 2134.86 | 113.624 | -9.989 | 268.274 | -61.254 |
| 2012-05-01 01:02:00 | 3720 | 7227006.15 | -577101.26 | -2762295.39 | 1394.98 | 6651.61 | 2279.74 | 111.546 | -11.995 | 267.790 | -59.610 |
| 2012-05-01 01:03:00 | 3780 | 7299560.99 | -177322.64 | -2621338.20 | 1022.89 | 6670.93 | 2417.64 | 109.614 | -13.958 | 267.351 | -57.959 |
| 2012-05-01 01:04:00 | 3840 | 7349688.49 | 223000.76 | -2472327.17 | 647.59 | 6669.76 | 2548.13 | 107.808 | -15.882 | 266.950 | -56.300 |
| 2012-05-01 01:05:00 | 3900 | 7377231.04 | 622638.87 | -2315718.91 | 270.25 | 6648.10 | 2670.81 | 106.110 | -17.770 | 266.582 | -54.635 |
| 2012-05-01 01:06:00 | 3960 | 7382100.46 | 1020363.33 | -2151993.54 | -107.99 | 6605.99 | 2785.30 | 104.504 | -19.625 | 266.240 | -52.963 |
| 2012-05-01 01:07:00 | 4020 | 7364278.32 | 1414951.30 | -1981653.26 | -485.94 | 6543.57 | 2891.25 | 102.977 | -21.450 | 265.922 | -51.282 |
| 2012-05-01 01:08:00 | 4080 | 7323816.01 | 1805189.22 | -1805220.75 | -862.46 | 6461.02 | 2988.33 | 101.518 | -23.248 | 265.623 | -49.593 |
| 2012-05-01 01:09:00 | 4140 | 7260834.67 | 2189876.58 | -1623237.61 | -1236.39 | 6358.60 | 3076.23 | 100.117 | -25.021 | 265.340 | -47.895 |
| 2012-05-01 01:10:00 | 4200 | 7175524.81 | 2567829.60 | -1436262.71 | -1606.56 | 6236.60 | 3154.68 | 98.765 | -26.772 | 265.071 | -46.187 |
| 2012-05-01 01:11:00 | 4260 | 7068145.81 | 2937884.94 | -1244870.43 | -1971.83 | 6095.41 | 3223.43 | 97.455 | -28.502 | 264.813 | -44.469 |
| 2012-05-01 01:12:00 | 4320 | 6939025.09 | 3298903.28 | -1049648.92 | -2331.09 | 5935.45 | 3282.28 | 96.180 | -30.212 | 264.563 | -42.739 |
| 2012-05-01 01:13:00 | 4380 | 6788557.20 | 3649772.87 | -851198.29 | -2683.22 | 5757.20 | 3331.04 | 94.934 | -31.906 | 264.320 | -40.997 |
| 2012-05-01 01:14:00 | 4440 | 6617202.58 | 3989412.97 | -650128.74 | -3027.14 | 5561.23 | 3369.56 | 93.710 | -33.583 | 264.081 | -39.242 |
| 2012-05-01 01:15:00 | 4500 | 6425486.21 | 4316777.25 | -447058.70 | -3361.77 | 5348.11 | 3397.71 | 92.504 | -35.245 | 263.845 | -37.471 |
| 2012-05-01 01:16:00 | 4560 | 6213995.96 | 4630857.00 | -242612.88 | -3686.10 | 5118.52 | 3415.40 | 91.310 | -36.892 | 263.609 | -35.685 |
| 2012-05-01 01:17:00 | 4620 | 5983380.84 | 4930684.30 | -37420.38 | -3999.11 | 4873.15 | 3422.59 | 90.123 | -38.527 | 263.372 | -33.880 |
| 2012-05-01 01:18:00 | 4680 | 5734348.98 | 5215335.02 | 167887.30 | -4299.83 | 4612.77 | 3419.24 | 88.940 | -40.149 | 263.132 | -32.055 |
| 2012-05-01 01:19:00 | 4740 | 5467665.48 | 5483931.72 | 372678.16 | -4587.34 | 4338.16 | 3405.37 | 87.755 | -41.759 | 262.885 | -30.207 |
| 2012-05-01 01:20:00 | 4800 | 5184150.03 | 5735646.37 | 576321.75 | -4860.75 | 4050.18 | 3381.01 | 86.563 | -43.357 | 262.631 | -28.335 |
| 2012-05-01 01:21:00 | 4860 | 4884674.40 | 5969702.97 | 778191.04 | -5119.21 | 3749.71 | 3346.24 | 85.360 | -44.945 | 262.367 | -26.434 |
| 2012-05-01 01:22:00 | 4920 | 4570159.74 | 6185379.91 | 977664.41 | -5361.92 | 3437.68 | 3301.17 | 84.141 | -46.522 | 262.089 | -24.501 |
| 2012-05-01 01:23:00 | 4980 | 4241573.73 | 6382012.28 | 1174127.59 | -5588.14 | 3115.05 | 3245.93 | 82.900 | -48.088 | 261.795 | -22.533 |
| 2012-05-01 01:24:00 | 5040 | 3899927.57 | 6558993.92 | 1366975.52 | -5797.15 | 2782.82 | 3180.69 | 81.632 | -49.644 | 261.481 | -20.523 |
| 2012-05-01 01:25:00 | 5100 | 3546272.88 | 6715779.31 | 1555614.31 | -5988.32 | 2442.02 | 3105.66 | 80.331 | -51.189 | 261.143 | -18.466 |
| 2012-05-01 01:26:00 | 5160 | 3181698.42 | 6851885.27 | 1739463.00 | -6161.05 | 2093.69 | 3021.06 | 78.989 | -52.724 | 260.775 | -16.355 |
| 2012-05-01 01:27:00 | 5220 | 2807326.67 | 6966892.50 | 1917955.44 | -6314.81 | 1738.91 | 2927.16 | 77.599 | -54.247 | 260.372 | -14.182 |
| 2012-05-01 01:28:00 | 5280 | 2424310.43 | 7060446.84 | 2090541.98 | -6449.12 | 1378.77 | 2824.25 | 76.153 | -55.759 | 259.926 | -11.934 |
| 2012-05-01 01:29:00 | 5340 | 2033829.16 | 7132260.37 | 2256691.25 | -6563.58 | 1014.40 | 2712.64 | 74.641 | -57.259 | 259.428 | -9.600 |
| 2012-05-01 01:30:00 | 5400 | 1637085.35 | 7182112.37 | 2415891.75 | -6657.82 | 646.91 | 2592.68 | 73.052 | -58.746 | 258.866 | -7.163 |
| 2012-05-01 01:31:00 | 5460 | 1235300.78 | 7209849.90 | 2567653.48 | -6731.56 | 277.44 | 2464.74 | 71.372 | -60.219 | 258.225 | -4.604 |
| 2012-05-01 01:32:00 | 5520 | 829712.72 | 7215388.36 | 2711509.42 | -6784.57 | -92.87 | 2329.22 | 69.587 | -61.677 | 257.485 | -1.895 |
| 2012-05-01 01:33:00 | 5580 | 421570.08 | 7198711.69 | 2847017.02 | -6816.69 | -462.87 | 2186.54 | 67.679 | -63.117 | 256.620 | 0.994 |
| 2012-05-01 01:34:00 | 5640 | 12129.54 | 7159872.40 | 2973759.54 | -6827.82 | -831.43 | 2037.13 | 65.628 | -64.538 | 255.594 | 4.107 |
| 2012-05-01 01:35:00 | 5700 | -397348.34 | 7098991.44 | 3091347.35 | -6817.94 | -1197.41 | 1881.46 | 63.407 | -65.938 | 254.357 | 7.498 |
| 2012-05-01 01:36:00 | 5760 | -805603.10 | 7016257.78 | 3199419.11 | -6787.06 | -1559.67 | 1720.01 | 60.989 | -67.311 | 252.835 | 11.239 |
| 2012-05-01 01:37:00 | 5820 | -1211378.19 | 6911927.79 | 3297642.94 | -6735.30 | -1917.10 | 1553.28 | 58.337 | -68.656 | 250.920 | 15.421 |
| 2012-05-01 01:38:00 | 5880 | -1613424.95 | 6786324.45 | 3385717.38 | -6662.82 | -2268.60 | 1381.78 | 55.410 | -69.965 | 248.441 | 20.164 |
| 2012-05-01 01:39:00 | 5940 | -2010506.45 | 6639836.31 | 3463372.34 | -6569.84 | -2613.08 | 1206.05 | 52.159 | -71.233 | 245.121 | 25.605 |
| 2012-05-01 01:40:00 | 6000 | -2401401.30 | 6472916.27 | 3530369.92 | -6456.65 | -2949.49 | 1026.63 | 48.526 | -72.450 | 240.484 | 31.882 |
| 2012-05-01 01:41:00 | 6060 | -2784907.46 | 6286080.13 | 3586505.15 | -6323.61 | -3276.78 | 844.07 | 44.446 | -73.607 | 233.672 | 39.039 |
| 2012-05-01 01:42:00 | 6120 | -3159845.94 | 6079905.00 | 3631606.58 | -6171.14 | -3593.96 | 658.93 | 39.848 | -74.689 | 223.117 | 46.765 |
| 2012-05-01 01:43:00 | 6180 | -3525064.44 | 5855027.47 | 3665536.82 | -5999.70 | -3900.03 | 471.79 | 34.661 | -75.681 | 206.312 | 53.832 |
| 2012-05-01 01:44:00 | 6240 | -3879440.91 | 5612141.57 | 3688192.96 | -5809.83 | -4194.08 | 283.22 | 28.822 | -76.563 | 181.918 | 57.567 |
| 2012-05-01 01:45:00 | 6300 | -4221886.98 | 5351996.68 | 3699506.82 | -5602.12 | -4475.19 | 93.81 | 22.299 | -77.312 | 155.883 | 55.624 |
| 2012-05-01 01:46:00 | 6360 | -4551351.35 | 5075395.10 | 3699445.20 | -5377.22 | -4742.50 | -95.86 | 15.107 | -77.904 | 136.468 | 49.328 |
| 2012-05-01 01:47:00 | 6420 | -4866822.96 | 4783189.59 | 3688009.92 | -5135.82 | -4995.19 | -285.21 | 7.341 | -78.314 | 124.141 | 41.637 |
| 2012-05-01 01:48:00 | 6480 | -5167334.15 | 4476280.70 | 3665237.83 | -4878.66 | -5232.48 | -473.66 | 359.181 | -78.525 | 116.305 | 34.244 |
| 2012-05-01 01:49:00 | 6540 | -5451963.57 | 4155613.94 | 3631200.63 | -4606.56 | -5453.66 | -660.62 | 350.881 | -78.524 | 111.068 | 27.684 |
| 2012-05-01 01:50:00 | 6600 | -5719839.02 | 3822176.88 | 3586004.68 | -4320.34 | -5658.05 | -845.52 | 342.724 | -78.311 | 107.380 | 21.989 |
| 2012-05-01 01:51:00 | 6660 | -5970140.12 | 3476996.03 | 3529790.61 | -4020.90 | -5845.02 | -1027.80 | 334.965 | -77.899 | 104.663 | 17.039 |
| 2012-05-01 01:52:00 | 6720 | -6202100.75 | 3121133.70 | 3462732.90 | -3709.15 | -6014.01 | -1206.88 | 327.783 | -77.305 | 102.588 | 12.692 |
| 2012-05-01 01:53:00 | 6780 | -6415011.43 | 2755684.66 | 3385039.28 | -3386.06 | -6164.50 | -1382.24 | 321.271 | -76.554 | 100.955 | 8.821 |
| 2012-05-01 01:54:00 | 6840 | -6608221.44 | 2381772.81 | 3296950.09 | -3052.63 | -6296.03 | -1553.31 | 315.445 | -75.671 | 99.639 | 5.327 |
| 2012-05-01 01:55:00 | 6900 | -6781140.79 | 2000547.65 | 3198737.55 | -2709.88 | -6408.22 | -1719.60 | 310.272 | -74.677 | 98.557 | 2.132 |
| 2012-05-01 01:56:00 | 6960 | -6933241.99 | 1613180.75 | 3090704.84 | -2358.87 | -6500.71 | -1880.57 | 305.689 | -73.593 | 97.651 | -0.823 |
| 2012-05-01 01:57:00 | 7020 | -7064061.60 | 1220862.16 | 2973185.20 | -2000.68 | -6573.23 | -2035.75 | 301.626 | -72.434 | 96.882 | -3.584 |
| 2012-05-01 01:58:00 | 7080 | -7173201.67 | 824796.71 | 2846540.87 | -1636.40 | -6625.57 | -2184.65 | 298.011 | -71.215 | 96.220 | -6.187 |
| 2012-05-01 01:59:00 | 7140 | -7260330.87 | 426200.36 | 2711161.95 | -1267.17 | -6657.58 | -2326.82 | 294.779 | -69.945 | 95.644 | -8.660 |
| 2012-05-01 02:00:00 | 7200 | -7325185.46 | 26296.39 | 2567465.20 | -894.11 | -6669.15 | -2461.84 | 291.872 | -68.634 | 95.137 | -11.024 |
| 2012-05-01 02:01:00 | 7260 | -7367570.09 | -373688.27 | 2415892.76 | -518.36 | -6660.27 | -2589.28 | 289.243 | -67.287 | 94.687 | -13.296 |
| 2012-05-01 02:02:00 | 7320 | -7387358.31 | -772526.85 | 2256910.74 | -141.09 | -6630.96 | -2708.76 | 286.848 | -65.911 | 94.284 | -15.491 |
| 2012-05-01 02:03:00 | 7380 | -7384492.94 | -1168996.49 | 2091007.81 | 236.56 | -6581.32 | -2819.92 | 284.652 | -64.509 | 93.919 | -17.619 |
| 2012-05-01 02:04:00 | 7440 | -7358986.20 | -1561881.94 | 1918693.69 | 613.43 | -6511.52 | -2922.41 | 282.628 | -63.084 | 93.587 | -19.691 |
| 2012-05-01 02:05:00 | 7500 | -7310919.64 | -1949979.28 | 1740497.58 | 988.37 | -6421.76 | -3015.94 | 280.749 | -61.641 | 93.281 | -21.713 |
| 2012-05-01 02:06:00 | 7560 | -7240443.80 | -2332099.62 | 1556966.52 | 1360.22 | -6312.33 | -3100.20 | 278.994 | -60.179 | 92.998 | -23.693 |
| 2012-05-01 02:07:00 | 7620 | -7147777.79 | -2707072.67 | 1368663.74 | 1727.85 | -6183.58 | -3174.95 | 277.347 | -58.702 | 92.734 | -25.635 |
| 2012-05-01 02:08:00 | 7680 | -7033208.51 | -3073750.35 | 1176166.88 | 2090.14 | -6035.89 | -3239.97 | 275.793 | -57.211 | 92.485 | -27.544 |
| 2012-05-01 02:09:00 | 7740 | -6897089.78 | -3431010.21 | 980066.29 | 2445.98 | -5869.73 | -3295.05 | 274.317 | -55.707 | 92.249 | -29.423 |
| 2012-05-01 02:10:00 | 7800 | -6739841.23 | -3777758.90 | 780963.17 | 2794.29 | -5685.61 | -3340.03 | 272.910 | -54.190 | 92.022 | -31.277 |
| 2012-05-01 02:11:00 | 7860 | -6561946.98 | -4112935.47 | 579467.76 | 3134.00 | -5484.09 | -3374.77 | 271.562 | -52.661 | 91.804 | -33.108 |
| 2012-05-01 02:12:00 | 7920 | -6363954.11 | -4435514.53 | 376197.50 | 3464.07 | -5265.80 | -3399.17 | 270.264 | -51.120 | 91.591 | -34.917 |
| 2012-05-01 02:13:00 | 7980 | -6146471.04 | -4744509.43 | 171775.08 | 3783.51 | -5031.40 | -3413.17 | 269.008 | -49.568 | 91.382 | -36.708 |
| 2012-05-01 02:14:00 | 8040 | -5910165.57 | -5038975.21 | -33173.37 | 4091.33 | -4781.62 | -3416.71 | 267.789 | -48.006 | 91.174 | -38.483 |
| 2012-05-01 02:15:00 | 8100 | -5655762.91 | -5318011.43 | -238020.26 | 4386.59 | -4517.22 | -3409.78 | 266.599 | -46.432 | 90.967 | -40.242 |
| 2012-05-01 02:16:00 | 8160 | -5384043.39 | -5580764.94 | -442138.38 | 4668.41 | -4239.00 | -3392.42 | 265.434 | -44.846 | 90.758 | -41.987 |
| 2012-05-01 02:17:00 | 8220 | -5095840.10 | -5826432.42 | -644902.85 | 4935.91 | -3947.83 | -3364.67 | 264.289 | -43.250 | 90.546 | -43.720 |
| 2012-05-01 02:18:00 | 8280 | -4792036.36 | -6054262.82 | -845693.03 | 5188.29 | -3644.58 | -3326.63 | 263.157 | -41.641 | 90.328 | -45.441 |
| 2012-05-01 02:19:00 | 8340 | -4473563.00 | -6263559.64 | -1043894.33 | 5424.78 | -3330.20 | -3278.40 | 262.036 | -40.021 | 90.102 | -47.151 |
| 2012-05-01 02:20:00 | 8400 | -4141395.56 | -6453682.99 | -1238900.16 | 5644.65 | -3005.63 | -3220.14 | 260.920 | -38.387 | 89.867 | -48.852 |
| 2012-05-01 02:21:00 | 8460 | -3796551.26 | -6624051.57 | -1430113.70 | 5847.23 | -2671.87 | -3152.02 | 259.805 | -36.740 | 89.620 | -50.544 |
| 2012-05-01 02:22:00 | 8520 | -3440085.99 | -6774144.39 | -1616949.78 | 6031.91 | -2329.95 | -3074.26 | 258.686 | -35.078 | 89.358 | -52.228 |
| 2012-05-01 02:23:00 | 8580 | -3073091.02 | -6903502.39 | -1798836.56 | 6198.13 | -1980.89 | -2987.09 | 257.559 | -33.401 | 89.078 | -53.904 |
| 2012-05-01 02:24:00 | 8640 | -2696689.77 | -7011729.76 | -1975217.37 | 6345.38 | -1625.77 | -2890.77 | 256.418 | -31.707 | 88.778 | -55.573 |
| 2012-05-01 02:25:00 | 8700 | -2312034.33 | -7098495.19 | -2145552.31 | 6473.20 | -1265.67 | -2785.61 | 255.260 | -29.995 | 88.452 | -57.235 |
| 2012-05-01 02:26:00 | 8760 | -1920301.99 | -7163532.86 | -2309319.93 | 6581.21 | -901.70 | -2671.92 | 254.078 | -28.264 | 88.096 | -58.890 |
| 2012-05-01 02:27:00 | 8820 | -1522691.70 | -7206643.28 | -2466018.83 | 6669.08 | -534.95 | -2550.05 | 252.867 | -26.511 | 87.705 | -60.539 |
| 2012-05-01 02:28:00 | 8880 | -1120420.38 | -7227693.82 | -2615169.12 | 6736.54 | -166.55 | -2420.36 | 251.620 | -24.735 | 87.271 | -62.183 |
| 2012-05-01 02:29:00 | 8940 | -714719.28 | -7226619.21 | -2756313.95 | 6783.38 | 202.37 | -2283.26 | 250.330 | -22.934 | 86.786 | -63.820 |
| 2012-05-01 02:30:00 | 9000 | -306830.22 | -7203421.70 | -2889020.85 | 6809.45 | 570.69 | -2139.17 | 248.989 | -21.104 | 86.239 | -65.452 |
| 2012-05-01 02:31:00 | 9060 | 101998.18 | -7158171.07 | -3012883.08 | 6814.68 | 937.28 | -1988.52 | 247.586 | -19.242 | 85.617 | -67.078 |
| 2012-05-01 02:32:00 | 9120 | 510514.29 | -7091004.45 | -3127520.83 | 6799.05 | 1301.04 | -1831.76 | 246.112 | -17.346 | 84.902 | -68.697 |
| 2012-05-01 02:33:00 | 9180 | 917467.23 | -7002125.90 | -3232582.43 | 6762.59 | 1660.83 | -1669.39 | 244.553 | -15.412 | 84.071 | -70.310 |
| 2012-05-01 02:34:00 | 9240 | 1321610.74 | -6891805.83 | -3327745.38 | 6705.42 | 2015.57 | -1501.89 | 242.894 | -13.434 | 83.093 | -71.916 |
| 2012-05-01 02:35:00 | 9300 | 1721706.91 | -6760380.18 | -3412717.39 | 6627.71 | 2364.17 | -1329.78 | 241.118 | -11.409 | 81.925 | -73.513 |
| 2012-05-01 02:36:00 | 9360 | 2116529.96 | -6608249.43 | -3487237.22 | 6529.70 | 2705.57 | -1153.58 | 239.204 | -9.330 | 80.508 | -75.100 |
| 2012-05-01 02:37:00 | 9420 | 2504870.00 | -6435877.44 | -3551075.54 | 6411.67 | 3038.71 | -973.82 | 237.124 | -7.192 | 78.755 | -76.673 |
| 2012-05-01 02:38:00 | 9480 | 2885536.67 | -6243789.98 | -3604035.60 | 6273.98 | 3362.58 | -791.06 | 234.850 | -4.990 | 76.535 | -78.229 |
| 2012-05-01 02:39:00 | 9540 | 3257362.82 | -6032573.26 | -3645953.87 | 6117.06 | 3676.19 | -605.85 | 232.341 | -2.716 | 73.643 | -79.760 |
| 2012-05-01 02:40:00 | 9600 | 3619208.04 | -5802872.10 | -3676700.54 | 5941.37 | 3978.57 | -418.77 | 229.554 | -0.367 | 69.747 | -81.256 |
| 2012-05-01 02:41:00 | 9660 | 3969962.15 | -5555388.03 | -3696179.94 | 5747.45 | 4268.80 | -230.37 | 226.431 | 2.061 | 64.282 | -82.693 |
| 2012-05-01 02:42:00 | 9720 | 4308548.62 | -5290877.15 | -3704330.85 | 5535.88 | 4545.98 | -41.25 | 222.904 | 4.564 | 56.259 | -84.031 |
| 2012-05-01 02:43:00 | 9780 | 4633927.86 | -5010147.91 | -3701126.69 | 5307.32 | 4809.27 | 148.03 | 218.894 | 7.133 | 44.021 | -85.187 |
| 2012-05-01 02:44:00 | 9840 | 4945100.40 | -4714058.61 | -3686575.64 | 5062.45 | 5057.85 | 336.88 | 214.306 | 9.743 | 25.571 | -86.001 |
| 2012-05-01 02:45:00 | 9900 | 5241110.00 | -4403514.85 | -3660720.62 | 4802.01 | 5290.96 | 524.73 | 209.041 | 12.348 | 1.639 | -86.244 |
| 2012-05-01 02:46:00 | 9960 | 5521046.56 | -4079466.75 | -3623639.19 | 4526.81 | 5507.88 | 711.00 | 202.999 | 14.872 | 338.826 | -85.816 |
| 2012-05-01 02:47:00 | 10020 | 5784048.93 | -3742906.14 | -3575443.33 | 4237.68 | 5707.94 | 895.12 | 196.110 | 17.198 | 322.297 | -84.881 |
| 2012-05-01 02:48:00 | 10080 | 6029307.62 | -3394863.48 | -3516279.14 | 3935.51 | 5890.52 | 1076.52 | 188.365 | 19.169 | 311.473 | -83.661 |
| 2012-05-01 02:49:00 | 10140 | 6256067.24 | -3036404.78 | -3446326.34 | 3621.21 | 6055.05 | 1254.65 | 179.861 | 20.604 | 304.313 | -82.288 |
| 2012-05-01 02:50:00 | 10200 | 6463628.94 | -2668628.34 | -3365797.86 | 3295.74 | 6201.03 | 1428.95 | 170.831 | 21.336 | 299.368 | -80.830 |
| 2012-05-01 02:51:00 | 10260 | 6651352.49 | -2292661.39 | -3274939.10 | 2960.10 | 6328.00 | 1598.90 | 161.630 | 21.267 | 295.797 | -79.322 |
| 2012-05-01 02:52:00 | 10320 | 6818658.36 | -1909656.68 | -3174027.26 | 2615.32 | 6435.56 | 1763.97 | 152.656 | 20.405 | 293.115 | -77.782 |
| 2012-05-01 02:53:00 | 10380 | 6965029.48 | -1520788.96 | -3063370.49 | 2262.46 | 6523.38 | 1923.65 | 144.249 | 18.861 | 291.035 | -76.219 |
| 2012-05-01 02:54:00 | 10440 | 7090012.92 | -1127251.34 | -2943306.97 | 1902.58 | 6591.18 | 2077.45 | 136.626 | 16.806 | 289.377 | -74.640 |
| 2012-05-01 02:55:00 | 10500 | 7193221.29 | -730251.72 | -2814203.89 | 1536.81 | 6638.75 | 2224.89 | 129.867 | 14.420 | 288.026 | -73.049 |
| 2012-05-01 02:56:00 | 10560 | 7274334.00 | -331008.99 | -2676456.34 | 1166.25 | 6665.93 | 2365.52 | 123.954 | 11.856 | 286.903 | -71.448 |
| 2012-05-01 02:57:00 | 10620 | 7333098.25 | 69250.65 | -2530486.10 | 792.05 | 6672.64 | 2498.91 | 118.808 | 9.225 | 285.954 | -69.838 |
| 2012-05-01 02:58:00 | 10680 | 7369329.90 | 469297.46 | -2376740.38 | 415.35 | 6658.84 | 2624.64 | 114.331 | 6.598 | 285.141 | -68.221 |
| 2012-05-01 02:59:00 | 10740 | 7382914.05 | 867902.00 | -2215690.45 | 37.33 | 6624.57 | 2742.32 | 110.422 | 4.017 | 284.435 | -66.598 |
| 2012-05-01 03:00:00 | 10800 | 7373805.44 | 1263838.84 | -2047830.21 | -340.88 | 6569.94 | 2851.59 | 106.990 | 1.505 | 283.814 | -64.969 |
| 2012-05-01 03:01:00 | 10860 | 7342028.67 | 1655890.43 | -1873674.64 | -718.09 | 6495.10 | 2952.11 | 103.956 | -0.930 | 283.264 | -63.333 |
| 2012-05-01 03:02:00 | 10920 | 7287678.11 | 2042850.78 | -1693758.29 | -1093.14 | 6400.27 | 3043.57 | 101.254 | -3.286 | 282.770 | -61.692 |
| 2012-05-01 03:03:00 | 10980 | 7210917.67 | 2423529.27 | -1508633.58 | -1464.89 | 6285.75 | 3125.68 | 98.830 | -5.568 | 282.322 | -60.044 |
| 2012-05-01 03:04:00 | 11040 | 7111980.35 | 2796754.27 | -1318869.14 | -1832.18 | 6151.89 | 3198.18 | 96.638 | -7.780 | 281.913 | -58.391 |
| 2012-05-01 03:05:00 | 11100 | 6991167.56 | 3161376.80 | -1125048.06 | -2193.89 | 5999.08 | 3260.86 | 94.642 | -9.927 | 281.537 | -56.731 |
| 2012-05-01 03:06:00 | 11160 | 6848848.18 | 3516274.12 | -927766.05 | -2548.88 | 5827.80 | 3313.52 | 92.811 | -12.017 | 281.186 | -55.065 |
| 2012-05-01 03:07:00 | 11220 | 6685457.51 | 3860353.18 | -727629.67 | -2896.08 | 5638.56 | 3355.98 | 91.121 | -14.055 | 280.857 | -53.391 |
| 2012-05-01 03:08:00 | 11280 | 6501495.90 | 4192554.04 | -525254.41 | -3234.41 | 5431.96 | 3388.13 | 89.550 | -16.047 | 280.547 | -51.711 |
| 2012-05-01 03:09:00 | 11340 | 6297527.28 | 4511853.14 | -321262.80 | -3562.81 | 5208.61 | 3409.85 | 88.083 | -17.997 | 280.250 | -50.022 |
| 2012-05-01 03:10:00 | 11400 | 6074177.39 | 4817266.55 | -116282.48 | -3880.28 | 4969.22 | 3421.08 | 86.703 | -19.909 | 279.965 | -48.325 |
| 2012-05-01 03:11:00 | 11460 | 5832131.91 | 5107852.97 | 89055.73 | -4185.84 | 4714.51 | 3421.78 | 85.399 | -21.788 | 279.689 | -46.619 |
| 2012-05-01 03:12:00 | 11520 | 5572134.32 | 5382716.69 | 294119.81 | -4478.53 | 4445.26 | 3411.94 | 84.160 | -23.638 | 279.418 | -44.903 |
| 2012-05-01 03:13:00 | 11580 | 5294983.63 | 5641010.39 | 498278.48 | -4757.46 | 4162.32 | 3391.60 | 82.977 | -25.460 | 279.152 | -43.176 |
| 2012-05-01 03:14:00 | 11640 | 5001531.93 | 5881937.79 | 700903.14 | -5021.76 | 3866.53 | 3360.82 | 81.843 | -27.258 | 278.887 | -41.437 |
| 2012-05-01 03:15:00 | 11700 | 4692681.72 | 6104756.10 | 901369.89 | -5270.61 | 3558.83 | 3319.69 | 80.749 | -29.034 | 278.622 | -39.686 |
| 2012-05-01 03:16:00 | 11760 | 4369383.16 | 6308778.39 | 1099061.39 | -5503.25 | 3240.16 | 3268.34 | 79.691 | -30.790 | 278.355 | -37.920 |
| 2012-05-01 03:17:00 | 11820 | 4032631.09 | 6493375.70 | 1293368.84 | -5718.94 | 2911.50 | 3206.92 | 78.662 | -32.529 | 278.083 | -36.139 |
| 2012-05-01 03:18:00 | 11880 | 3683462.00 | 6657979.00 | 1483693.82 | -5917.04 | 2573.87 | 3135.62 | 77.657 | -34.251 | 277.804 | -34.341 |
| 2012-05-01 03:19:00 | 11940 | 3322950.76 | 6802080.98 | 1669450.19 | -6096.92 | 2228.30 | 3054.67 | 76.672 | -35.957 | 277.515 | -32.524 |
| 2012-05-01 03:20:00 | 12000 | 2952207.32 | 6925237.64 | 1850065.91 | -6258.03 | 1875.87 | 2964.31 | 75.702 | -37.650 | 277.216 | -30.685 |
| 2012-05-01 03:21:00 | 12060 | 2572373.25 | 7027069.65 | 2024984.77 | -6399.86 | 1517.66 | 2864.82 | 74.744 | -39.330 | 276.902 | -28.823 |
| 2012-05-01 03:22:00 | 12120 | 2184618.19 | 7107263.54 | 2193668.21 | -6521.99 | 1154.78 | 2756.52 | 73.793 | -40.998 | 276.571 | -26.934 |
| 2012-05-01 03:23:00 | 12180 | 1790136.23 | 7165572.68 | 2355596.88 | -6624.03 | 788.36 | 2639.72 | 72.844 | -42.656 | 276.219 | -25.015 |
| 2012-05-01 03:24:00 | 12240 | 1390142.19 | 7201818.04 | 2510272.34 | -6705.68 | 419.51 | 2514.80 | 71.895 | -44.302 | 275.844 | -23.062 |
| 2012-05-01 03:25:00 | 12300 | 985867.84 | 7215888.71 | 2657218.58 | -6766.67 | 49.39 | 2382.15 | 70.941 | -45.939 | 275.439 | -21.071 |
| 2012-05-01 03:26:00 | 12360 | 578558.08 | 7207742.30 | 2795983.48 | -6806.83 | -320.87 | 2242.16 | 69.978 | -47.567 | 275.000 | -19.036 |
| 2012-05-01 03:27:00 | 12420 | 169467.06 | 7177405.00 | 2926140.23 | -6826.03 | -690.11 | 2095.28 | 69.001 | -49.186 | 274.522 | -16.951 |
| 2012-05-01 03:28:00 | 12480 | -240145.68 | 7124971.49 | 3047288.63 | -6824.22 | -1057.22 | 1941.96 | 68.005 | -50.796 | 273.995 | -14.807 |
| 2012-05-01 03:29:00 | 12540 | -649019.18 | 7050604.69 | 3159056.36 | -6801.40 | -1421.04 | 1782.67 | 66.984 | -52.398 | 273.412 | -12.595 |
| 2012-05-01 03:30:00 | 12600 | -1055894.93 | 6954535.15 | 3261100.08 | -6757.65 | -1780.45 | 1617.91 | 65.934 | -53.992 | 272.761 | -10.304 |
| 2012-05-01 03:31:00 | 12660 | -1459520.77 | 6837060.38 | 3353106.53 | -6693.10 | -2134.36 | 1448.19 | 64.847 | -55.577 | 272.028 | -7.919 |
| 2012-05-01 03:32:00 | 12720 | -1858654.82 | 6698543.89 | 3434793.47 | -6607.96 | -2481.67 | 1274.02 | 63.716 | -57.154 | 271.195 | -5.424 |
| 2012-05-01 03:33:00 | 12780 | -2252069.27 | 6539414.02 | 3505910.56 | -6502.49 | -2821.30 | 1095.95 | 62.530 | -58.723 | 270.238 | -2.795 |
| 2012-05-01 03:34:00 | 12840 | -2638554.20 | 6360162.58 | 3566240.08 | -6377.04 | -3152.21 | 914.52 | 61.281 | -60.282 | 269.128 | -0.007 |
| 2012-05-01 03:35:00 | 12900 | -3016921.33 | 6161343.35 | 3615597.65 | -6231.97 | -3473.39 | 730.31 | 59.955 | -61.832 | 267.821 | 2.977 |
| 2012-05-01 03:36:00 | 12960 | -3386007.69 | 5943570.28 | 3653832.75 | -6067.76 | -3783.84 | 543.87 | 58.537 | -63.372 | 266.263 | 6.198 |
| 2012-05-01 03:37:00 | 13020 | -3744679.18 | 5707515.62 | 3680829.17 | -5884.90 | -4082.62 | 355.78 | 57.009 | -64.900 | 264.372 | 9.713 |
| 2012-05-01 03:38:00 | 13080 | -4091834.11 | 5453907.73 | 3696505.37 | -5683.97 | -4368.80 | 166.63 | 55.348 | -66.416 | 262.034 | 13.587 |
| 2012-05-01 03:39:00 | 13140 | -4426406.54 | 5183528.88 | 3700814.70 | -5465.59 | -4641.51 | -23.02 | 53.527 | -67.917 | 259.075 | 17.900 |
| 2012-05-01 03:40:00 | 13200 | -4747369.59 | 4897212.75 | 3693745.53 | -5230.44 | -4899.91 | -212.56 | 51.510 | -69.401 | 255.233 | 22.734 |
| 2012-05-01 03:41:00 | 13260 | -5053738.57 | 4595841.87 | 3675321.24 | -4979.25 | -5143.21 | -401.42 | 49.253 | -70.864 | 250.092 | 28.143 |
| 2012-05-01 03:42:00 | 13320 | -5344574.00 | 4280344.82 | 3645600.20 | -4712.79 | -5370.66 | -589.02 | 46.699 | -72.302 | 242.999 | 34.078 |
| 2012-05-01 03:43:00 | 13380 | -5618984.47 | 3951693.37 | 3604675.48 | -4431.89 | -5581.58 | -774.78 | 43.774 | -73.710 | 232.969 | 40.211 |
| 2012-05-01 03:44:00 | 13440 | -5876129.41 | 3610899.45 | 3552674.59 | -4137.42 | -5775.31 | -958.13 | 40.384 | -75.078 | 218.846 | 45.653 |
| 2012-05-01 03:45:00 | 13500 | -6115221.55 | 3259011.98 | 3489759.08 | -3830.29 | -5951.27 | -1138.51 | 36.406 | -76.396 | 200.409 | 48.842 |
| 2012-05-01 03:46:00 | 13560 | -6335529.41 | 2897113.69 | 3416123.96 | -3511.44 | -6108.92 | -1315.36 | 31.683 | -77.649 | 180.303 | 48.376 |
| 2012-05-01 03:47:00 | 13620 | -6536379.46 | 2526317.68 | 3331997.16 | -3181.86 | -6247.78 | -1488.14 | 26.023 | -78.814 | 162.798 | 44.492 |
| 2012-05-01 03:48:00 | 13680 | -6717158.16 | 2147764.04 | 3237638.72 | -2842.56 | -6367.44 | -1656.33 | 19.209 | -79.861 | 149.749 | 38.773 |
| 2012-05-01 03:49:00 | 13740 | -6877313.80 | 1762616.28 | 3133340.06 | -2494.60 | -6467.53 | -1819.40 | 11.039 | -80.752 | 140.529 | 32.644 |
| 2012-05-01 03:50:00 | 13800 | -7016358.19 | 1372057.79 | 3019422.99 | -2139.04 | -6547.75 | -1976.86 | 1.425 | -81.437 | 133.973 | 26.832 |
| 2012-05-01 03:51:00 | 13860 | -7133868.06 | 977288.16 | 2896238.73 | -1776.97 | -6607.87 | -2128.23 | 350.534 | -81.864 | 129.180 | 21.573 |
| 2012-05-01 03:52:00 | 13920 | -7229486.37 | 579519.52 | 2764166.82 | -1409.50 | -6647.70 | -2273.04 | 338.906 | -81.991 | 125.563 | 16.880 |
| 2012-05-01 03:53:00 | 13980 | -7302923.31 | 179972.77 | 2623613.93 | -1037.78 | -6667.12 | -2410.85 | 327.359 | -81.805 | 122.752 | 12.687 |
| 2012-05-01 03:54:00 | 14040 | -7353957.19 | -220126.12 | 2475012.62 | -662.93 | -6666.10 | -2541.25 | 316.682 | -81.325 | 120.508 | 8.913 |
| 2012-05-01 03:55:00 | 14100 | -7382435.05 | -619549.86 | 2318819.94 | -286.10 | -6644.63 | -2663.84 | 307.344 | -80.596 | 118.677 | 5.481 |
| 2012-05-01 03:56:00 | 14160 | -7388273.08 | -1017073.65 | 2155516.09 | 91.54 | -6602.78 | -2778.23 | 299.455 | -79.670 | 117.152 | 2.328 |
| 2012-05-01 03:57:00 | 14220 | -7371456.83 | -1411478.88 | 1985602.87 | 468.85 | -6540.70 | -2884.09 | 292.896 | -78.595 | 115.861 | -0.600 |
| 2012-05-01 03:58:00 | 14280 | -7332041.25 | -1801556.83 | 1809602.21 | 844.66 | -6458.58 | -2981.10 | 287.458 | -77.407 | 114.750 | -3.341 |
| 2012-05-01 03:59:00 | 14340 | -7270150.42 | -2186112.42 | 1628054.48 | 1217.83 | -6356.67 | -3068.95 | 282.925 | -76.136 | 113.781 | -5.930 |
| 2012-05-01 04:00:00 | 14400 | -7185977.20 | -2563967.79 | 1441516.88 | 1587.22 | -6235.29 | -3147.38 | 279.112 | -74.801 | 112.926 | -8.391 |
| 2012-05-01 04:01:00 | 14460 | -7079782.53 | -2933965.90 | 1250561.75 | 1951.69 | -6094.83 | -3216.16 | 275.870 | -73.418 | 112.162 | -10.745 |
| 2012-05-01 04:02:00 | 14520 | -6951894.67 | -3294974.06 | 1055774.74 | 2310.14 | -5935.71 | -3275.08 | 273.080 | -71.996 | 111.472 | -13.010 |
| 2012-05-01 04:03:00 | 14580 | -6802708.10 | -3645887.35 | 857753.13 | 2661.47 | -5758.42 | -3323.96 | 270.652 | -70.544 | 110.843 | -15.198 |
| 2012-05-01 04:04:00 | 14640 | -6632682.34 | -3985631.96 | 657103.89 | 3004.60 | -5563.51 | -3362.64 | 268.518 | -69.067 | 110.264 | -17.320 |
| 2012-05-01 04:05:00 | 14700 | -6442340.49 | -4313168.50 | 454441.91 | 3338.50 | -5351.59 | -3391.03 | 266.621 | -67.570 | 109.726 | -19.386 |
| 2012-05-01 04:06:00 | 14760 | -6232267.61 | -4627495.08 | 250388.10 | 3662.14 | -5123.29 | -3409.03 | 264.920 | -66.054 | 109.222 | -21.402 |
| 2012-05-01 04:07:00 | 14820 | -6003108.93 | -4927650.36 | 45567.49 | 3974.53 | -4879.33 | -3416.58 | 263.381 | -64.523 | 108.745 | -23.376 |
| 2012-05-01 04:08:00 | 14880 | -5755567.87 | -5212716.48 | -159392.67 | 4274.73 | -4620.45 | -3413.68 | 261.978 | -62.979 | 108.291 | -25.312 |
| 2012-05-01 04:09:00 | 14940 | -5490403.84 | -5481821.81 | -363864.79 | 4561.81 | -4347.44 | -3400.32 | 260.690 | -61.422 | 107.855 | -27.215 |
| 2012-05-01 04:10:00 | 15000 | -5208429.98 | -5734143.61 | -567222.88 | 4834.91 | -4061.14 | -3376.55 | 259.498 | -59.854 | 107.433 | -29.088 |
| 2012-05-01 04:11:00 | 15060 | -4910510.63 | -5968910.47 | -768844.44 | 5093.20 | -3762.43 | -3342.45 | 258.388 | -58.275 | 107.022 | -30.935 |
| 2012-05-01 04:12:00 | 15120 | -4597558.69 | -6185404.67 | -968112.33 | 5335.87 | -3452.21 | -3298.12 | 257.349 | -56.686 | 106.617 | -32.759 |
| 2012-05-01 04:13:00 | 15180 | -4270532.87 | -6382964.34 | -1164416.69 | 5562.21 | -3131.44 | -3243.69 | 256.371 | -55.087 | 106.217 | -34.561 |
| 2012-05-01 04:14:00 | 15240 | -3930434.74 | -6560985.47 | -1357156.71 | 5771.51 | -2801.09 | -3179.34 | 255.445 | -53.479 | 105.818 | -36.344 |
| 2012-05-01 04:15:00 | 15300 | -3578305.72 | -6718923.71 | -1545742.53 | 5963.13 | -2462.18 | -3105.25 | 254.565 | -51.860 | 105.418 | -38.110 |
| 2012-05-01 04:16:00 | 15360 | -3215223.87 | -6856296.02 | -1729597.00 | 6136.51 | -2115.73 | -3021.67 | 253.723 | -50.232 | 105.014 | -39.860 |
| 2012-05-01 04:17:00 | 15420 | -2842300.67 | -6972682.16 | -1908157.41 | 6291.10 | -1762.82 | -2928.83 | 252.915 | -48.593 | 104.603 | -41.596 |
| 2012-05-01 04:18:00 | 15480 | -2460677.59 | -7067725.91 | -2080877.22 | 6426.43 | -1404.50 | -2827.03 | 252.135 | -46.943 | 104.182 | -43.318 |
| 2012-05-01 04:19:00 | 15540 | -2071522.68 | -7141136.20 | -2247227.72 | 6542.09 | -1041.88 | -2716.57 | 251.380 | -45.283 | 103.748 | -45.028 |
| 2012-05-01 04:20:00 | 15600 | -1676026.98 | -7192687.96 | -2406699.61 | 6637.73 | -676.07 | -2597.80 | 250.646 | -43.610 | 103.299 | -46.727 |
| 2012-05-01 04:21:00 | 15660 | -1275400.95 | -7222222.84 | -2558804.56 | 6713.06 | -308.17 | -2471.07 | 249.929 | -41.926 | 102.831 | -48.415 |
| 2012-05-01 04:22:00 | 15720 | -870870.78 | -7229649.65 | -2703076.71 | 6767.84 | 60.68 | -2336.77 | 249.226 | -40.228 | 102.340 | -50.093 |
| 2012-05-01 04:23:00 | 15780 | -463674.67 | -7214944.71 | -2839074.07 | 6801.90 | 429.36 | -2195.31 | 248.533 | -38.515 | 101.822 | -51.761 |
| 2012-05-01 04:24:00 | 15840 | -55059.07 | -7178151.85 | -2966379.85 | 6815.14 | 796.76 | -2047.13 | 247.848 | -36.787 | 101.272 | -53.420 |
| 2012-05-01 04:25:00 | 15900 | 353725.08 | -7119382.39 | -3084603.77 | 6807.52 | 1161.73 | -1892.66 | 247.168 | -35.042 | 100.684 | -55.070 |
| 2012-05-01 04:26:00 | 15960 | 761426.18 | -7038814.74 | -3193383.21 | 6779.05 | 1523.18 | -1732.39 | 246.489 | -33.279 | 100.053 | -56.712 |
| 2012-05-01 04:27:00 | 16020 | 1166795.74 | -6936693.90 | -3292384.36 | 6729.82 | 1879.99 | -1566.80 | 245.810 | -31.496 | 99.371 | -58.345 |
| 2012-05-01 04:28:00 | 16080 | 1568592.19 | -6813330.75 | -3381303.20 | 6659.98 | 2231.07 | -1396.40 | 245.125 | -29.690 | 98.629 | -59.970 |
| 2012-05-01 04:29:00 | 16140 | 1965584.65 | -6669101.12 | -3459866.46 | 6569.73 | 2575.36 | -1221.70 | 244.433 | -27.859 | 97.816 | -61.586 |
| 2012-05-01 04:30:00 | 16200 | 2356556.69 | -6504444.67 | -3527832.47 | 6459.35 | 2911.79 | -1043.25 | 243.730 | -26.001 | 96.919 | -63.193 |
| 2012-05-01 04:31:00 | 16260 | 2740310.01 | -6319863.58 | -3584991.89 | 6329.16 | 3239.35 | -861.58 | 243.010 | -24.112 | 95.922 | -64.789 |
| 2012-05-01 04:32:00 | 16320 | 3115668.13 | -6115921.03 | -3631168.37 | 6179.58 | 3557.01 | -677.24 | 242.271 | -22.188 | 94.805 | -66.376 |
| 2012-05-01 04:33:00 | 16380 | 3481479.94 | -5893239.55 | -3666219.09 | 6011.03 | 3863.82 | -490.81 | 241.506 | -20.224 | 93.542 | -67.950 |
| 2012-05-01 04:34:00 | 16440 | 3836623.25 | -5652499.14 | -3690035.22 | 5824.05 | 4158.82 | -302.85 | 240.710 | -18.216 | 92.101 | -69.511 |
| 2012-05-01 04:35:00 | 16500 | 4180008.21 | -5394435.19 | -3702542.29 | 5619.19 | 4441.12 | -113.94 | 239.875 | -16.156 | 90.439 | -71.056 |
| 2012-05-01 04:36:00 | 16560 | 4510580.66 | -5119836.33 | -3703700.38 | 5397.07 | 4709.84 | 75.35 | 238.993 | -14.037 | 88.501 | -72.582 |
| 2012-05-01 04:37:00 | 16620 | 4827325.34 | -4829542.01 | -3693504.33 | 5158.38 | 4964.17 | 264.44 | 238.054 | -11.850 | 86.212 | -74.084 |
| 2012-05-01 04:38:00 | 16680 | 5129269.05 | -4524439.97 | -3671983.71 | 4903.83 | 5203.31 | 452.74 | 237.044 | -9.582 | 83.470 | -75.556 |
| 2012-05-01 04:39:00 | 16740 | 5415483.64 | -4205463.57 | -3639202.81 | 4634.21 | 5426.53 | 639.68 | 235.948 | -7.221 | 80.136 | -76.988 |
| 2012-05-01 04:40:00 | 16800 | 5685088.82 | -3873588.98 | -3595260.42 | 4350.33 | 5633.14 | 824.69 | 234.745 | -4.749 | 76.016 | -78.367 |
| 2012-05-01 04:41:00 | 16860 | 5937254.96 | -3529832.17 | -3540289.56 | 4053.05 | 5822.50 | 1007.20 | 233.410 | -2.145 | 70.848 | -79.672 |
| 2012-05-01 04:42:00 | 16920 | 6171205.61 | -3175245.89 | -3474457.12 | 3743.30 | 5994.03 | 1186.65 | 231.907 | 0.618 | 64.281 | -80.872 |
| 2012-05-01 04:43:00 | 16980 | 6386219.92 | -2810916.40 | -3397963.33 | 3422.00 | 6147.19 | 1362.49 | 230.192 | 3.572 | 55.900 | -81.921 |
| 2012-05-01 04:44:00 | 17040 | 6581634.92 | -2437960.25 | -3311041.20 | 3090.15 | 6281.51 | 1534.18 | 228.200 | 6.759 | 45.347 | -82.754 |
| 2012-05-01 04:45:00 | 17100 | 6756847.52 | -2057520.80 | -3213955.81 | 2748.76 | 6396.57 | 1701.18 | 225.844 | 10.227 | 32.629 | -83.289 |
| 2012-05-01 04:46:00 | 17160 | 6911316.47 | -1670764.75 | -3107003.53 | 2398.88 | 6492.01 | 1862.99 | 222.999 | 14.036 | 18.532 | -83.455 |
| 2012-05-01 04:47:00 | 17220 | 7044564.02 | -1278878.60 | -2990511.12 | 2041.56 | 6567.53 | 2019.10 | 219.478 | 18.250 | 4.572 | -83.224 |
| 2012-05-01 04:48:00 | 17280 | 7156177.45 | -883065.01 | -2864834.76 | 1677.92 | 6622.89 | 2169.04 | 215.006 | 22.927 | 352.181 | -82.633 |
| 2012-05-01 04:49:00 | 17340 | 7245810.38 | -484539.08 | -2730358.94 | 1309.07 | 6657.91 | 2312.34 | 209.164 | 28.077 | 341.996 | -81.759 |
| 2012-05-01 04:50:00 | 17400 | 7313183.86 | -84524.66 | -2587495.36 | 936.13 | 6672.49 | 2448.56 | 201.329 | 33.585 | 333.933 | -80.682 |
| 2012-05-01 04:51:00 | 17460 | 7358087.30 | 315749.43 | -2436681.61 | 560.26 | 6666.56 | 2577.28 | 190.667 | 39.049 | 327.606 | -79.462 |
| 2012-05-01 04:52:00 | 17520 | 7380379.15 | 715053.18 | -2278379.89 | 182.60 | 6640.15 | 2698.10 | 176.426 | 43.580 | 322.609 | -78.144 |
| 2012-05-01 04:53:00 | 17580 | 7379987.37 | 1112159.19 | -2113075.58 | -195.67 | 6593.32 | 2810.64 | 158.948 | 45.867 | 318.605 | -76.757 |
| 2012-05-01 04:54:00 | 17640 | 7356909.72 | 1505846.41 | -1941275.74 | -573.40 | 6526.22 | 2914.56 | 140.691 | 44.979 | 315.344 | -75.319 |
| 2012-05-01 04:55:00 | 17700 | 7311213.77 | 1894903.99 | -1763507.60 | -949.42 | 6439.04 | 3009.53 | 124.789 | 41.294 | 312.643 | -73.843 |
| 2012-05-01 04:56:00 | 17760 | 7243036.78 | 2278134.96 | -1580316.91 | -1322.57 | 6332.05 | 3095.26 | 112.538 | 36.088 | 310.369 | -72.339 |
| 2012-05-01 04:57:00 | 17820 | 7152585.29 | 2654359.95 | -1392266.29 | -1691.71 | 6205.57 | 3171.49 | 103.524 | 30.494 | 308.428 | -70.813 |
| 2012-05-01 04:58:00 | 17880 | 7040134.53 | 3022420.87 | -1199933.47 | -2055.69 | 6059.98 | 3237.97 | 96.878 | 25.123 | 306.747 | -69.268 |
| 2012-05-01 04:59:00 | 17940 | 6906027.57 | 3381184.48 | -1003909.54 | -2413.40 | 5895.74 | 3294.49 | 91.872 | 20.200 | 305.272 | -67.708 |
| 2012-05-01 05:00:00 | 18000 | 6750674.34 | 3729545.91 | -804797.12 | -2763.72 | 5713.33 | 3340.89 | 87.998 | 15.756 | 303.964 | -66.136 |
| 2012-05-01 05:01:00 | 18060 | 6574550.40 | 4066432.12 | -603208.48 | -3105.58 | 5513.33 | 3377.01 | 84.922 | 11.748 | 302.791 | -64.552 |
| 2012-05-01 05:02:00 | 18120 | 6378195.43 | 4390805.19 | -399763.66 | -3437.91 | 5296.33 | 3402.74 | 82.421 | 8.112 | 301.728 | -62.958 |
| 2012-05-01 05:03:00 | 18180 | 6162211.66 | 4701665.59 | -195088.60 | -3759.70 | 5063.02 | 3418.01 | 80.345 | 4.786 | 300.756 | -61.355 |
| 2012-05-01 05:04:00 | 18240 | 5927261.98 | 4998055.31 | 10186.90 | -4069.95 | 4814.10 | 3422.75 | 78.590 | 1.714 | 299.858 | -59.744 |
| 2012-05-01 05:05:00 | 18300 | 5674067.93 | 5279060.77 | 215431.03 | -4367.69 | 4550.34 | 3416.96 | 77.084 | -1.148 | 299.023 | -58.124 |
| 2012-05-01 05:06:00 | 18360 | 5403407.48 | 5543815.75 | 420012.00 | -4652.01 | 4272.56 | 3400.65 | 75.774 | -3.838 | 298.238 | -56.496 |
| 2012-05-01 05:07:00 | 18420 | 5116112.63 | 5791504.07 | 623299.95 | -4922.03 | 3981.60 | 3373.87 | 74.619 | -6.384 | 297.496 | -54.861 |
| 2012-05-01 05:08:00 | 18480 | 4813066.85 | 6021362.11 | 824668.94 | -5176.91 | 3678.37 | 3336.70 | 73.592 | -8.810 | 296.789 | -53.218 |
| 2012-05-01 05:09:00 | 18540 | 4495202.32 | 6232681.22 | 1023498.89 | -5415.86 | 3363.79 | 3289.26 | 72.669 | -11.136 | 296.110 | -51.567 |
| 2012-05-01 05:10:00 | 18600 | 4163497.08 | 6424809.93 | 1219177.49 | -5638.15 | 3038.85 | 3231.69 | 71.833 | -13.376 | 295.453 | -49.907 |
| 2012-05-01 05:11:00 | 18660 | 3818972.01 | 6597155.97 | 1411102.11 | -5843.08 | 2704.54 | 3164.16 | 71.070 | -15.544 | 294.814 | -48.239 |
| 2012-05-01 05:12:00 | 18720 | 3462687.61 | 6749188.13 | 1598681.68 | -6030.02 | 2361.90 | 3086.89 | 70.368 | -17.649 | 294.187 | -46.562 |
| 2012-05-01 05:13:00 | 18780 | 3095740.77 | 6880437.93 | 1781338.52 | -6198.40 | 2011.97 | 3000.11 | 69.720 | -19.700 | 293.569 | -44.876 |
| 2012-05-01 05:14:00 | 18840 | 2719261.33 | 6990501.03 | 1958510.15 | -6347.69 | 1655.85 | 2904.09 | 69.117 | -21.705 | 292.955 | -43.179 |
| 2012-05-01 05:15:00 | 18900 | 2334408.56 | 7079038.54 | 2129651.02 | -6477.44 | 1294.64 | 2799.14 | 68.554 | -23.668 | 292.342 | -41.471 |
| 2012-05-01 05:16:00 | 18960 | 1942367.62 | 7145778.08 | 2294234.20 | -6587.24 | 929.44 | 2685.56 | 68.026 | -25.596 | 291.726 | -39.752 |
| 2012-05-01 05:17:00 | 19020 | 1544345.79 | 7190514.55 | 2451753.05 | -6676.75 | 561.39 | 2563.72 | 67.527 | -27.492 | 291.103 | -38.020 |
| 2012-05-01 05:18:00 | 19080 | 1141568.78 | 7213110.85 | 2601722.75 | -6745.70 | 191.63 | 2433.99 | 67.055 | -29.360 | 290.470 | -36.275 |
| 2012-05-01 05:19:00 | 19140 | 735276.92 | 7213498.23 | 2743681.84 | -6793.88 | -178.71 | 2296.77 | 66.606 | -31.204 | 289.822 | -34.514 |
| 2012-05-01 05:20:00 | 19200 | 326721.26 | 7191676.53 | 2877193.63 | -6821.14 | -548.49 | 2152.48 | 66.178 | -33.025 | 289.156 | -32.737 |
| 2012-05-01 05:21:00 | 19260 | -82840.27 | 7147714.18 | 3001847.55 | -6827.40 | -916.55 | 2001.58 | 65.768 | -34.827 | 288.467 | -30.941 |
| 2012-05-01 05:22:00 | 19320 | -492146.80 | 7081747.91 | 3117260.43 | -6812.65 | -1281.76 | 1844.53 | 65.375 | -36.610 | 287.751 | -29.126 |
| 2012-05-01 05:23:00 | 19380 | -899938.40 | 6993982.39 | 3223077.66 | -6776.92 | -1643.00 | 1681.81 | 64.995 | -38.378 | 287.002 | -27.288 |
| 2012-05-01 05:24:00 | 19440 | -1304960.02 | 6884689.52 | 3318974.32 | -6720.34 | -1999.15 | 1513.93 | 64.628 | -40.131 | 286.215 | -25.425 |
| 2012-05-01 05:25:00 | 19500 | -1705965.38 | 6754207.56 | 3404656.15 | -6643.08 | -2349.12 | 1341.40 | 64.272 | -41.871 | 285.383 | -23.535 |
| 2012-05-01 05:26:00 | 19560 | -2101720.79 | 6602940.11 | 3479860.45 | -6545.39 | -2691.83 | 1164.77 | 63.925 | -43.600 | 284.498 | -21.614 |
| 2012-05-01 05:27:00 | 19620 | -2491009.04 | 6431354.78 | 3544356.89 | -6427.56 | -3026.21 | 984.57 | 63.585 | -45.317 | 283.551 | -19.658 |
| 2012-05-01 05:28:00 | 19680 | -2872633.08 | 6239981.74 | 3597948.22 | -6289.98 | -3351.25 | 801.36 | 63.253 | -47.025 | 282.533 | -17.663 |
| 2012-05-01 05:29:00 | 19740 | -3245419.80 | 6029412.05 | 3640470.86 | -6133.06 | -3665.94 | 615.70 | 62.925 | -48.723 | 281.431 | -15.624 |
| 2012-05-01 05:30:00 | 19800 | -3608223.60 | 5800295.81 | 3671795.36 | -5957.30 | -3969.31 | 428.18 | 62.601 | -50.414 | 280.230 | -13.534 |
| 2012-05-01 05:31:00 | 19860 | -3959929.91 | 5553340.09 | 3691826.84 | -5763.24 | -4260.43 | 239.37 | 62.279 | -52.097 | 278.914 | -11.387 |
| 2012-05-01 05:32:00 | 19920 | -4299458.69 | 5289306.73 | 3700505.23 | -5551.49 | -4538.42 | 49.84 | 61.958 | -53.773 | 277.459 | -9.174 |
| 2012-05-01 05:33:00 | 19980 | -4625767.66 | 5009009.97 | 3697805.44 | -5322.70 | -4802.41 | -139.81 | 61.636 | -55.442 | 275.841 | -6.887 |
| 2012-05-01 05:34:00 | 20040 | -4937855.57 | 4713313.90 | 3683737.40 | -5077.58 | -5051.60 | -329.00 | 61.311 | -57.106 | 274.026 | -4.516 |
| 2012-05-01 05:35:00 | 20100 | -5234765.23 | 4403129.74 | 3658346.08 | -4816.88 | -5285.22 | -517.15 | 60.981 | -58.765 | 271.974 | -2.049 |
| 2012-05-01 05:36:00 | 20160 | -5515586.44 | 4079413.01 | 3621711.23 | -4541.43 | -5502.57 | -703.69 | 60.643 | -60.418 | 269.633 | 0.526 |
| 2012-05-01 05:37:00 | 20220 | -5779458.78 | 3743160.60 | 3573947.19 | -4252.07 | -5702.97 | -888.03 | 60.295 | -62.067 | 266.937 | 3.220 |
| 2012-05-01 05:38:00 | 20280 | -6025574.22 | 3395407.61 | 3515202.48 | -3949.69 | -5885.82 | -1069.62 | 59.934 | -63.712 | 263.805 | 6.040 |
| 2012-05-01 05:39:00 | 20340 | -6253179.59 | 3037224.16 | 3445659.34 | -3635.22 | -6050.57 | -1247.89 | 59.554 | -65.352 | 260.133 | 8.987 |
| 2012-05-01 05:40:00 | 20400 | -6461578.84 | 2669712.09 | 3365533.13 | -3309.65 | -6196.70 | -1422.30 | 59.150 | -66.989 | 255.791 | 12.048 |
| 2012-05-01 05:41:00 | 20460 | -6650135.16 | 2294001.53 | 3275071.65 | -2973.96 | -6323.78 | -1592.31 | 58.716 | -68.623 | 250.627 | 15.180 |
| 2012-05-01 05:42:00 | 20520 | -6818272.90 | 1911247.43 | 3174554.37 | -2629.20 | -6431.42 | -1757.40 | 58.241 | -70.253 | 244.473 | 18.297 |
| 2012-05-01 05:43:00 | 20580 | -6965479.27 | 1522625.98 | 3064291.54 | -2276.43 | -6519.31 | -1917.08 | 57.715 | -71.880 | 237.176 | 21.244 |
| 2012-05-01 05:44:00 | 20640 | -7091305.93 | 1129330.96 | 2944623.21 | -1916.73 | -6587.17 | -2070.84 | 57.119 | -73.504 | 228.662 | 23.786 |
| 2012-05-01 05:45:00 | 20700 | -7195370.25 | 732570.14 | 2815918.18 | -1551.20 | -6634.81 | -2218.22 | 56.430 | -75.125 | 219.030 | 25.627 |
| 2012-05-01 05:46:00 | 20760 | -7277356.49 | 333561.48 | 2678572.85 | -1180.98 | -6662.08 | -2358.78 | 55.610 | -76.742 | 208.638 | 26.484 |
| 2012-05-01 05:47:00 | 20820 | -7337016.72 | -66470.54 | 2533009.99 | -807.19 | -6668.91 | -2492.07 | 54.605 | -78.355 | 198.076 | 26.203 |
| 2012-05-01 05:48:00 | 20880 | -7374171.50 | -466298.69 | 2379677.40 | -430.99 | -6655.29 | -2617.70 | 53.322 | -79.964 | 187.993 | 24.837 |
| 2012-05-01 05:49:00 | 20940 | -7388710.44 | -864696.79 | 2219046.56 | -53.53 | -6621.26 | -2735.29 | 51.603 | -81.565 | 178.874 | 22.619 |
| 2012-05-01 05:50:00 | 21000 | -7380592.44 | -1260443.39 | 2051611.17 | 324.05 | -6566.93 | -2844.46 | 49.140 | -83.156 | 170.944 | 19.852 |
| 2012-05-01 05:51:00 | 21060 | -7349845.80 | -1652325.54 | 1877885.59 | 700.57 | -6492.48 | -2944.91 | 45.269 | -84.728 | 164.204 | 16.806 |
| 2012-05-01 05:52:00 | 21120 | -7296568.09 | -2039142.50 | 1698403.32 | 1074.89 | -6398.13 | -3036.31 | 38.274 | -86.257 | 158.528 | 13.677 |
| 2012-05-01 05:53:00 | 21180 | -7220925.82 | -2419709.34 | 1513715.29 | 1445.87 | -6284.19 | -3118.39 | 22.481 | -87.661 | 153.753 | 10.584 |
| 2012-05-01 05:54:00 | 21240 | -7123153.87 | -2792860.63 | 1324388.23 | 1812.36 | -6151.01 | -3190.90 | 339.096 | -88.524 | 149.714 | 7.587 |
| 2012-05-01 05:55:00 | 21300 | -7003554.75 | -3157453.91 | 1131002.89 | 2173.25 | -5999.00 | -3253.63 | 286.953 | -87.966 | 146.270 | 4.713 |
| 2012-05-01 05:56:00 | 21360 | -6862497.64 | -3512373.18 | 934152.29 | 2527.44 | -5828.63 | -3306.38 | 266.640 | -86.632 | 143.306 | 1.968 |
| 2012-05-01 05:57:00 | 21420 | -6700417.25 | -3856532.31 | 734439.91 | 2873.85 | -5640.42 | -3349.00 | 258.270 | -85.122 | 140.728 | -0.653 |
| 2012-05-01 05:58:00 | 21480 | -6517812.43 | -4188878.30 | 532477.81 | 3211.42 | -5434.96 | -3381.35 | 253.870 | -83.557 | 138.462 | -3.159 |
| 2012-05-01 05:59:00 | 21540 | -6315244.65 | -4508394.49 | 328884.81 | 3539.11 | -5212.87 | -3403.35 | 251.173 | -81.969 | 136.451 | -5.564 |
| 2012-05-01 06:00:00 | 21600 | -6093336.26 | -4814103.61 | 124284.59 | 3855.94 | -4974.84 | -3414.92 | 249.349 | -80.369 | 134.649 | -7.877 |
| 2012-05-01 06:01:00 | 21660 | -5852768.57 | -5105070.79 | -80696.24 | 4160.93 | -4721.60 | -3416.03 | 248.030 | -78.761 | 133.018 | -10.110 |
| 2012-05-01 06:02:00 | 21720 | -5594279.76 | -5380406.30 | -285429.98 | 4453.16 | -4453.92 | -3406.69 | 247.029 | -77.148 | 131.530 | -12.272 |
| 2012-05-01 06:03:00 | 21780 | -5318662.61 | -5639268.33 | -489289.82 | 4731.73 | -4172.62 | -3386.91 | 246.242 | -75.531 | 130.160 | -14.371 |
| 2012-05-01 06:04:00 | 21840 | -5026762.10 | -5880865.46 | -691651.67 | 4995.80 | -3878.57 | -3356.77 | 245.606 | -73.910 | 128.889 | -16.416 |
| 2012-05-01 06:05:00 | 21900 | -4719472.81 | -6104459.10 | -891896.13 | 5244.56 | -3572.66 | -3316.35 | 245.082 | -72.285 | 127.700 | -18.412 |
| 2012-05-01 06:06:00 | 21960 | -4397736.19 | -6309365.68 | -1089410.31 | 5477.26 | -3255.82 | -3265.78 | 244.641 | -70.657 | 126.580 | -20.365 |
| 2012-05-01 06:07:00 | 22020 | -4062537.70 | -6494958.74 | -1283589.71 | 5693.18 | -2929.04 | -3205.22 | 244.267 | -69.025 | 125.517 | -22.280 |
| 2012-05-01 06:08:00 | 22080 | -3714903.82 | -6660670.80 | -1473840.06 | 5891.66 | -2593.29 | -3134.84 | 243.945 | -67.391 | 124.501 | -24.161 |
| 2012-05-01 06:09:00 | 22140 | -3355898.93 | -6805995.10 | -1659579.10 | 6072.11 | -2249.62 | -3054.88 | 243.666 | -65.752 | 123.524 | -26.011 |
| 2012-05-01 06:10:00 | 22200 | -2986622.06 | -6930487.12 | -1840238.39 | 6233.98 | -1899.06 | -2965.56 | 243.424 | -64.110 | 122.579 | -27.833 |
| 2012-05-01 06:11:00 | 22260 | -2608203.56 | -7033765.92 | -2015264.98 | 6376.76 | -1542.69 | -2867.17 | 243.211 | -62.463 | 121.657 | -29.630 |
| 2012-05-01 06:12:00 | 22320 | -2221801.70 | -7115515.28 | -2184123.10 | 6500.02 | -1181.59 | -2760.00 | 243.025 | -60.812 | 120.754 | -31.404 |
| 2012-05-01 06:13:00 | 22380 | -1828599.11 | -7175484.69 | -2346295.81 | 6603.39 | -816.88 | -2644.38 | 242.862 | -59.156 | 119.864 | -33.157 |
| 2012-05-01 06:14:00 | 22440 | -1429799.23 | -7213490.11 | -2501286.57 | 6686.55 | -449.65 | -2520.66 | 242.720 | -57.496 | 118.982 | -34.891 |
| 2012-05-01 06:15:00 | 22500 | -1026622.63 | -7229414.51 | -2648620.71 | 6749.24 | -81.03 | -2389.22 | 242.596 | -55.829 | 118.103 | -36.607 |
| 2012-05-01 06:16:00 | 22560 | -620303.36 | -7223208.26 | -2787846.89 | 6791.28 | 287.85 | -2250.46 | 242.488 | -54.157 | 117.221 | -38.306 |
| 2012-05-01 06:17:00 | 22620 | -212085.12 | -7194889.27 | -2918538.51 | 6812.52 | 655.88 | -2104.81 | 242.396 | -52.478 | 116.333 | -39.990 |
| 2012-05-01 06:18:00 | 22680 | 196782.40 | -7144542.95 | -3040294.95 | 6812.92 | 1021.91 | -1952.70 | 242.318 | -50.791 | 115.433 | -41.659 |
| 2012-05-01 06:19:00 | 22740 | 605047.41 | -7072322.01 | -3152742.83 | 6792.45 | 1384.84 | -1794.60 | 242.254 | -49.097 | 114.518 | -43.313 |
| 2012-05-01 06:20:00 | 22800 | 1011459.78 | -6978445.94 | -3255537.15 | 6751.18 | 1743.56 | -1631.00 | 242.203 | -47.395 | 113.580 | -44.954 |
| 2012-05-01 06:21:00 | 22860 | 1414774.83 | -6863200.42 | -3348362.34 | 6689.23 | 2096.98 | -1462.38 | 242.165 | -45.683 | 112.617 | -46.582 |
| 2012-05-01 06:22:00 | 22920 | 1813757.13 | -6726936.46 | -3430933.22 | 6606.79 | 2444.00 | -1289.27 | 242.138 | -43.961 | 111.621 | -48.197 |
| 2012-05-01 06:23:00 | 22980 | 2207184.27 | -6570069.35 | -3502995.87 | 6504.10 | 2783.57 | -1112.20 | 242.124 | -42.228 | 110.587 | -49.799 |
| 2012-05-01 06:24:00 | 23040 | 2593850.54 | -6393077.40 | -3564328.47 | 6381.48 | 3114.66 | -931.70 | 242.122 | -40.482 | 109.507 | -51.388 |
| 2012-05-01 06:25:00 | 23100 | 2972570.65 | -6196500.59 | -3614741.92 | 6239.30 | 3436.24 | -748.32 | 242.133 | -38.723 | 108.375 | -52.964 |
| 2012-05-01 06:26:00 | 23160 | 3342183.32 | -5980938.84 | -3654080.45 | 6077.98 | 3747.33 | -562.63 | 242.155 | -36.949 | 107.182 | -54.527 |
| 2012-05-01 06:27:00 | 23220 | 3701554.85 | -5747050.32 | -3682222.15 | 5898.01 | 4046.97 | -375.19 | 242.191 | -35.159 | 105.917 | -56.075 |
| 2012-05-01 06:28:00 | 23280 | 4049582.57 | -5495549.40 | -3699079.31 | 5699.95 | 4334.26 | -186.57 | 242.239 | -33.350 | 104.571 | -57.609 |
| 2012-05-01 06:29:00 | 23340 | 4385198.21 | -5227204.53 | -3704598.72 | 5484.38 | 4608.29 | 2.64 | 242.302 | -31.520 | 103.130 | -59.127 |
| 2012-05-01 06:30:00 | 23400 | 4707371.22 | -4942835.93 | -3698761.85 | 5251.97 | 4868.24 | 191.87 | 242.379 | -29.668 | 101.579 | -60.627 |
| 2012-05-01 06:31:00 | 23460 | 5015111.86 | -4643313.11 | -3681584.95 | 5003.42 | 5113.31 | 380.55 | 242.472 | -27.790 | 99.901 | -62.108 |
| 2012-05-01 06:32:00 | 23520 | 5307474.34 | -4329552.24 | -3653118.98 | 4739.50 | 5342.72 | 568.08 | 242.582 | -25.883 | 98.075 | -63.568 |
| 2012-05-01 06:33:00 | 23580 | 5583559.66 | -4002513.38 | -3613449.51 | 4460.99 | 5555.79 | 753.90 | 242.710 | -23.943 | 96.078 | -65.004 |
| 2012-05-01 06:34:00 | 23640 | 5842518.41 | -3663197.58 | -3562696.45 | 4168.75 | 5751.85 | 937.44 | 242.858 | -21.966 | 93.879 | -66.412 |
| 2012-05-01 06:35:00 | 23700 | 6083553.44 | -3312643.83 | -3501013.74 | 3863.68 | 5930.29 | 1118.13 | 243.029 | -19.946 | 91.447 | -67.788 |
| 2012-05-01 06:36:00 | 23760 | 6305922.26 | -2951925.91 | -3428588.86 | 3546.71 | 6090.57 | 1295.42 | 243.225 | -17.878 | 88.739 | -69.127 |
| 2012-05-01 06:37:00 | 23820 | 6508939.41 | -2582149.13 | -3345642.31 | 3218.79 | 6232.18 | 1468.76 | 243.450 | -15.752 | 85.710 | -70.422 |
| 2012-05-01 06:38:00 | 23880 | 6691978.57 | -2204446.97 | -3252426.95 | 2880.94 | 6354.68 | 1637.63 | 243.707 | -13.561 | 82.305 | -71.663 |
| 2012-05-01 06:39:00 | 23940 | 6854474.53 | -1819977.57 | -3149227.22 | 2534.19 | 6457.69 | 1801.49 | 244.002 | -11.292 | 78.463 | -72.842 |
| 2012-05-01 06:40:00 | 24000 | 6995924.97 | -1429920.28 | -3036358.33 | 2179.60 | 6540.89 | 1959.85 | 244.341 | -8.931 | 74.120 | -73.943 |
| 2012-05-01 06:41:00 | 24060 | 7115892.01 | -1035471.95 | -2914165.27 | 1818.26 | 6604.02 | 2112.22 | 244.733 | -6.461 | 69.210 | -74.951 |
| 2012-05-01 06:42:00 | 24120 | 7214003.67 | -637843.37 | -2783021.81 | 1451.28 | 6646.87 | 2258.12 | 245.189 | -3.860 | 63.680 | -75.847 |
| 2012-05-01 06:43:00 | 24180 | 7289954.96 | -238255.46 | -2643329.33 | 1079.77 | 6669.31 | 2397.11 | 245.724 | -1.099 | 57.501 | -76.609 |
| 2012-05-01 06:44:00 | 24240 | 7343508.97 | 162064.42 | -2495515.63 | 704.89 | 6671.27 | 2528.76 | 246.357 | 1.860 | 50.693 | -77.213 |
| 2012-05-01 06:45:00 | 24300 | 7374497.54 | 561886.27 | -2340033.62 | 327.79 | 6652.72 | 2652.65 | 247.116 | 5.063 | 43.342 | -77.637 |
| 2012-05-01 06:46:00 | 24360 | 7382821.90 | 959981.24 | -2177359.94 | -50.39 | 6613.72 | 2768.42 | 248.040 | 8.576 | 35.614 | -77.862 |
| 2012-05-01 06:47:00 | 24420 | 7368452.98 | 1355125.38 | -2007993.52 | -428.46 | 6554.38 | 2875.69 | 249.188 | 12.484 | 27.737 | -77.877 |
| 2012-05-01 06:48:00 | 24480 | 7331431.55 | 1746103.49 | -1832454.03 | -805.28 | 6474.88 | 2974.13 | 250.649 | 16.903 | 19.970 | -77.682 |
| 2012-05-01 06:49:00 | 24540 | 7271868.15 | 2131712.83 | -1651280.30 | -1179.67 | 6375.47 | 3063.45 | 252.568 | 21.986 | 12.549 | -77.287 |
| 2012-05-01 06:50:00 | 24600 | 7189942.75 | 2510766.82 | -1465028.67 | -1550.49 | 6256.43 | 3143.35 | 255.192 | 27.936 | 5.646 | -76.710 |
| 2012-05-01 06:51:00 | 24660 | 7085904.29 | 2882098.79 | -1274271.27 | -1916.58 | 6118.13 | 3213.60 | 258.983 | 34.988 | 359.359 | -75.973 |
| 2012-05-01 06:52:00 | 24720 | 6960069.90 | 3244565.51 | -1079594.28 | -2276.83 | 5960.99 | 3273.97 | 264.882 | 43.335 | 353.712 | -75.099 |
| 2012-05-01 06:53:00 | 24780 | 6812823.97 | 3597050.83 | -881596.10 | -2630.12 | 5785.50 | 3324.28 | 275.012 | 52.829 | 348.684 | -74.112 |
| 2012-05-01 06:54:00 | 24840 | 6644617.00 | 3938469.06 | -680885.53 | -2975.35 | 5592.19 | 3364.36 | 294.311 | 62.067 | 344.221 | -73.032 |
| 2012-05-01 06:55:00 | 24900 | 6455964.24 | 4267768.42 | -478079.85 | -3311.47 | 5381.64 | 3394.10 | 328.543 | 66.628 | 340.260 | -71.873 |
| 2012-05-01 06:56:00 | 24960 | 6247444.09 | 4583934.25 | -273802.95 | -3637.43 | 5154.51 | 3413.39 | 3.194 | 62.317 | 336.735 | -70.651 |
| 2012-05-01 06:57:00 | 25020 | 6019696.39 | 4885992.24 | -68683.38 | -3952.22 | 4911.50 | 3422.18 | 22.939 | 53.127 | 333.585 | -69.377 |
| 2012-05-01 06:58:00 | 25080 | 5773420.39 | 5173011.39 | 136647.58 | -4254.88 | 4653.35 | 3420.43 | 33.296 | 43.591 | 330.754 | -68.059 |
| 2012-05-01 06:59:00 | 25140 | 5509372.68 | 5444106.98 | 341557.90 | -4544.46 | 4380.85 | 3408.16 | 39.322 | 35.186 | 328.195 | -66.704 |
| 2012-05-01 07:00:00 | 25200 | 5228364.79 | 5698443.28 | 545416.76 | -4820.07 | 4094.85 | 3385.39 | 43.206 | 28.083 | 325.866 | -65.318 |
| 2012-05-01 07:01:00 | 25260 | 4931260.73 | 5935236.21 | 747596.47 | -5080.86 | 3796.22 | 3352.20 | 45.910 | 22.093 | 323.732 | -63.906 |
| 2012-05-01 07:02:00 | 25320 | 4618974.32 | 6153755.73 | 947474.48 | -5326.02 | 3485.89 | 3308.69 | 47.907 | 16.979 | 321.765 | -62.472 |
| 2012-05-01 07:03:00 | 25380 | 4292466.33 | 6353328.15 | 1144435.27 | -5554.79 | 3164.81 | 3254.99 | 49.450 | 12.538 | 319.938 | -61.018 |
| 2012-05-01 07:04:00 | 25440 | 3952741.56 | 6533338.25 | 1337872.25 | -5766.46 | 2833.98 | 3191.26 | 50.686 | 8.613 | 318.231 | -59.546 |
| 2012-05-01 07:05:00 | 25500 | 3600845.68 | 6693231.15 | 1527189.71 | -5960.39 | 2494.41 | 3117.70 | 51.705 | 5.088 | 316.626 | -58.059 |
| 2012-05-01 07:06:00 | 25560 | 3237861.98 | 6832514.10 | 1711804.61 | -6135.96 | 2147.16 | 3034.55 | 52.568 | 1.876 | 315.107 | -56.559 |
| 2012-05-01 07:07:00 | 25620 | 2864908.07 | 6950757.96 | 1891148.42 | -6292.64 | 1793.29 | 2942.05 | 53.314 | -1.089 | 313.661 | -55.046 |
| 2012-05-01 07:08:00 | 25680 | 2483132.34 | 7047598.58 | 2064668.88 | -6429.95 | 1433.90 | 2840.49 | 53.971 | -3.854 | 312.276 | -53.521 |
| 2012-05-01 07:09:00 | 25740 | 2093710.44 | 7122737.91 | 2231831.71 | -6547.45 | 1070.10 | 2730.18 | 54.560 | -6.458 | 310.942 | -51.985 |
| 2012-05-01 07:10:00 | 25800 | 1697841.58 | 7175944.93 | 2392122.30 | -6644.79 | 703.01 | 2611.47 | 55.095 | -8.929 | 309.650 | -50.440 |
| 2012-05-01 07:11:00 | 25860 | 1296744.86 | 7207056.36 | 2545047.26 | -6721.67 | 333.77 | 2484.72 | 55.589 | -11.289 | 308.392 | -48.885 |
| 2012-05-01 07:12:00 | 25920 | 891655.44 | 7215977.17 | 2690135.99 | -6777.85 | -36.48 | 2350.33 | 56.049 | -13.557 | 307.159 | -47.320 |
| 2012-05-01 07:13:00 | 25980 | 483820.71 | 7202680.84 | 2826942.11 | -6813.16 | -406.61 | 2208.71 | 56.484 | -15.746 | 305.946 | -45.746 |
| 2012-05-01 07:14:00 | 26040 | 74496.42 | 7167209.47 | 2955044.87 | -6827.49 | -775.46 | 2060.29 | 56.898 | -17.869 | 304.746 | -44.164 |
| 2012-05-01 07:15:00 | 26100 | -335057.23 | 7109673.61 | 3074050.43 | -6820.80 | -1141.90 | 1905.54 | 57.297 | -19.934 | 303.553 | -42.572 |
| 2012-05-01 07:16:00 | 26160 | -743579.47 | 7030251.89 | 3183593.09 | -6793.12 | -1504.80 | 1744.94 | 57.684 | -21.949 | 302.362 | -40.972 |
| 2012-05-01 07:17:00 | 26220 | -1149812.94 | 6929190.48 | 3283336.39 | -6744.53 | -1863.04 | 1578.98 | 58.063 | -23.921 | 301.166 | -39.362 |
| 2012-05-01 07:18:00 | 26280 | -1552507.51 | 6806802.28 | 3372974.19 | -6675.18 | -2215.51 | 1408.18 | 58.437 | -25.855 | 299.959 | -37.743 |
| 2012-05-01 07:19:00 | 26340 | -1950424.21 | 6663465.93 | 3452231.55 | -6585.31 | -2561.13 | 1233.06 | 58.807 | -27.755 | 298.738 | -36.114 |
| 2012-05-01 07:20:00 | 26400 | -2342339.05 | 6499624.62 | 3520865.65 | -6475.17 | -2898.84 | 1054.16 | 59.178 | -29.625 | 297.494 | -34.476 |
| 2012-05-01 07:21:00 | 26460 | -2727046.79 | 6315784.69 | 3578666.44 | -6345.13 | -3227.58 | 872.04 | 59.550 | -31.469 | 296.223 | -32.827 |
| 2012-05-01 07:22:00 | 26520 | -3103364.70 | 6112514.01 | 3625457.34 | -6195.59 | -3546.36 | 687.26 | 59.925 | -33.289 | 294.918 | -31.167 |
| 2012-05-01 07:23:00 | 26580 | -3470136.19 | 5890440.24 | 3661095.75 | -6027.00 | -3854.19 | 500.39 | 60.307 | -35.088 | 293.572 | -29.495 |
| 2012-05-01 07:24:00 | 26640 | -3826234.36 | 5650248.81 | 3685473.47 | -5839.90 | -4150.13 | 312.00 | 60.696 | -36.867 | 292.179 | -27.812 |
| 2012-05-01 07:25:00 | 26700 | -4170565.51 | 5392680.83 | 3698517.04 | -5634.87 | -4433.26 | 122.68 | 61.095 | -38.629 | 290.729 | -26.116 |
| 2012-05-01 07:26:00 | 26760 | -4502072.47 | 5118530.68 | 3700187.92 | -5412.54 | -4702.73 | -66.99 | 61.506 | -40.375 | 289.214 | -24.406 |
| 2012-05-01 07:27:00 | 26820 | -4819737.86 | 4828643.63 | 3690482.60 | -5173.60 | -4957.69 | -256.43 | 61.930 | -42.106 | 287.626 | -22.683 |
| 2012-05-01 07:28:00 | 26880 | -5122587.21 | 4523913.12 | 3669432.58 | -4918.80 | -5197.38 | -445.05 | 62.371 | -43.824 | 285.951 | -20.946 |
| 2012-05-01 07:29:00 | 26940 | -5409691.95 | 4205278.01 | 3637104.30 | -4648.92 | -5421.06 | -632.28 | 62.831 | -45.529 | 284.180 | -19.193 |
| 2012-05-01 07:30:00 | 27000 | -5680172.21 | 3873719.65 | 3593598.85 | -4364.79 | -5628.05 | -817.53 | 63.312 | -47.222 | 282.297 | -17.425 |
| 2012-05-01 07:31:00 | 27060 | -5933199.55 | 3530258.84 | 3539051.67 | -4067.30 | -5817.71 | -1000.24 | 63.817 | -48.905 | 280.289 | -15.643 |
| 2012-05-01 07:32:00 | 27120 | -6167999.44 | 3175952.65 | 3473632.11 | -3757.37 | -5989.47 | -1179.85 | 64.351 | -50.577 | 278.136 | -13.846 |
| 2012-05-01 07:33:00 | 27180 | -6383853.63 | 2811891.12 | 3397542.89 | -3435.94 | -6142.80 | -1355.80 | 64.916 | -52.239 | 275.820 | -12.035 |
| 2012-05-01 07:34:00 | 27240 | -6580102.34 | 2439193.95 | 3311019.45 | -3104.02 | -6277.25 | -1527.57 | 65.518 | -53.892 | 273.318 | -10.214 |
| 2012-05-01 07:35:00 | 27300 | -6756146.19 | 2059006.94 | 3214329.17 | -2762.62 | -6392.40 | -1694.61 | 66.162 | -55.536 | 270.607 | -8.386 |
| 2012-05-01 07:36:00 | 27360 | -6911448.08 | 1672498.57 | 3107770.61 | -2412.80 | -6487.91 | -1856.43 | 66.854 | -57.171 | 267.659 | -6.558 |
| 2012-05-01 07:37:00 | 27420 | -7045534.74 | 1280856.30 | 2991672.48 | -2055.63 | -6563.49 | -2012.52 | 67.601 | -58.797 | 264.445 | -4.738 |
| 2012-05-01 07:38:00 | 27480 | -7157998.14 | 885282.95 | 2866392.69 | -1692.21 | -6618.91 | -2162.40 | 68.412 | -60.414 | 260.936 | -2.940 |
| 2012-05-01 07:39:00 | 27540 | -7248496.76 | 486993.02 | 2732317.17 | -1323.65 | -6654.02 | -2305.63 | 69.299 | -62.022 | 257.101 | -1.182 |
| 2012-05-01 07:40:00 | 27600 | -7316756.52 | 87208.93 | 2589858.70 | -951.10 | -6668.71 | -2441.76 | 70.273 | -63.621 | 252.914 | 0.513 |
| 2012-05-01 07:41:00 | 27660 | -7362571.59 | -312842.71 | 2439455.64 | -575.69 | -6662.94 | -2570.39 | 71.353 | -65.210 | 248.353 | 2.111 |
| 2012-05-01 07:42:00 | 27720 | -7385805.04 | -711934.87 | 2281570.55 | -198.57 | -6636.73 | -2691.10 | 72.557 | -66.787 | 243.409 | 3.575 |
| 2012-05-01 07:43:00 | 27780 | -7386389.11 | -1108843.84 | 2116688.76 | 179.10 | -6590.18 | -2803.55 | 73.913 | -68.353 | 238.091 | 4.859 |
| 2012-05-01 07:44:00 | 27840 | -7364325.46 | -1502352.96 | 1945316.90 | 556.16 | -6523.44 | -2907.38 | 75.453 | -69.904 | 232.428 | 5.912 |
| 2012-05-01 07:45:00 | 27900 | -7319685.09 | -1891256.37 | 1767981.28 | 931.46 | -6436.70 | -3002.29 | 77.222 | -71.440 | 226.478 | 6.688 |
| 2012-05-01 07:46:00 | 27960 | -7252608.09 | -2274362.64 | 1585226.36 | 1303.86 | -6330.25 | -3087.98 | 79.277 | -72.956 | 220.326 | 7.145 |
| 2012-05-01 07:47:00 | 28020 | -7163303.15 | -2650498.43 | 1397612.99 | 1672.21 | -6204.41 | -3164.20 | 81.695 | -74.447 | 214.076 | 7.258 |
| 2012-05-01 07:48:00 | 28080 | -7052046.91 | -3018512.03 | 1205716.75 | 2035.38 | -6059.58 | -3230.71 | 84.584 | -75.908 | 207.847 | 7.020 |
| 2012-05-01 07:49:00 | 28140 | -6919183.07 | -3377276.88 | 1010126.16 | 2392.28 | -5896.20 | -3287.31 | 88.088 | -77.328 | 201.751 | 6.448 |
| 2012-05-01 07:50:00 | 28200 | -6765121.33 | -3725694.99 | 811440.90 | 2741.80 | -5714.78 | -3333.84 | 92.412 | -78.693 | 195.887 | 5.572 |
| 2012-05-01 07:51:00 | 28260 | -6590336.07 | -4062700.24 | 610269.95 | 3082.88 | -5515.87 | -3370.15 | 97.834 | -79.982 | 190.327 | 4.438 |
| 2012-05-01 07:52:00 | 28320 | -6395364.91 | -4387261.65 | 407229.76 | 3414.49 | -5300.09 | -3396.13 | 104.722 | -81.163 | 185.120 | 3.096 |
| 2012-05-01 07:53:00 | 28380 | -6180807.03 | -4698386.47 | 202942.36 | 3735.61 | -5068.10 | -3411.71 | 113.509 | -82.186 | 180.283 | 1.594 |
| 2012-05-01 07:54:00 | 28440 | -5947321.34 | -4995123.20 | -1966.53 | 4045.26 | -4820.61 | -3416.84 | 124.549 | -82.985 | 175.819 | -0.026 |
| 2012-05-01 07:55:00 | 28500 | -5695624.44 | -5276564.44 | -206869.41 | 4342.49 | -4558.38 | -3411.51 | 137.763 | -83.476 | 171.712 | -1.725 |
| 2012-05-01 07:56:00 | 28560 | -5426488.41 | -5541849.68 | -411138.89 | 4626.41 | -4282.21 | -3395.73 | 152.228 | -83.589 | 167.937 | -3.477 |
| 2012-05-01 07:57:00 | 28620 | -5140738.49 | -5790167.85 | -614149.62 | 4896.15 | -3992.95 | -3369.56 | 166.323 | -83.306 | 164.468 | -5.258 |
| 2012-05-01 07:58:00 | 28680 | -4839250.52 | -6020759.79 | -815280.18 | 5150.88 | -3691.49 | -3333.08 | 178.656 | -82.673 | 161.272 | -7.053 |
| 2012-05-01 07:59:00 | 28740 | -4522948.27 | -6232920.55 | -1013914.96 | 5389.83 | -3378.74 | -3286.39 | 188.706 | -81.770 | 158.320 | -8.851 |
| 2012-05-01 08:00:00 | 28800 | -4192800.67 | -6426001.51 | -1209446.00 | 5612.28 | -3055.66 | -3229.65 | 196.640 | -80.675 | 155.585 | -10.643 |
| 2012-05-01 08:01:00 | 28860 | -3849818.79 | -6599412.32 | -1401274.90 | 5817.53 | -2723.23 | -3163.02 | 202.877 | -79.447 | 153.039 | -12.424 |
| 2012-05-01 08:02:00 | 28920 | -3495052.84 | -6752622.72 | -1588814.59 | 6004.98 | -2382.48 | -3086.71 | 207.829 | -78.128 | 150.659 | -14.191 |
| 2012-05-01 08:03:00 | 28980 | -3129588.92 | -6885164.10 | -1771491.09 | 6174.04 | -2034.44 | -3000.95 | 211.826 | -76.745 | 148.424 | -15.942 |
| 2012-05-01 08:04:00 | 29040 | -2754545.76 | -6996630.95 | -1948745.31 | 6324.21 | -1680.17 | -2906.01 | 215.111 | -75.315 | 146.316 | -17.675 |
| 2012-05-01 08:05:00 | 29100 | -2371071.33 | -7086682.09 | -2120034.69 | 6455.01 | -1320.76 | -2802.18 | 217.860 | -73.850 | 144.316 | -19.390 |
| 2012-05-01 08:06:00 | 29160 | -1980339.30 | -7155041.66 | -2284834.87 | 6566.06 | -957.31 | -2689.76 | 220.199 | -72.359 | 142.411 | -21.087 |
| 2012-05-01 08:07:00 | 29220 | -1583545.57 | -7201500.02 | -2442641.28 | 6657.02 | -590.91 | -2569.11 | 222.221 | -70.847 | 140.587 | -22.767 |
| 2012-05-01 08:08:00 | 29280 | -1181904.55 | -7225914.34 | -2592970.69 | 6727.60 | -222.69 | -2440.59 | 223.995 | -69.318 | 138.833 | -24.429 |
| 2012-05-01 08:09:00 | 29340 | -776645.56 | -7228209.09 | -2735362.63 | 6777.59 | 146.22 | -2304.60 | 225.572 | -67.776 | 137.137 | -26.075 |
| 2012-05-01 08:10:00 | 29400 | -369009.05 | -7208376.22 | -2869380.84 | 6806.83 | 514.71 | -2161.54 | 226.990 | -66.222 | 135.489 | -27.704 |
| 2012-05-01 08:11:00 | 29460 | 39757.15 | -7166475.24 | -2994614.58 | 6815.23 | 881.64 | -2011.85 | 228.281 | -64.658 | 133.882 | -29.317 |
| 2012-05-01 08:12:00 | 29520 | 448401.62 | -7102633.06 | -3110679.85 | 6802.77 | 1245.90 | -1856.00 | 229.467 | -63.085 | 132.307 | -30.914 |
| 2012-05-01 08:13:00 | 29580 | 855673.13 | -7017043.58 | -3217220.65 | 6769.48 | 1606.36 | -1694.45 | 230.568 | -61.503 | 130.757 | -32.496 |
| 2012-05-01 08:14:00 | 29640 | 1260324.45 | -6909967.15 | -3313909.96 | 6715.45 | 1961.94 | -1527.70 | 231.599 | -59.914 | 129.224 | -34.064 |
| 2012-05-01 08:15:00 | 29700 | 1661116.17 | -6781729.81 | -3400450.82 | 6640.86 | 2311.55 | -1356.26 | 232.572 | -58.318 | 127.702 | -35.617 |
| 2012-05-01 08:16:00 | 29760 | 2056820.42 | -6632722.28 | -3476577.23 | 6545.91 | 2654.11 | -1180.64 | 233.498 | -56.715 | 126.185 | -37.155 |
| 2012-05-01 08:17:00 | 29820 | 2446224.64 | -6463398.85 | -3542054.94 | 6430.91 | 2988.57 | -1001.39 | 234.385 | -55.104 | 124.665 | -38.679 |
| 2012-05-01 08:18:00 | 29880 | 2828135.28 | -6274275.98 | -3596682.20 | 6296.19 | 3313.92 | -819.05 | 235.242 | -53.487 | 123.138 | -40.188 |
| 2012-05-01 08:19:00 | 29940 | 3201381.40 | -6065930.78 | -3640290.40 | 6142.17 | 3629.15 | -634.18 | 236.073 | -51.863 | 121.597 | -41.683 |
| 2012-05-01 08:20:00 | 30000 | 3564818.31 | -5838999.27 | -3672744.55 | 5969.30 | 3933.31 | -447.34 | 236.886 | -50.232 | 120.036 | -43.163 |
| 2012-05-01 08:21:00 | 30060 | 3917330.99 | -5594174.49 | -3693943.78 | 5778.12 | 4225.44 | -259.11 | 237.684 | -48.593 | 118.448 | -44.627 |
| 2012-05-01 08:22:00 | 30120 | 4257837.59 | -5332204.37 | -3703821.59 | 5569.20 | 4504.67 | -70.06 | 238.473 | -46.946 | 116.827 | -46.076 |
| 2012-05-01 08:23:00 | 30180 | 4585292.68 | -5053889.54 | -3702346.13 | 5343.18 | 4770.13 | 119.23 | 239.257 | -45.291 | 115.167 | -47.509 |
| 2012-05-01 08:24:00 | 30240 | 4898690.49 | -4760080.88 | -3689520.29 | 5100.74 | 5021.00 | 308.19 | 240.040 | -43.627 | 113.460 | -48.924 |
| 2012-05-01 08:25:00 | 30300 | 5197067.98 | -4451676.97 | -3665381.70 | 4842.63 | 5256.51 | 496.23 | 240.825 | -41.953 | 111.699 | -50.321 |
| 2012-05-01 08:26:00 | 30360 | 5479507.86 | -4129621.33 | -3630002.67 | 4569.62 | 5475.94 | 682.78 | 241.616 | -40.269 | 109.875 | -51.699 |
| 2012-05-01 08:27:00 | 30420 | 5745141.35 | -3794899.64 | -3583489.95 | 4282.56 | 5678.60 | 867.26 | 242.418 | -38.575 | 107.980 | -53.056 |
| 2012-05-01 08:28:00 | 30480 | 5993150.93 | -3448536.68 | -3525984.46 | 3982.31 | 5863.88 | 1049.11 | 243.234 | -36.868 | 106.004 | -54.391 |
| 2012-05-01 08:29:00 | 30540 | 6222772.83 | -3091593.25 | -3457660.86 | 3669.79 | 6031.20 | 1227.77 | 244.067 | -35.148 | 103.938 | -55.702 |
| 2012-05-01 08:30:00 | 30600 | 6433299.42 | -2725162.95 | -3378727.04 | 3345.96 | 6180.03 | 1402.69 | 244.922 | -33.415 | 101.770 | -56.987 |
| 2012-05-01 08:31:00 | 30660 | 6624081.44 | -2350368.82 | -3289423.51 | 3011.81 | 6309.92 | 1573.34 | 245.804 | -31.665 | 99.490 | -58.243 |
| 2012-05-01 08:32:00 | 30720 | 6794529.99 | -1968359.98 | -3190022.69 | 2668.35 | 6420.46 | 1739.18 | 246.717 | -29.899 | 97.083 | -59.467 |
| 2012-05-01 08:33:00 | 30780 | 6944118.38 | -1580308.03 | -3080828.07 | 2316.64 | 6511.30 | 1899.71 | 247.666 | -28.114 | 94.538 | -60.656 |
| 2012-05-01 08:34:00 | 30840 | 7072383.83 | -1187403.55 | -2962173.32 | 1957.77 | 6582.17 | 2054.44 | 248.657 | -26.308 | 91.839 | -61.806 |
| 2012-05-01 08:35:00 | 30900 | 7178928.87 | -790852.41 | -2834421.30 | 1592.82 | 6632.82 | 2202.88 | 249.697 | -24.479 | 88.972 | -62.913 |
| 2012-05-01 08:36:00 | 30960 | 7263422.69 | -391872.06 | -2697962.90 | 1222.91 | 6663.12 | 2344.57 | 250.794 | -22.624 | 85.923 | -63.972 |
| 2012-05-01 08:37:00 | 31020 | 7325602.10 | 8312.14 | -2553215.94 | 849.19 | 6672.94 | 2479.09 | 251.956 | -20.741 | 82.677 | -64.977 |
| 2012-05-01 08:38:00 | 31080 | 7365272.47 | 408470.77 | -2400623.85 | 472.80 | 6662.26 | 2606.01 | 253.193 | -18.826 | 79.221 | -65.923 |
| 2012-05-01 08:39:00 | 31140 | 7382308.33 | 807374.08 | -2240654.32 | 94.90 | 6631.11 | 2724.94 | 254.517 | -16.875 | 75.544 | -66.801 |
| 2012-05-01 08:40:00 | 31200 | 7376653.81 | 1203795.81 | -2073797.89 | -283.35 | 6579.56 | 2835.52 | 255.942 | -14.885 | 71.639 | -67.606 |
| 2012-05-01 08:41:00 | 31260 | 7348322.87 | 1596516.93 | -1900566.46 | -660.78 | 6507.78 | 2937.39 | 257.484 | -12.849 | 67.504 | -68.329 |
| 2012-05-01 08:42:00 | 31320 | 7297399.25 | 1984329.48 | -1721491.68 | -1036.24 | 6415.99 | 3030.24 | 259.164 | -10.764 | 63.143 | -68.962 |
| 2012-05-01 08:43:00 | 31380 | 7224036.33 | 2366040.25 | -1537123.37 | -1408.57 | 6304.44 | 3113.79 | 261.004 | -8.622 | 58.570 | -69.498 |
| 2012-05-01 08:44:00 | 31440 | 7128456.63 | 2740474.47 | -1348027.80 | -1776.61 | 6173.50 | 3187.78 | 263.035 | -6.419 | 53.806 | -69.929 |
| 2012-05-01 08:45:00 | 31500 | 7010951.19 | 3106479.49 | -1154785.98 | -2139.24 | 6023.54 | 3251.97 | 265.290 | -4.147 | 48.886 | -70.248 |
| 2012-05-01 08:46:00 | 31560 | 6871878.69 | 3462928.36 | -957991.83 | -2495.33 | 5855.04 | 3306.16 | 267.814 | -1.801 | 43.851 | -70.451 |
| 2012-05-01 08:47:00 | 31620 | 6711664.40 | 3808723.26 | -758250.35 | -2843.78 | 5668.50 | 3350.18 | 270.658 | 0.623 | 38.752 | -70.535 |
| 2012-05-01 08:48:00 | 31680 | 6530798.86 | 4142799.01 | -556175.82 | -3183.53 | 5464.50 | 3383.91 | 273.888 | 3.126 | 33.641 | -70.497 |
| 2012-05-01 08:49:00 | 31740 | 6329836.44 | 4464126.32 | -352389.80 | -3513.51 | 5243.66 | 3407.22 | 277.579 | 5.701 | 28.574 | -70.340 |
| 2012-05-01 08:50:00 | 31800 | 6109393.58 | 4771715.00 | -147519.29 | -3832.71 | 5006.66 | 3420.05 | 281.823 | 8.333 | 23.602 | -70.067 |
| 2012-05-01 08:51:00 | 31860 | 5870146.98 | 5064617.08 | 57805.25 | -4140.14 | 4754.23 | 3422.35 | 286.722 | 10.984 | 18.769 | -69.682 |
| 2012-05-01 08:52:00 | 31920 | 5612831.46 | 5341929.72 | 262951.85 | -4434.85 | 4487.15 | 3414.12 | 292.379 | 13.591 | 14.110 | -69.192 |
| 2012-05-01 08:53:00 | 31980 | 5338237.74 | 5602798.07 | 467288.99 | -4715.93 | 4206.23 | 3395.37 | 298.885 | 16.052 | 9.650 | -68.604 |
| 2012-05-01 08:54:00 | 32040 | 5047209.99 | 5846417.90 | 670187.54 | -4982.51 | 3912.34 | 3366.18 | 306.281 | 18.223 | 5.404 | -67.927 |
| 2012-05-01 08:55:00 | 32100 | 4740643.22 | 6072038.15 | 871022.76 | -5233.76 | 3606.40 | 3326.61 | 314.515 | 19.921 | 1.378 | -67.168 |
| 2012-05-01 08:56:00 | 32160 | 4419480.51 | 6278963.23 | 1069176.19 | -5468.91 | 3289.33 | 3276.81 | 323.402 | 20.964 | 357.572 | -66.337 |
| 2012-05-01 08:57:00 | 32220 | 4084710.12 | 6466555.22 | 1264037.59 | -5687.24 | 2962.13 | 3216.91 | 332.616 | 21.216 | 353.979 | -65.439 |
| 2012-05-01 08:58:00 | 32280 | 3737362.38 | 6634235.85 | 1455006.86 | -5888.05 | 2625.79 | 3147.10 | 341.751 | 20.644 | 350.588 | -64.484 |
| 2012-05-01 08:59:00 | 32340 | 3378506.56 | 6781488.32 | 1641495.86 | -6070.74 | 2281.36 | 3067.60 | 350.421 | 19.325 | 347.388 | -63.476 |
| 2012-05-01 09:00:00 | 32400 | 3009247.46 | 6907858.89 | 1822930.29 | -6234.74 | 1929.91 | 2978.66 | 358.354 | 17.424 | 344.364 | -62.422 |
| 2012-05-01 09:01:00 | 32460 | 2630722.09 | 7012958.31 | 1998751.42 | -6379.54 | 1572.51 | 2880.54 | 5.422 | 15.127 | 341.501 | -61.327 |
| 2012-05-01 09:02:00 | 32520 | 2244096.04 | 7096463.01 | 2168417.88 | -6504.69 | 1210.27 | 2773.56 | 11.618 | 12.603 | 338.784 | -60.196 |
| 2012-05-01 09:03:00 | 32580 | 1850559.93 | 7158116.15 | 2331407.31 | -6609.81 | 844.31 | 2658.03 | 17.007 | 9.981 | 336.198 | -59.034 |
| 2012-05-01 09:04:00 | 32640 | 1451325.65 | 7197728.36 | 2487217.98 | -6694.58 | 475.76 | 2534.33 | 21.688 | 7.346 | 333.731 | -57.842 |
| 2012-05-01 09:05:00 | 32700 | 1047622.65 | 7215178.33 | 2635370.36 | -6758.73 | 105.76 | 2402.82 | 25.765 | 4.748 | 331.370 | -56.626 |
| 2012-05-01 09:06:00 | 32760 | 640694.08 | 7210413.24 | 2775408.61 | -6802.07 | -264.55 | 2263.92 | 29.338 | 2.217 | 329.101 | -55.387 |
| 2012-05-01 09:07:00 | 32820 | 231792.93 | 7183448.83 | 2906901.98 | -6824.47 | -634.03 | 2118.06 | 32.491 | -0.237 | 326.914 | -54.128 |
| 2012-05-01 09:08:00 | 32880 | -177821.83 | 7134369.38 | 3029446.13 | -6825.85 | -1001.53 | 1965.69 | 35.296 | -2.610 | 324.799 | -52.852 |
| 2012-05-01 09:09:00 | 32940 | -586889.19 | 7063327.45 | 3142664.43 | -6806.23 | -1365.92 | 1807.28 | 37.812 | -4.905 | 322.745 | -51.559 |
| 2012-05-01 09:10:00 | 33000 | -994150.02 | 6970543.34 | 3246209.05 | -6765.65 | -1726.08 | 1643.32 | 40.088 | -7.126 | 320.745 | -50.253 |
| 2012-05-01 09:11:00 | 33060 | -1398350.95 | 6856304.44 | 3339762.09 | -6704.26 | -2080.90 | 1474.32 | 42.165 | -9.280 | 318.788 | -48.934 |
| 2012-05-01 09:12:00 | 33120 | -1798248.28 | 6720964.26 | 3423036.54 | -6622.24 | -2429.28 | 1300.79 | 44.074 | -11.373 | 316.868 | -47.604 |
| 2012-05-01 09:13:00 | 33180 | -2192611.80 | 6564941.36 | 3495777.13 | -6519.85 | -2770.15 | 1123.28 | 45.844 | -13.410 | 314.978 | -46.264 |
| 2012-05-01 09:14:00 | 33240 | -2580228.63 | 6388718.00 | 3557761.16 | -6397.41 | -3102.45 | 942.33 | 47.495 | -15.397 | 313.109 | -44.915 |
| 2012-05-01 09:15:00 | 33300 | -2959906.96 | 6192838.63 | 3608799.11 | -6255.30 | -3425.18 | 758.50 | 49.047 | -17.340 | 311.256 | -43.559 |
| 2012-05-01 09:16:00 | 33360 | -3330479.73 | 5977908.17 | 3648735.30 | -6093.97 | -3737.33 | 572.37 | 50.516 | -19.242 | 309.412 | -42.196 |
| 2012-05-01 09:17:00 | 33420 | -3690808.23 | 5744590.11 | 3677448.26 | -5913.91 | -4037.94 | 384.49 | 51.915 | -21.109 | 307.571 | -40.826 |
| 2012-05-01 09:18:00 | 33480 | -4039785.59 | 5493604.43 | 3694851.15 | -5715.69 | -4326.10 | 195.46 | 53.255 | -22.942 | 305.726 | -39.452 |
| 2012-05-01 09:19:00 | 33540 | -4376340.24 | 5225725.35 | 3700892.01 | -5499.93 | -4600.91 | 5.85 | 54.547 | -24.746 | 303.872 | -38.074 |
| 2012-05-01 09:20:00 | 33600 | -4699439.13 | 4941778.90 | 3695553.86 | -5267.28 | -4861.54 | -183.74 | 55.799 | -26.523 | 302.003 | -36.693 |
| 2012-05-01 09:21:00 | 33660 | -5008090.95 | 4642640.33 | 3678854.79 | -5018.48 | -5107.19 | -372.75 | 57.018 | -28.275 | 300.113 | -35.309 |
| 2012-05-01 09:22:00 | 33720 | -5301349.16 | 4329231.40 | 3650847.81 | -4754.29 | -5337.10 | -560.58 | 58.212 | -30.004 | 298.195 | -33.924 |
| 2012-05-01 09:23:00 | 33780 | -5578314.86 | 4002517.51 | 3611620.76 | -4475.54 | -5550.57 | -746.65 | 59.387 | -31.713 | 296.244 | -32.539 |
| 2012-05-01 09:24:00 | 33840 | -5838139.55 | 3663504.66 | 3561295.93 | -4183.07 | -5746.96 | -930.41 | 60.549 | -33.402 | 294.253 | -31.154 |
| 2012-05-01 09:25:00 | 33900 | -6080027.71 | 3313236.38 | 3500029.70 | -3877.81 | -5925.66 | -1111.27 | 61.703 | -35.073 | 292.215 | -29.771 |
| 2012-05-01 09:26:00 | 33960 | -6303239.22 | 2952790.44 | 3428012.05 | -3560.68 | -6086.13 | -1288.70 | 62.854 | -36.727 | 290.124 | -28.391 |
| 2012-05-01 09:27:00 | 34020 | -6507091.58 | 2583275.56 | 3345465.92 | -3232.67 | -6227.88 | -1462.13 | 64.008 | -38.365 | 287.973 | -27.017 |
| 2012-05-01 09:28:00 | 34080 | -6690962.00 | 2205827.93 | 3252646.54 | -2894.79 | -6350.49 | -1631.05 | 65.168 | -39.987 | 285.755 | -25.648 |
| 2012-05-01 09:29:00 | 34140 | -6854289.26 | 1821607.74 | 3149840.59 | -2548.07 | -6453.57 | -1794.93 | 66.341 | -41.595 | 283.461 | -24.289 |
| 2012-05-01 09:30:00 | 34200 | -6996575.39 | 1431795.60 | 3037365.33 | -2193.60 | -6536.84 | -1953.28 | 67.531 | -43.188 | 281.085 | -22.940 |
| 2012-05-01 09:31:00 | 34260 | -7117387.17 | 1037588.85 | 2915567.57 | -1832.44 | -6600.02 | -2105.60 | 68.744 | -44.768 | 278.618 | -21.605 |
| 2012-05-01 09:32:00 | 34320 | -7216357.39 | 640197.96 | 2784822.61 | -1465.73 | -6642.95 | -2251.44 | 69.984 | -46.333 | 276.052 | -20.288 |
| 2012-05-01 09:33:00 | 34380 | -7293185.96 | 240842.74 | 2645533.08 | -1094.58 | -6665.49 | -2390.35 | 71.257 | -47.886 | 273.379 | -18.991 |
| 2012-05-01 09:34:00 | 34440 | -7347640.78 | -159251.36 | 2498127.68 | -720.13 | -6667.57 | -2521.90 | 72.570 | -49.424 | 270.590 | -17.720 |
| 2012-05-01 09:35:00 | 34500 | -7379558.39 | -558857.03 | 2343059.83 | -343.53 | -6649.21 | -2645.70 | 73.930 | -50.948 | 267.678 | -16.480 |
| 2012-05-01 09:36:00 | 34560 | -7388844.42 | -956748.83 | 2180806.28 | 34.06 | -6610.46 | -2761.37 | 75.344 | -52.458 | 264.636 | -15.278 |
| 2012-05-01 09:37:00 | 34620 | -7375473.90 | -1351706.96 | 2011865.68 | 411.50 | -6551.45 | -2868.55 | 76.819 | -53.953 | 261.458 | -14.119 |
| 2012-05-01 09:38:00 | 34680 | -7339491.19 | -1742520.98 | 1836756.96 | 787.61 | -6472.36 | -2966.92 | 78.366 | -55.433 | 258.138 | -13.013 |
| 2012-05-01 09:39:00 | 34740 | -7281009.90 | -2127993.47 | 1656017.80 | 1161.26 | -6373.44 | -3056.18 | 79.994 | -56.895 | 254.673 | -11.968 |
| 2012-05-01 09:40:00 | 34800 | -7200212.44 | -2506943.73 | 1470202.97 | 1531.29 | -6255.01 | -3136.06 | 81.717 | -58.340 | 251.062 | -10.994 |
| 2012-05-01 09:41:00 | 34860 | -7097349.45 | -2878211.32 | 1279882.59 | 1896.59 | -6117.42 | -3206.33 | 83.546 | -59.766 | 247.309 | -10.101 |
| 2012-05-01 09:42:00 | 34920 | -6972738.99 | -3240659.62 | 1085640.43 | 2256.03 | -5961.11 | -3266.76 | 85.497 | -61.170 | 243.417 | -9.302 |
| 2012-05-01 09:43:00 | 34980 | -6826765.56 | -3593179.26 | 888072.09 | 2608.51 | -5786.55 | -3317.17 | 87.589 | -62.550 | 239.397 | -8.606 |
| 2012-05-01 09:44:00 | 35040 | -6659878.86 | -3934691.49 | 687783.19 | 2952.96 | -5594.29 | -3357.42 | 89.841 | -63.904 | 235.263 | -8.024 |
| 2012-05-01 09:45:00 | 35100 | -6472592.41 | -4264151.47 | 485387.55 | 3288.33 | -5384.91 | -3387.38 | 92.277 | -65.228 | 231.032 | -7.566 |
| 2012-05-01 09:46:00 | 35160 | -6265481.96 | -4580551.40 | 281505.28 | 3613.59 | -5159.06 | -3406.96 | 94.924 | -66.517 | 226.725 | -7.240 |
| 2012-05-01 09:47:00 | 35220 | -6039183.71 | -4882923.59 | 76760.90 | 3927.76 | -4917.44 | -3416.11 | 97.812 | -67.768 | 222.367 | -7.052 |
| 2012-05-01 09:48:00 | 35280 | -5794392.35 | -5170343.37 | -128218.55 | 4229.87 | -4660.78 | -3414.80 | 100.975 | -68.972 | 217.986 | -7.005 |
| 2012-05-01 09:49:00 | 35340 | -5531858.92 | -5441931.91 | -332805.41 | 4519.01 | -4389.87 | -3403.03 | 104.450 | -70.124 | 213.608 | -7.100 |
| 2012-05-01 09:50:00 | 35400 | -5252388.55 | -5696858.85 | -536373.33 | 4794.29 | -4105.53 | -3380.84 | 108.277 | -71.213 | 209.262 | -7.333 |
| 2012-05-01 09:51:00 | 35460 | -4956837.94 | -5934344.82 | -738299.14 | 5054.88 | -3808.65 | -3348.30 | 112.496 | -72.230 | 204.972 | -7.700 |
| 2012-05-01 09:52:00 | 35520 | -4646112.80 | -6153663.79 | -937964.77 | 5299.98 | -3500.12 | -3305.52 | 117.143 | -73.161 | 200.762 | -8.194 |
| 2012-05-01 09:53:00 | 35580 | -4321165.03 | -6354145.26 | -1134759.13 | 5528.85 | -3180.89 | -3252.62 | 122.245 | -73.992 | 196.650 | -8.804 |
| 2012-05-01 09:54:00 | 35640 | -3982989.89 | -6535176.28 | -1328079.92 | 5740.78 | -2851.94 | -3189.76 | 127.811 | -74.709 | 192.653 | -9.521 |
| 2012-05-01 09:55:00 | 35700 | -3632622.93 | -6696203.32 | -1517335.49 | 5935.14 | -2514.26 | -3117.15 | 133.824 | -75.294 | 188.780 | -10.332 |
| 2012-05-01 09:56:00 | 35760 | -3271136.85 | -6836733.92 | -1701946.65 | 6111.32 | -2168.90 | -3034.99 | 140.230 | -75.731 | 185.039 | -11.228 |
| 2012-05-01 09:57:00 | 35820 | -2899638.25 | -6956338.17 | -1881348.37 | 6268.80 | -1816.90 | -2943.54 | 146.938 | -76.009 | 181.433 | -12.196 |
| 2012-05-01 09:58:00 | 35880 | -2519264.27 | -7054650.05 | -2054991.55 | 6407.10 | -1459.33 | -2843.09 | 153.818 | -76.116 | 177.963 | -13.226 |
| 2012-05-01 09:59:00 | 35940 | -2131179.13 | -7131368.52 | -2222344.66 | 6525.78 | -1097.30 | -2733.93 | 160.716 | -76.051 | 174.626 | -14.309 |
| 2012-05-01 10:00:00 | 36000 | -1736570.62 | -7186258.41 | -2382895.33 | 6624.49 | -731.90 | -2616.40 | 167.478 | -75.815 | 171.418 | -15.437 |
| 2012-05-01 10:01:00 | 36060 | -1336646.48 | -7219151.17 | -2536151.96 | 6702.92 | -364.25 | -2490.85 | 173.970 | -75.417 | 168.332 | -16.601 |
| 2012-05-01 10:02:00 | 36120 | -932630.73 | -7229945.39 | -2681645.18 | 6760.84 | 4.53 | -2357.68 | 180.090 | -74.870 | 165.363 | -17.794 |
| 2012-05-01 10:03:00 | 36180 | -525760.00 | -7218607.08 | -2818929.25 | 6798.06 | 373.32 | -2217.29 | 185.778 | -74.189 | 162.502 | -19.012 |
| 2012-05-01 10:04:00 | 36240 | -117279.72 | -7185169.82 | -2947583.50 | 6814.47 | 740.98 | -2070.09 | 191.010 | -73.392 | 159.740 | -20.249 |
| 2012-05-01 10:05:00 | 36300 | 291559.60 | -7129734.67 | -3067213.50 | 6810.03 | 1106.39 | -1916.55 | 195.793 | -72.494 | 157.071 | -21.500 |
| 2012-05-01 10:06:00 | 36360 | 699506.22 | -7052469.85 | -3177452.37 | 6784.72 | 1468.45 | -1757.14 | 200.149 | -71.510 | 154.486 | -22.763 |
| 2012-05-01 10:07:00 | 36420 | 1105310.92 | -6953610.29 | -3277961.84 | 6738.65 | 1826.04 | -1592.32 | 204.115 | -70.452 | 151.976 | -24.033 |
| 2012-05-01 10:08:00 | 36480 | 1507730.85 | -6833456.88 | -3368433.30 | 6671.93 | 2178.06 | -1422.62 | 207.729 | -69.333 | 149.533 | -25.308 |
| 2012-05-01 10:09:00 | 36540 | 1905533.24 | -6692375.64 | -3448588.74 | 6584.77 | 2523.45 | -1248.54 | 211.031 | -68.161 | 147.151 | -26.586 |
| 2012-05-01 10:10:00 | 36600 | 2297499.23 | -6530796.61 | -3518181.63 | 6477.43 | 2861.15 | -1070.62 | 214.060 | -66.944 | 144.822 | -27.864 |
| 2012-05-01 10:11:00 | 36660 | 2682427.53 | -6349212.53 | -3576997.66 | 6350.24 | 3190.12 | -889.41 | 216.850 | -65.688 | 142.538 | -29.142 |
| 2012-05-01 10:12:00 | 36720 | 3059138.09 | -6148177.41 | -3624855.43 | 6203.57 | 3509.35 | -705.44 | 219.433 | -64.400 | 140.292 | -30.417 |
| 2012-05-01 10:13:00 | 36780 | 3426475.72 | -5928304.87 | -3661606.99 | 6037.88 | 3817.87 | -519.29 | 221.836 | -63.084 | 138.080 | -31.687 |
| 2012-05-01 10:14:00 | 36840 | 3783313.60 | -5690266.29 | -3687138.30 | 5853.67 | 4114.73 | -331.53 | 224.084 | -61.743 | 135.893 | -32.952 |
| 2012-05-01 10:15:00 | 36900 | 4128556.73 | -5434788.77 | -3701369.65 | 5651.49 | 4399.02 | -142.72 | 226.196 | -60.381 | 133.726 | -34.210 |
| 2012-05-01 10:16:00 | 36960 | 4461145.33 | -5162652.99 | -3704255.85 | 5431.96 | 4669.86 | 46.55 | 228.193 | -59.000 | 131.573 | -35.461 |
| 2012-05-01 10:17:00 | 37020 | 4780058.00 | -4874690.83 | -3695786.45 | 5195.74 | 4926.43 | 235.70 | 230.089 | -57.601 | 129.429 | -36.702 |
| 2012-05-01 10:18:00 | 37080 | 5084314.95 | -4571782.84 | -3675985.73 | 4943.56 | 5167.93 | 424.16 | 231.899 | -56.188 | 127.288 | -37.934 |
| 2012-05-01 10:19:00 | 37140 | 5372980.97 | -4254855.64 | -3644912.71 | 4676.18 | 5393.62 | 611.35 | 233.636 | -54.761 | 125.145 | -39.155 |
| 2012-05-01 10:20:00 | 37200 | 5645168.33 | -3924879.06 | -3602660.95 | 4394.41 | 5602.80 | 796.69 | 235.309 | -53.321 | 122.994 | -40.363 |
| 2012-05-01 10:21:00 | 37260 | 5900039.50 | -3582863.21 | -3549358.28 | 4099.12 | 5794.82 | 979.62 | 236.930 | -51.870 | 120.829 | -41.558 |
| 2012-05-01 10:22:00 | 37320 | 6136809.80 | -3229855.44 | -3485166.46 | 3791.20 | 5969.10 | 1159.57 | 238.507 | -50.409 | 118.647 | -42.739 |
| 2012-05-01 10:23:00 | 37380 | 6354749.76 | -2866937.15 | -3410280.69 | 3471.60 | 6125.09 | 1335.99 | 240.047 | -48.937 | 116.441 | -43.904 |
| 2012-05-01 10:24:00 | 37440 | 6553187.46 | -2495220.47 | -3324929.03 | 3141.29 | 6262.30 | 1508.34 | 241.558 | -47.456 | 114.207 | -45.053 |
| 2012-05-01 10:25:00 | 37500 | 6731510.57 | -2115844.92 | -3229371.73 | 2801.28 | 6380.32 | 1676.09 | 243.046 | -45.965 | 111.938 | -46.183 |
| 2012-05-01 10:26:00 | 37560 | 6889168.32 | -1729973.88 | -3123900.46 | 2452.62 | 6478.76 | 1838.72 | 244.519 | -44.466 | 109.631 | -47.294 |
| 2012-05-01 10:27:00 | 37620 | 7025673.19 | -1338791.06 | -3008837.41 | 2096.37 | 6557.33 | 1995.74 | 245.981 | -42.959 | 107.279 | -48.383 |
| 2012-05-01 10:28:00 | 37680 | 7140602.49 | -943496.91 | -2884534.34 | 1733.62 | 6615.77 | 2146.64 | 247.439 | -41.443 | 104.878 | -49.450 |
| 2012-05-01 10:29:00 | 37740 | 7233599.65 | -545304.87 | -2751371.54 | 1365.48 | 6653.90 | 2290.98 | 248.898 | -39.918 | 102.421 | -50.493 |
| 2012-05-01 10:30:00 | 37800 | 7304375.41 | -145437.71 | -2609756.62 | 993.09 | 6671.59 | 2428.31 | 250.364 | -38.386 | 99.905 | -51.509 |
| 2012-05-01 10:31:00 | 37860 | 7352708.73 | 254876.28 | -2460123.32 | 617.59 | 6668.79 | 2558.19 | 251.841 | -36.845 | 97.324 | -52.497 |
| 2012-05-01 10:32:00 | 37920 | 7378447.50 | 654407.01 | -2302930.19 | 240.14 | 6645.49 | 2680.24 | 253.336 | -35.295 | 94.673 | -53.455 |
| 2012-05-01 10:33:00 | 37980 | 7381509.09 | 1051926.43 | -2138659.17 | -138.12 | 6601.76 | 2794.06 | 254.854 | -33.737 | 91.946 | -54.380 |
| 2012-05-01 10:34:00 | 38040 | 7361880.64 | 1446212.30 | -1967814.12 | -516.01 | 6537.73 | 2899.32 | 256.401 | -32.170 | 89.141 | -55.269 |
| 2012-05-01 10:35:00 | 38100 | 7319619.09 | 1836051.96 | -1790919.32 | -892.36 | 6453.59 | 2995.67 | 257.984 | -30.595 | 86.252 | -56.121 |
| 2012-05-01 10:36:00 | 38160 | 7254851.13 | 2220246.08 | -1608517.80 | -1266.03 | 6349.59 | 3082.83 | 259.609 | -29.010 | 83.277 | -56.933 |
| 2012-05-01 10:37:00 | 38220 | 7167772.78 | 2597612.39 | -1421169.71 | -1635.85 | 6226.06 | 3160.51 | 261.283 | -27.416 | 80.212 | -57.701 |
| 2012-05-01 10:38:00 | 38280 | 7058648.87 | 2966989.32 | -1229450.61 | -2000.69 | 6083.35 | 3228.49 | 263.013 | -25.812 | 77.056 | -58.423 |
| 2012-05-01 10:39:00 | 38340 | 6927812.20 | 3327239.63 | -1033949.68 | -2359.42 | 5921.91 | 3286.54 | 264.809 | -24.199 | 73.807 | -59.096 |
| 2012-05-01 10:40:00 | 38400 | 6775662.63 | 3677253.90 | -835267.89 | -2710.94 | 5742.24 | 3334.49 | 266.679 | -22.576 | 70.468 | -59.717 |
| 2012-05-01 10:41:00 | 38460 | 6602665.79 | 4015954.01 | -634016.16 | -3054.15 | 5544.87 | 3372.18 | 268.634 | -20.944 | 67.038 | -60.282 |
| 2012-05-01 10:42:00 | 38520 | 6409351.72 | 4342296.51 | -430813.50 | -3388.00 | 5330.42 | 3399.50 | 270.685 | -19.303 | 63.523 | -60.789 |
| 2012-05-01 10:43:00 | 38580 | 6196313.25 | 4655275.82 | -226285.04 | -3711.46 | 5099.54 | 3416.36 | 272.844 | -17.652 | 59.927 | -61.236 |
| 2012-05-01 10:44:00 | 38640 | 5964204.16 | 4953927.40 | -21060.15 | -4023.53 | 4852.95 | 3422.71 | 275.125 | -15.995 | 56.257 | -61.619 |
| 2012-05-01 10:45:00 | 38700 | 5713737.21 | 5237330.74 | 184229.55 | -4323.23 | 4591.40 | 3418.52 | 277.544 | -14.331 | 52.523 | -61.936 |
| 2012-05-01 10:46:00 | 38760 | 5445681.94 | 5504612.24 | 388952.15 | -4609.65 | 4315.69 | 3403.81 | 280.116 | -12.664 | 48.735 | -62.186 |
| 2012-05-01 10:47:00 | 38820 | 5160862.28 | 5754947.91 | 592477.35 | -4881.90 | 4026.68 | 3378.62 | 282.861 | -10.998 | 44.905 | -62.367 |
| 2012-05-01 10:48:00 | 38880 | 4860154.04 | 5987565.99 | 794178.51 | -5139.14 | 3725.26 | 3343.03 | 285.799 | -9.337 | 41.045 | -62.477 |
| 2012-05-01 10:49:00 | 38940 | 4544482.18 | 6201749.32 | 993434.51 | -5380.56 | 3412.35 | 3297.14 | 288.949 | -7.689 | 37.171 | -62.517 |
| 2012-05-01 10:50:00 | 39000 | 4214817.96 | 6396837.59 | 1189631.75 | -5605.43 | 3088.92 | 3241.09 | 292.334 | -6.063 | 33.296 | -62.486 |
| 2012-05-01 10:51:00 | 39060 | 3872175.94 | 6572229.40 | 1382166.00 | -5813.04 | 2755.97 | 3175.07 | 295.976 | -4.472 | 29.434 | -62.385 |
| 2012-05-01 10:52:00 | 39120 | 3517610.79 | 6727384.14 | 1570444.31 | -6002.76 | 2414.53 | 3099.26 | 299.893 | -2.933 | 25.599 | -62.216 |
| 2012-05-01 10:53:00 | 39180 | 3152214.10 | 6861823.67 | 1753886.84 | -6174.00 | 2065.64 | 3013.92 | 304.102 | -1.466 | 21.804 | -61.979 |
| 2012-05-01 10:54:00 | 39240 | 2777110.90 | 6975133.83 | 1931928.70 | -6326.23 | 1710.39 | 2919.29 | 308.614 | -0.097 | 18.059 | -61.677 |
| 2012-05-01 10:55:00 | 39300 | 2393456.23 | 7066965.69 | 2104021.63 | -6458.98 | 1349.88 | 2815.67 | 313.427 | 1.143 | 14.375 | -61.313 |
| 2012-05-01 10:56:00 | 39360 | 2002431.54 | 7137036.67 | 2269635.78 | -6571.83 | 985.22 | 2703.38 | 318.531 | 2.223 | 10.758 | -60.889 |
| 2012-05-01 10:57:00 | 39420 | 1605240.99 | 7185131.41 | 2428261.30 | -6664.45 | 617.53 | 2582.78 | 323.896 | 3.107 | 7.217 | -60.409 |
| 2012-05-01 10:58:00 | 39480 | 1203107.74 | 7211102.43 | 2579409.97 | -6736.55 | 247.95 | 2454.22 | 329.478 | 3.763 | 3.754 | -59.875 |
| 2012-05-01 10:59:00 | 39540 | 797270.13 | 7214870.55 | 2722616.68 | -6787.90 | -122.38 | 2318.11 | 335.214 | 4.164 | 0.373 | -59.291 |
| 2012-05-01 11:00:00 | 39600 | 388977.83 | 7196425.19 | 2857440.86 | -6818.35 | -492.31 | 2174.88 | 341.029 | 4.292 | 357.074 | -58.659 |
| 2012-05-01 11:01:00 | 39660 | -20512.01 | 7155824.32 | 2983467.90 | -6827.81 | -860.70 | 2024.95 | 346.839 | 4.142 | 353.859 | -57.984 |
| 2012-05-01 11:02:00 | 39720 | -429938.73 | 7093194.34 | 3100310.38 | -6816.25 | -1226.42 | 1868.80 | 352.564 | 3.722 | 350.724 | -57.269 |
| 2012-05-01 11:03:00 | 39780 | -838041.99 | 7008729.60 | 3207609.27 | -6783.71 | -1588.34 | 1706.91 | 358.130 | 3.050 | 347.668 | -56.516 |
| 2012-05-01 11:04:00 | 39840 | -1243565.76 | 6902691.81 | 3305035.07 | -6730.29 | -1945.34 | 1539.78 | 3.476 | 2.156 | 344.689 | -55.729 |
| 2012-05-01 11:05:00 | 39900 | -1645262.15 | 6775409.23 | 3392288.78 | -6656.16 | -2296.32 | 1367.93 | 8.561 | 1.072 | 341.782 | -54.911 |
| 2012-05-01 11:06:00 | 39960 | -2041895.33 | 6627275.57 | 3469102.86 | -6561.56 | -2640.20 | 1191.89 | 13.358 | -0.168 | 338.942 | -54.064 |
| 2012-05-01 11:07:00 | 40020 | -2432245.32 | 6458748.79 | 3535242.00 | -6446.77 | -2975.92 | 1012.19 | 17.859 | -1.529 | 336.167 | -53.191 |
| 2012-05-01 11:08:00 | 40080 | -2815111.78 | 6270349.67 | 3590503.87 | -6312.17 | -3302.44 | 829.40 | 22.066 | -2.984 | 333.450 | -52.294 |
| 2012-05-01 11:09:00 | 40140 | -3189317.71 | 6062660.12 | 3634719.74 | -6158.17 | -3618.76 | 644.09 | 25.989 | -4.507 | 330.787 | -51.376 |
| 2012-05-01 11:10:00 | 40200 | -3553713.08 | 5836321.35 | 3667754.95 | -5985.24 | -3923.92 | 456.81 | 29.645 | -6.077 | 328.173 | -50.439 |
| 2012-05-01 11:11:00 | 40260 | -3907178.40 | 5592031.93 | 3689509.32 | -5793.92 | -4216.97 | 268.15 | 33.055 | -7.679 | 325.603 | -49.485 |
| 2012-05-01 11:12:00 | 40320 | -4248628.14 | 5330545.52 | 3699917.48 | -5584.82 | -4497.01 | 78.70 | 36.241 | -9.300 | 323.072 | -48.515 |
| 2012-05-01 11:13:00 | 40380 | -4577014.08 | 5052668.54 | 3698949.01 | -5358.58 | -4763.18 | -110.97 | 39.223 | -10.932 | 320.575 | -47.532 |
| 2012-05-01 11:14:00 | 40440 | -4891328.51 | 4759257.66 | 3686608.52 | -5115.89 | -5014.67 | -300.27 | 42.023 | -12.566 | 318.108 | -46.538 |
| 2012-05-01 11:15:00 | 40500 | -5190607.35 | 4451217.11 | 3662935.64 | -4857.52 | -5250.71 | -488.62 | 44.660 | -14.199 | 315.665 | -45.534 |
| 2012-05-01 11:16:00 | 40560 | -5473933.06 | 4129495.89 | 3628004.86 | -4584.26 | -5470.58 | -675.44 | 47.153 | -15.826 | 313.242 | -44.521 |
| 2012-05-01 11:17:00 | 40620 | -5740437.48 | 3795084.76 | 3581925.27 | -4296.95 | -5673.60 | -860.15 | 49.518 | -17.445 | 310.834 | -43.502 |
| 2012-05-01 11:18:00 | 40680 | -5989304.45 | 3449013.22 | 3524840.23 | -3996.50 | -5859.16 | -1042.19 | 51.771 | -19.054 | 308.437 | -42.477 |
| 2012-05-01 11:19:00 | 40740 | -6219772.30 | 3092346.28 | 3456926.86 | -3683.81 | -6026.69 | -1221.00 | 53.924 | -20.653 | 306.046 | -41.449 |
| 2012-05-01 11:20:00 | 40800 | -6431136.14 | 2726181.18 | 3378395.55 | -3359.86 | -6175.68 | -1396.03 | 55.991 | -22.241 | 303.658 | -40.418 |
| 2012-05-01 11:21:00 | 40860 | -6622750.02 | 2351643.95 | 3289489.20 | -3025.65 | -6305.69 | -1566.75 | 57.982 | -23.816 | 301.267 | -39.387 |
| 2012-05-01 11:22:00 | 40920 | -6794028.87 | 1969886.02 | 3190482.52 | -2682.20 | -6416.32 | -1732.63 | 59.908 | -25.380 | 298.869 | -38.356 |
| 2012-05-01 11:23:00 | 40980 | -6944450.25 | 1582080.55 | 3081681.13 | -2330.57 | -6507.22 | -1893.16 | 61.778 | -26.932 | 296.461 | -37.326 |
| 2012-05-01 11:24:00 | 41040 | -7073555.89 | 1189418.90 | 2963420.62 | -1971.86 | -6578.15 | -2047.85 | 63.600 | -28.472 | 294.039 | -36.301 |
| 2012-05-01 11:25:00 | 41100 | -7180953.11 | 793106.95 | 2836065.49 | -1607.15 | -6628.87 | -2196.23 | 65.382 | -29.999 | 291.598 | -35.280 |
| 2012-05-01 11:26:00 | 41160 | -7266315.92 | 394361.36 | 2700008.02 | -1237.56 | -6659.25 | -2337.85 | 67.132 | -31.516 | 289.135 | -34.266 |
| 2012-05-01 11:27:00 | 41220 | -7329386.02 | -5594.15 | 2555667.04 | -864.25 | -6669.19 | -2472.28 | 68.855 | -33.020 | 286.645 | -33.260 |
| 2012-05-01 11:28:00 | 41280 | -7369973.52 | -405532.54 | 2403486.64 | -488.34 | -6658.68 | -2599.10 | 70.559 | -34.513 | 284.126 | -32.264 |
| 2012-05-01 11:29:00 | 41340 | -7387957.49 | -804227.19 | 2243934.79 | -110.99 | -6627.75 | -2717.93 | 72.249 | -35.994 | 281.572 | -31.279 |
| 2012-05-01 11:30:00 | 41400 | -7383286.29 | -1200455.72 | 2077501.90 | 266.64 | -6576.50 | -2828.41 | 73.931 | -37.463 | 278.981 | -30.309 |
| 2012-05-01 11:31:00 | 41460 | -7355977.65 | -1593003.63 | 1904699.30 | 643.40 | -6505.09 | -2930.20 | 75.611 | -38.921 | 276.350 | -29.354 |
| 2012-05-01 11:32:00 | 41520 | -7306118.64 | -1980668.08 | 1726057.67 | 1018.13 | -6413.76 | -3023.00 | 77.294 | -40.366 | 273.675 | -28.416 |
| 2012-05-01 11:33:00 | 41580 | -7233865.29 | -2362261.52 | 1542125.40 | 1389.69 | -6302.78 | -3106.51 | 78.986 | -41.800 | 270.952 | -27.499 |
| 2012-05-01 11:34:00 | 41640 | -7139442.13 | -2736615.31 | 1353466.93 | 1756.94 | -6172.50 | -3180.50 | 80.692 | -43.222 | 268.180 | -26.605 |
| 2012-05-01 11:35:00 | 41700 | -7023141.45 | -3102583.26 | 1160661.00 | 2118.75 | -6023.32 | -3244.73 | 82.419 | -44.630 | 265.355 | -25.736 |
| 2012-05-01 11:36:00 | 41760 | -6885322.34 | -3459045.13 | 964298.87 | 2474.04 | -5855.71 | -3299.01 | 84.171 | -46.026 | 262.476 | -24.895 |
| 2012-05-01 11:37:00 | 41820 | -6726409.64 | -3804910.04 | 764982.54 | 2821.70 | -5670.18 | -3343.17 | 85.956 | -47.407 | 259.541 | -24.084 |
| 2012-05-01 11:38:00 | 41880 | -6546892.52 | -4139119.73 | 563322.90 | 3160.67 | -5467.30 | -3377.10 | 87.779 | -48.775 | 256.547 | -23.307 |
| 2012-05-01 11:39:00 | 41940 | -6347323.04 | -4460651.81 | 359937.86 | 3489.94 | -5247.70 | -3400.67 | 89.647 | -50.126 | 253.496 | -22.568 |
| 2012-05-01 11:40:00 | 42000 | -6128314.41 | -4768522.83 | 155450.47 | 3808.48 | -5012.05 | -3413.83 | 91.567 | -51.462 | 250.385 | -21.868 |
| 2012-05-01 11:41:00 | 42060 | -5890539.10 | -5061791.27 | -49512.96 | 4115.34 | -4761.07 | -3416.54 | 93.548 | -52.781 | 247.217 | -21.212 |
| 2012-05-01 11:42:00 | 42120 | -5634726.78 | -5339560.37 | -254324.79 | 4409.56 | -4495.54 | -3408.78 | 95.597 | -54.080 | 243.991 | -20.603 |
| 2012-05-01 11:43:00 | 42180 | -5361662.08 | -5600980.85 | -458357.94 | 4690.27 | -4216.26 | -3390.59 | 97.723 | -55.360 | 240.711 | -20.044 |
| 2012-05-01 11:44:00 | 42240 | -5072182.20 | -5845253.48 | -660987.79 | 4956.60 | -3924.09 | -3362.02 | 99.936 | -56.617 | 237.378 | -19.538 |
| 2012-05-01 11:45:00 | 42300 | -4767174.35 | -6071631.46 | -861594.10 | 5207.74 | -3619.92 | -3323.15 | 102.247 | -57.850 | 233.997 | -19.089 |
| 2012-05-01 11:46:00 | 42360 | -4447573.02 | -6279422.73 | -1059562.88 | 5442.92 | -3304.69 | -3274.12 | 104.666 | -59.057 | 230.572 | -18.698 |
| 2012-05-01 11:47:00 | 42420 | -4114357.19 | -6467992.01 | -1254288.21 | 5661.44 | -2979.35 | -3215.07 | 107.206 | -60.234 | 227.109 | -18.370 |
| 2012-05-01 11:48:00 | 42480 | -3768547.28 | -6636762.74 | -1445174.17 | 5862.62 | -2644.91 | -3146.17 | 109.880 | -61.380 | 223.613 | -18.105 |
| 2012-05-01 11:49:00 | 42540 | -3411202.12 | -6785218.81 | -1631636.56 | 6045.85 | -2302.37 | -3067.65 | 112.702 | -62.489 | 220.090 | -17.906 |
| 2012-05-01 11:50:00 | 42600 | -3043415.67 | -6912906.14 | -1813104.70 | 6210.57 | -1952.79 | -2979.74 | 115.685 | -63.558 | 216.549 | -17.774 |
| 2012-05-01 11:51:00 | 42660 | -2666313.76 | -7019434.04 | -1989023.19 | 6356.29 | -1597.23 | -2882.71 | 118.845 | -64.583 | 212.996 | -17.709 |
| 2012-05-01 11:52:00 | 42720 | -2281050.61 | -7104476.38 | -2158853.53 | 6482.54 | -1236.79 | -2776.86 | 122.195 | -65.558 | 209.439 | -17.711 |
| 2012-05-01 11:53:00 | 42780 | -1888805.39 | -7167772.60 | -2322075.84 | 6588.96 | -872.55 | -2662.50 | 125.748 | -66.477 | 205.885 | -17.781 |
| 2012-05-01 11:54:00 | 42840 | -1490778.60 | -7209128.50 | -2478190.34 | 6675.21 | -505.63 | -2539.99 | 129.516 | -67.334 | 202.343 | -17.917 |
| 2012-05-01 11:55:00 | 42900 | -1088188.46 | -7228416.84 | -2626718.95 | 6741.03 | -137.15 | -2409.70 | 133.507 | -68.121 | 198.818 | -18.118 |
| 2012-05-01 11:56:00 | 42960 | -682267.19 | -7225577.70 | -2767206.71 | 6786.22 | 231.77 | -2272.03 | 137.725 | -68.833 | 195.318 | -18.382 |
| 2012-05-01 11:57:00 | 43020 | -274257.29 | -7200618.73 | -2899223.14 | 6810.64 | 599.99 | -2127.39 | 142.165 | -69.460 | 191.849 | -18.705 |
| 2012-05-01 11:58:00 | 43080 | 134592.23 | -7153615.09 | -3022363.61 | 6814.20 | 966.40 | -1976.24 | 146.818 | -69.996 | 188.415 | -19.087 |
| 2012-05-01 11:59:00 | 43140 | 543029.66 | -7084709.25 | -3136250.52 | 6796.90 | 1329.88 | -1819.02 | 151.663 | -70.432 | 185.022 | -19.522 |
| 2012-05-01 12:00:00 | 43200 | 949804.36 | -6994110.60 | -3240534.47 | 6758.79 | 1689.31 | -1656.22 | 156.671 | -70.763 | 181.673 | -20.009 |
| 2012-05-01 12:01:00 | 43260 | 1353670.58 | -6882094.80 | -3334895.34 | 6699.98 | 2043.60 | -1488.34 | 161.801 | -70.983 | 178.370 | -20.543 |
| 2012-05-01 12:02:00 | 43320 | 1753391.24 | -6749002.97 | -3419043.29 | 6620.64 | 2391.67 | -1315.88 | 167.006 | -71.089 | 175.117 | -21.122 |
| 2012-05-01 12:03:00 | 43380 | 2147741.69 | -6595240.70 | -3492719.58 | 6521.02 | 2732.44 | -1139.37 | 172.234 | -71.078 | 171.913 | -21.741 |
| 2012-05-01 12:04:00 | 43440 | 2535513.47 | -6421276.83 | -3555697.47 | 6401.41 | 3064.88 | -959.35 | 177.429 | -70.952 | 168.761 | -22.397 |
| 2012-05-01 12:05:00 | 43500 | 2915517.92 | -6227642.01 | -3607782.82 | 6262.17 | 3387.97 | -776.38 | 182.541 | -70.712 | 165.659 | -23.087 |
| 2012-05-01 12:06:00 | 43560 | 3286589.90 | -6014927.20 | -3648814.78 | 6103.73 | 3700.72 | -591.00 | 187.524 | -70.363 | 162.607 | -23.808 |
| 2012-05-01 12:07:00 | 43620 | 3647591.27 | -5783781.80 | -3678666.24 | 5926.57 | 4002.17 | -403.79 | 192.339 | -69.910 | 159.604 | -24.556 |
| 2012-05-01 12:08:00 | 43680 | 3997414.43 | -5534911.79 | -3697244.28 | 5731.22 | 4291.39 | -215.32 | 196.960 | -69.360 | 156.649 | -25.328 |
| 2012-05-01 12:09:00 | 43740 | 4334985.65 | -5269077.52 | -3704490.43 | 5518.28 | 4567.50 | -26.16 | 201.369 | -68.722 | 153.738 | -26.122 |
| 2012-05-01 12:10:00 | 43800 | 4659268.44 | -4987091.50 | -3700380.87 | 5288.38 | 4829.64 | 163.11 | 205.557 | -68.002 | 150.870 | -26.934 |
| 2012-05-01 12:11:00 | 43860 | 4969266.66 | -4689815.91 | -3684926.56 | 5042.24 | 5077.02 | 351.91 | 209.522 | -67.208 | 148.043 | -27.763 |
| 2012-05-01 12:12:00 | 43920 | 5264027.62 | -4378160.00 | -3658173.19 | 4780.60 | 5308.86 | 539.65 | 213.270 | -66.348 | 145.253 | -28.605 |
| 2012-05-01 12:13:00 | 43980 | 5542645.04 | -4053077.33 | -3620201.07 | 4504.26 | 5524.46 | 725.77 | 216.810 | -65.428 | 142.498 | -29.459 |
| 2012-05-01 12:14:00 | 44040 | 5804261.80 | -3715562.91 | -3571124.91 | 4214.06 | 5723.15 | 909.69 | 220.153 | -64.455 | 139.775 | -30.323 |
| 2012-05-01 12:15:00 | 44100 | 6048072.60 | -3366650.16 | -3511093.50 | 3910.88 | 5904.31 | 1090.85 | 223.312 | -63.435 | 137.080 | -31.194 |
| 2012-05-01 12:16:00 | 44160 | 6273326.52 | -3007407.79 | -3440289.25 | 3595.65 | 6067.38 | 1268.69 | 226.303 | -62.373 | 134.410 | -32.071 |
| 2012-05-01 12:17:00 | 44220 | 6479329.25 | -2638936.53 | -3358927.69 | 3269.34 | 6211.86 | 1442.67 | 229.139 | -61.272 | 131.762 | -32.951 |
| 2012-05-01 12:18:00 | 44280 | 6665445.37 | -2262365.77 | -3267256.78 | 2932.94 | 6337.29 | 1612.25 | 231.836 | -60.139 | 129.134 | -33.834 |
| 2012-05-01 12:19:00 | 44340 | 6831100.25 | -1878850.15 | -3165556.24 | 2587.47 | 6443.29 | 1776.91 | 234.405 | -58.975 | 126.520 | -34.718 |
| 2012-05-01 12:20:00 | 44400 | 6975781.87 | -1489566.01 | -3054136.63 | 2234.01 | 6529.53 | 1936.13 | 236.861 | -57.784 | 123.919 | -35.601 |
| 2012-05-01 12:21:00 | 44460 | 7099042.49 | -1095707.79 | -2933338.48 | 1873.62 | 6595.72 | 2089.44 | 239.215 | -56.569 | 121.328 | -36.482 |
| 2012-05-01 12:22:00 | 44520 | 7200499.99 | -698484.40 | -2803531.27 | 1507.42 | 6641.67 | 2236.36 | 241.478 | -55.332 | 118.742 | -37.359 |
| 2012-05-01 12:23:00 | 44580 | 7279839.15 | -299115.44 | -2665112.24 | 1136.53 | 6667.22 | 2376.43 | 243.660 | -54.075 | 116.159 | -38.231 |
| 2012-05-01 12:24:00 | 44640 | 7336812.62 | 101172.45 | -2518505.26 | 762.09 | 6672.29 | 2509.22 | 245.772 | -52.800 | 113.576 | -39.097 |
| 2012-05-01 12:25:00 | 44700 | 7371241.76 | 501149.42 | -2364159.50 | 385.25 | 6656.86 | 2634.32 | 247.822 | -51.508 | 110.990 | -39.956 |
| 2012-05-01 12:26:00 | 44760 | 7383017.22 | 899586.21 | -2202548.11 | 7.16 | 6620.96 | 2751.35 | 249.818 | -50.202 | 108.397 | -40.805 |
| 2012-05-01 12:27:00 | 44820 | 7372099.30 | 1295257.88 | -2034166.69 | -371.01 | 6564.71 | 2859.93 | 251.768 | -48.881 | 105.796 | -41.645 |
| 2012-05-01 12:28:00 | 44880 | 7338518.16 | 1686947.65 | -1859531.88 | -748.09 | 6488.27 | 2959.74 | 253.679 | -47.548 | 103.182 | -42.474 |
| 2012-05-01 12:29:00 | 44940 | 7282373.73 | 2073450.64 | -1679179.69 | -1122.92 | 6391.86 | 3050.46 | 255.557 | -46.202 | 100.554 | -43.290 |
| 2012-05-01 12:30:00 | 45000 | 7203835.46 | 2453577.59 | -1493663.91 | -1494.36 | 6275.79 | 3131.81 | 257.410 | -44.846 | 97.908 | -44.092 |
| 2012-05-01 12:31:00 | 45060 | 7103141.83 | 2826158.54 | -1303554.39 | -1861.24 | 6140.39 | 3203.55 | 259.242 | -43.479 | 95.242 | -44.880 |
| 2012-05-01 12:32:00 | 45120 | 6980599.65 | 3190046.49 | -1109435.27 | -2222.45 | 5986.09 | 3265.43 | 261.060 | -42.103 | 92.554 | -45.651 |
| 2012-05-01 12:33:00 | 45180 | 6836583.16 | 3544120.92 | -911903.23 | -2576.87 | 5813.36 | 3317.28 | 262.870 | -40.717 | 89.841 | -46.405 |
| 2012-05-01 12:34:00 | 45240 | 6671532.89 | 3887291.29 | -711565.59 | -2923.40 | 5622.72 | 3358.93 | 264.677 | -39.322 | 87.100 | -47.140 |
| 2012-05-01 12:35:00 | 45300 | 6485954.32 | 4218500.46 | -509038.47 | -3260.97 | 5414.75 | 3390.24 | 266.486 | -37.920 | 84.331 | -47.855 |
| 2012-05-01 12:36:00 | 45360 | 6280416.37 | 4536727.91 | -304944.89 | -3588.54 | 5190.10 | 3411.13 | 268.304 | -36.509 | 81.530 | -48.549 |
| 2012-05-01 12:37:00 | 45420 | 6055549.64 | 4840992.98 | -99912.81 | -3905.10 | 4949.46 | 3421.52 | 270.135 | -35.091 | 78.696 | -49.220 |
| 2012-05-01 12:38:00 | 45480 | 5812044.50 | 5130357.91 | 105426.78 | -4209.66 | 4693.56 | 3421.38 | 271.986 | -33.666 | 75.827 | -49.866 |
| 2012-05-01 12:39:00 | 45540 | 5550648.95 | 5403930.74 | 310441.84 | -4501.29 | 4423.19 | 3410.70 | 273.861 | -32.234 | 72.923 | -50.488 |
| 2012-05-01 12:40:00 | 45600 | 5272166.32 | 5660868.14 | 514501.24 | -4779.08 | 4139.19 | 3389.53 | 275.768 | -30.795 | 69.982 | -51.082 |
| 2012-05-01 12:41:00 | 45660 | 4977452.82 | 5900377.99 | 716976.68 | -5042.18 | 3842.42 | 3357.92 | 277.713 | -29.351 | 67.004 | -51.648 |
| 2012-05-01 12:42:00 | 45720 | 4667414.85 | 6121721.88 | 917244.71 | -5289.77 | 3533.81 | 3315.97 | 279.702 | -27.902 | 63.988 | -52.185 |
| 2012-05-01 12:43:00 | 45780 | 4343006.26 | 6324217.42 | 1114688.60 | -5521.08 | 3214.31 | 3263.81 | 281.743 | -26.448 | 60.934 | -52.691 |
| 2012-05-01 12:44:00 | 45840 | 4005225.32 | 6507240.36 | 1308700.30 | -5735.40 | 2884.89 | 3201.59 | 283.843 | -24.989 | 57.843 | -53.164 |
| 2012-05-01 12:45:00 | 45900 | 3655111.70 | 6670226.52 | 1498682.31 | -5932.06 | 2546.59 | 3129.52 | 286.010 | -23.528 | 54.715 | -53.604 |
| 2012-05-01 12:46:00 | 45960 | 3293743.20 | 6812673.60 | 1684049.55 | -6110.46 | 2200.43 | 3047.81 | 288.253 | -22.065 | 51.551 | -54.010 |
| 2012-05-01 12:47:00 | 46020 | 2922232.41 | 6934142.70 | 1864231.17 | -6270.05 | 1847.50 | 2956.71 | 290.582 | -20.601 | 48.354 | -54.379 |
| 2012-05-01 12:48:00 | 46080 | 2541723.28 | 7034259.69 | 2038672.32 | -6410.33 | 1488.88 | 2856.51 | 293.007 | -19.139 | 45.125 | -54.712 |
| 2012-05-01 12:49:00 | 46140 | 2153387.53 | 7112716.42 | 2206835.88 | -6530.87 | 1125.68 | 2747.51 | 295.538 | -17.680 | 41.866 | -55.006 |
| 2012-05-01 12:50:00 | 46200 | 1758421.05 | 7169271.61 | 2368204.13 | -6631.30 | 759.01 | 2630.05 | 298.186 | -16.228 | 38.582 | -55.262 |
| 2012-05-01 12:51:00 | 46260 | 1358040.15 | 7203751.64 | 2522280.36 | -6711.31 | 390.03 | 2504.50 | 300.964 | -14.786 | 35.274 | -55.478 |
| 2012-05-01 12:52:00 | 46320 | 953477.79 | 7216051.08 | 2668590.41 | -6770.65 | 19.85 | 2371.25 | 303.885 | -13.359 | 31.946 | -55.654 |
| 2012-05-01 12:53:00 | 46380 | 545979.76 | 7206132.97 | 2806684.12 | -6809.14 | -350.37 | 2230.70 | 306.961 | -11.954 | 28.602 | -55.790 |
| 2012-05-01 12:54:00 | 46440 | 136800.77 | 7174029.00 | 2936136.76 | -6826.66 | -719.49 | 2083.29 | 310.204 | -10.576 | 25.247 | -55.884 |
| 2012-05-01 12:55:00 | 46500 | -272799.37 | 7119839.29 | 3056550.32 | -6823.17 | -1086.37 | 1929.47 | 313.627 | -9.236 | 21.885 | -55.937 |
| 2012-05-01 12:56:00 | 46560 | -681559.76 | 7043732.19 | 3167554.72 | -6798.68 | -1449.88 | 1769.73 | 317.239 | -7.944 | 18.519 | -55.950 |
| 2012-05-01 12:57:00 | 46620 | -1088222.24 | 6945943.61 | 3268809.02 | -6753.26 | -1808.90 | 1604.55 | 321.051 | -6.713 | 15.155 | -55.921 |
| 2012-05-01 12:58:00 | 46680 | -1491535.34 | 6826776.37 | 3360002.37 | -6687.06 | -2162.32 | 1434.45 | 325.065 | -5.558 | 11.796 | -55.851 |
| 2012-05-01 12:59:00 | 46740 | -1890258.14 | 6686599.18 | 3440855.05 | -6600.29 | -2509.05 | 1259.95 | 329.284 | -4.494 | 8.446 | -55.742 |
| 2012-05-01 13:00:00 | 46800 | -2283164.13 | 6525845.51 | 3511119.30 | -6493.22 | -2848.03 | 1081.59 | 333.700 | -3.542 | 5.111 | -55.593 |
| 2012-05-01 13:01:00 | 46860 | -2669044.97 | 6345012.17 | 3570580.04 | -6366.18 | -3178.20 | 899.93 | 338.302 | -2.719 | 1.792 | -55.406 |
| 2012-05-01 13:02:00 | 46920 | -3046714.28 | 6144657.81 | 3619055.56 | -6219.57 | -3498.56 | 715.51 | 343.067 | -2.045 | 358.494 | -55.180 |
| 2012-05-01 13:03:00 | 46980 | -3415011.23 | 5925401.10 | 3656398.06 | -6053.85 | -3808.12 | 528.92 | 347.968 | -1.536 | 355.220 | -54.918 |
| 2012-05-01 13:04:00 | 47040 | -3772804.21 | 5687918.84 | 3682494.10 | -5869.53 | -4105.92 | 340.73 | 352.968 | -1.208 | 351.972 | -54.621 |
| 2012-05-01 13:05:00 | 47100 | -4118994.23 | 5432943.78 | 3697264.90 | -5667.18 | -4391.06 | 151.51 | 358.022 | -1.068 | 348.753 | -54.289 |
| 2012-05-01 13:06:00 | 47160 | -4452518.37 | 5161262.40 | 3700666.61 | -5447.44 | -4662.66 | -38.15 | 3.086 | -1.122 | 345.564 | -53.924 |
| 2012-05-01 13:07:00 | 47220 | -4772352.97 | 4873712.36 | 3692690.38 | -5210.99 | -4919.88 | -227.66 | 8.112 | -1.368 | 342.406 | -53.528 |
| 2012-05-01 13:08:00 | 47280 | -5077516.85 | 4571179.97 | 3673362.41 | -4958.54 | -5161.94 | -416.44 | 13.055 | -1.798 | 339.282 | -53.101 |
| 2012-05-01 13:09:00 | 47340 | -5367074.27 | 4254597.36 | 3642743.81 | -4690.90 | -5388.10 | -603.91 | 17.876 | -2.400 | 336.190 | -52.645 |
| 2012-05-01 13:10:00 | 47400 | -5640137.77 | 3924939.61 | 3600930.44 | -4408.89 | -5597.67 | -789.50 | 22.543 | -3.157 | 333.132 | -52.162 |
| 2012-05-01 13:11:00 | 47460 | -5895870.92 | 3583221.74 | 3548052.54 | -4113.37 | -5790.00 | -972.64 | 27.032 | -4.052 | 330.108 | -51.654 |
| 2012-05-01 13:12:00 | 47520 | -6133490.84 | 3230495.49 | 3484274.34 | -3805.27 | -5964.52 | -1152.75 | 31.328 | -5.065 | 327.116 | -51.121 |
| 2012-05-01 13:13:00 | 47580 | -6352270.58 | 2867846.12 | 3409793.51 | -3485.53 | -6120.69 | -1329.30 | 35.421 | -6.177 | 324.157 | -50.565 |
| 2012-05-01 13:14:00 | 47640 | -6551541.31 | 2496389.03 | 3324840.55 | -3155.14 | -6258.04 | -1501.74 | 39.310 | -7.372 | 321.229 | -49.987 |
| 2012-05-01 13:15:00 | 47700 | -6730694.38 | 2117266.29 | 3229678.06 | -2815.11 | -6376.15 | -1669.53 | 42.999 | -8.634 | 318.331 | -49.390 |
| 2012-05-01 13:16:00 | 47760 | -6889183.08 | 1731643.14 | 3124599.90 | -2466.50 | -6474.66 | -1832.17 | 46.495 | -9.948 | 315.461 | -48.773 |
| 2012-05-01 13:17:00 | 47820 | -7026524.33 | 1340704.41 | 3009930.27 | -2110.38 | -6553.29 | -1989.17 | 49.807 | -11.304 | 312.619 | -48.140 |
| 2012-05-01 13:18:00 | 47880 | -7142300.12 | 945650.82 | 2886022.70 | -1747.84 | -6611.79 | -2140.03 | 52.947 | -12.692 | 309.802 | -47.491 |
| 2012-05-01 13:19:00 | 47940 | -7236158.73 | 547695.33 | 2753258.93 | -1379.99 | -6650.00 | -2284.30 | 55.927 | -14.104 | 307.009 | -46.827 |
| 2012-05-01 13:20:00 | 48000 | -7307815.76 | 148059.39 | 2612047.75 | -1007.97 | -6667.79 | -2421.54 | 58.760 | -15.533 | 304.237 | -46.150 |
| 2012-05-01 13:21:00 | 48060 | -7357054.98 | -252030.77 | 2462823.71 | -632.92 | -6665.14 | -2551.32 | 61.460 | -16.975 | 301.484 | -45.461 |
| 2012-05-01 13:22:00 | 48120 | -7383728.92 | -651347.94 | 2306045.79 | -255.99 | -6642.04 | -2673.27 | 64.037 | -18.424 | 298.749 | -44.762 |
| 2012-05-01 13:23:00 | 48180 | -7387759.30 | -1048667.67 | 2142195.93 | 121.67 | -6598.57 | -2787.00 | 66.504 | -19.879 | 296.029 | -44.053 |
| 2012-05-01 13:24:00 | 48240 | -7369137.23 | -1442771.99 | 1971777.64 | 498.90 | -6534.88 | -2892.16 | 68.871 | -21.336 | 293.323 | -43.336 |
| 2012-05-01 13:25:00 | 48300 | -7327923.17 | -1832453.14 | 1795314.33 | 874.54 | -6451.17 | -2988.44 | 71.149 | -22.793 | 290.626 | -42.611 |
| 2012-05-01 13:26:00 | 48360 | -7264246.71 | -2216517.28 | 1613347.81 | 1247.45 | -6347.70 | -3075.55 | 73.348 | -24.248 | 287.939 | -41.881 |
| 2012-05-01 13:27:00 | 48420 | -7178306.16 | -2593788.08 | 1426436.55 | 1616.49 | -6224.79 | -3153.23 | 75.477 | -25.701 | 285.258 | -41.146 |
| 2012-05-01 13:28:00 | 48480 | -7070367.88 | -2963110.30 | 1235154.01 | 1980.53 | -6082.81 | -3221.23 | 77.543 | -27.151 | 282.580 | -40.408 |
| 2012-05-01 13:29:00 | 48540 | -6940765.43 | -3323353.35 | 1040086.84 | 2338.45 | -5922.22 | -3279.35 | 79.555 | -28.596 | 279.905 | -39.667 |
| 2012-05-01 13:30:00 | 48600 | -6789898.56 | -3673414.67 | 841833.14 | 2689.16 | -5743.51 | -3327.42 | 81.521 | -30.036 | 277.228 | -38.925 |
| 2012-05-01 13:31:00 | 48660 | -6618231.89 | -4012223.07 | 641000.61 | 3031.60 | -5547.22 | -3365.29 | 83.446 | -31.470 | 274.549 | -38.183 |
| 2012-05-01 13:32:00 | 48720 | -6426293.53 | -4338742.04 | 438204.69 | 3364.71 | -5333.97 | -3392.85 | 85.337 | -32.898 | 271.865 | -37.442 |
| 2012-05-01 13:33:00 | 48780 | -6214673.43 | -4651972.80 | 234066.66 | 3687.49 | -5104.40 | -3410.01 | 87.201 | -34.319 | 269.174 | -36.703 |
| 2012-05-01 13:34:00 | 48840 | -5984021.51 | -4950957.39 | 29211.82 | 3998.94 | -4859.22 | -3416.74 | 89.044 | -35.734 | 266.474 | -35.968 |
| 2012-05-01 13:35:00 | 48900 | -5735045.76 | -5234781.54 | -175732.47 | 4298.13 | -4599.18 | -3413.00 | 90.871 | -37.141 | 263.762 | -35.237 |
| 2012-05-01 13:36:00 | 48960 | -5468509.96 | -5502577.41 | -380138.70 | 4584.13 | -4325.08 | -3398.81 | 92.687 | -38.540 | 261.037 | -34.513 |
| 2012-05-01 13:37:00 | 49020 | -5185231.44 | -5753526.27 | -583381.09 | 4856.08 | -4037.75 | -3374.21 | 94.499 | -39.932 | 258.296 | -33.796 |
| 2012-05-01 13:38:00 | 49080 | -4886078.49 | -5986860.90 | -784837.48 | 5113.14 | -3738.09 | -3339.29 | 96.311 | -41.314 | 255.538 | -33.087 |
| 2012-05-01 13:39:00 | 49140 | -4571967.79 | -6201867.98 | -983891.25 | 5354.54 | -3426.99 | -3294.14 | 98.130 | -42.688 | 252.760 | -32.388 |
| 2012-05-01 13:40:00 | 49200 | -4243861.57 | -6397890.17 | -1179933.18 | 5579.54 | -3105.42 | -3238.92 | 99.960 | -44.052 | 249.962 | -31.701 |
| 2012-05-01 13:41:00 | 49260 | -3902764.71 | -6574328.15 | -1372363.29 | 5787.45 | -2774.35 | -3173.78 | 101.808 | -45.407 | 247.141 | -31.026 |
| 2012-05-01 13:42:00 | 49320 | -3549721.65 | -6730642.41 | -1560592.66 | 5977.65 | -2434.80 | -3098.93 | 103.679 | -46.750 | 244.296 | -30.366 |
| 2012-05-01 13:43:00 | 49380 | -3185813.27 | -6866354.87 | -1744045.21 | 6149.54 | -2087.80 | -3014.60 | 105.579 | -48.082 | 241.425 | -29.722 |
| 2012-05-01 13:44:00 | 49440 | -2812153.54 | -6981050.34 | -1922159.48 | 6302.61 | -1734.41 | -2921.03 | 107.515 | -49.401 | 238.528 | -29.095 |
| 2012-05-01 13:45:00 | 49500 | -2429886.19 | -7074377.79 | -2094390.29 | 6436.39 | -1375.71 | -2818.53 | 109.494 | -50.707 | 235.602 | -28.487 |
| 2012-05-01 13:46:00 | 49560 | -2040181.24 | -7146051.36 | -2260210.40 | 6550.47 | -1012.80 | -2707.40 | 111.522 | -51.998 | 232.649 | -27.899 |
| 2012-05-01 13:47:00 | 49620 | -1644231.42 | -7195851.29 | -2419112.14 | 6644.50 | -646.78 | -2587.98 | 113.608 | -53.274 | 229.666 | -27.334 |
| 2012-05-01 13:48:00 | 49680 | -1243248.58 | -7223624.57 | -2570608.92 | 6718.19 | -278.76 | -2460.63 | 115.760 | -54.532 | 226.653 | -26.793 |
| 2012-05-01 13:49:00 | 49740 | -838460.01 | -7229285.39 | -2714236.74 | 6771.32 | 90.12 | -2325.74 | 117.987 | -55.772 | 223.611 | -26.278 |
| 2012-05-01 13:50:00 | 49800 | -431104.69 | -7212815.45 | -2849555.55 | 6803.73 | 458.74 | -2183.73 | 120.298 | -56.990 | 220.539 | -25.790 |
| 2012-05-01 13:51:00 | 49860 | -22429.59 | -7174264.00 | -2976150.64 | 6815.30 | 825.98 | -2035.03 | 122.705 | -58.186 | 217.437 | -25.332 |
| 2012-05-01 13:52:00 | 49920 | 386314.16 | -7113747.69 | -3093633.90 | 6806.01 | 1190.72 | -1880.08 | 125.218 | -59.355 | 214.308 | -24.904 |
| 2012-05-01 13:53:00 | 49980 | 793875.07 | -7031450.29 | -3201644.96 | 6775.88 | 1551.83 | -1719.37 | 127.849 | -60.497 | 211.150 | -24.509 |
| 2012-05-01 13:54:00 | 50040 | 1199005.06 | -6927622.08 | -3299852.36 | 6725.00 | 1908.23 | -1553.37 | 130.612 | -61.607 | 207.967 | -24.148 |
| 2012-05-01 13:55:00 | 50100 | 1600463.28 | -6802579.16 | -3387954.48 | 6653.52 | 2258.81 | -1382.61 | 133.519 | -62.681 | 204.759 | -23.823 |
| 2012-05-01 13:56:00 | 50160 | 1997019.86 | -6656702.50 | -3465680.57 | 6561.66 | 2602.51 | -1207.59 | 136.584 | -63.716 | 201.528 | -23.535 |
| 2012-05-01 13:57:00 | 50220 | 2387459.67 | -6490436.80 | -3532791.50 | 6449.68 | 2938.27 | -1028.86 | 139.822 | -64.706 | 198.277 | -23.285 |
| 2012-05-01 13:58:00 | 50280 | 2770586.02 | -6304289.15 | -3589080.54 | 6317.94 | 3265.07 | -846.96 | 143.245 | -65.646 | 195.009 | -23.075 |
| 2012-05-01 13:59:00 | 50340 | 3145224.33 | -6098827.56 | -3634373.98 | 6166.82 | 3581.91 | -662.44 | 146.866 | -66.531 | 191.725 | -22.906 |
| 2012-05-01 14:00:00 | 50400 | 3510225.66 | -5874679.21 | -3668531.72 | 5996.78 | 3887.80 | -475.86 | 150.695 | -67.354 | 188.429 | -22.778 |
| 2012-05-01 14:01:00 | 50460 | 3864470.28 | -5632528.57 | -3691447.66 | 5808.35 | 4181.83 | -287.80 | 154.738 | -68.108 | 185.124 | -22.692 |
| 2012-05-01 14:02:00 | 50520 | 4206871.06 | -5373115.41 | -3703050.06 | 5602.09 | 4463.08 | -98.84 | 158.998 | -68.786 | 181.814 | -22.649 |
| 2012-05-01 14:03:00 | 50580 | 4536376.82 | -5097232.47 | -3703301.79 | 5378.63 | 4730.68 | 90.46 | 163.469 | -69.380 | 178.501 | -22.648 |
| 2012-05-01 14:04:00 | 50640 | 4851975.55 | -4805723.16 | -3692200.45 | 5138.64 | 4983.82 | 279.50 | 168.139 | -69.882 | 175.190 | -22.690 |
| 2012-05-01 14:05:00 | 50700 | 5152697.55 | -4499478.97 | -3669778.39 | 4882.86 | 5221.72 | 467.72 | 172.987 | -70.286 | 171.883 | -22.775 |
| 2012-05-01 14:06:00 | 50760 | 5437618.35 | -4179436.78 | -3636102.66 | 4612.07 | 5443.64 | 654.53 | 177.982 | -70.586 | 168.584 | -22.902 |
| 2012-05-01 14:07:00 | 50820 | 5705861.63 | -3846576.04 | -3591274.79 | 4327.08 | 5648.89 | 839.36 | 183.084 | -70.776 | 165.296 | -23.071 |
| 2012-05-01 14:08:00 | 50880 | 5956601.90 | -3501915.78 | -3535430.53 | 4028.78 | 5836.85 | 1021.65 | 188.246 | -70.853 | 162.022 | -23.280 |
| 2012-05-01 14:09:00 | 50940 | 6189067.07 | -3146511.53 | -3468739.44 | 3718.07 | 6006.93 | 1200.83 | 193.418 | -70.815 | 158.765 | -23.529 |
| 2012-05-01 14:10:00 | 51000 | 6402540.82 | -2781452.13 | -3391404.38 | 3395.89 | 6158.61 | 1376.35 | 198.548 | -70.664 | 155.527 | -23.817 |
| 2012-05-01 14:11:00 | 51060 | 6596364.88 | -2407856.36 | -3303660.97 | 3063.25 | 6291.40 | 1547.68 | 203.586 | -70.401 | 152.311 | -24.142 |
| 2012-05-01 14:12:00 | 51120 | 6769941.03 | -2026869.57 | -3205776.81 | 2721.14 | 6404.91 | 1714.29 | 208.491 | -70.031 | 149.119 | -24.503 |
| 2012-05-01 14:13:00 | 51180 | 6922733.04 | -1639660.21 | -3098050.72 | 2370.62 | 6498.77 | 1875.66 | 213.227 | -69.559 | 145.953 | -24.898 |
| 2012-05-01 14:14:00 | 51240 | 7054268.28 | -1247416.18 | -2980811.85 | 2012.76 | 6572.69 | 2031.30 | 217.769 | -68.993 | 142.813 | -25.326 |
| 2012-05-01 14:15:00 | 51300 | 7164139.28 | -851341.28 | -2854418.68 | 1648.66 | 6626.43 | 2180.73 | 222.101 | -68.338 | 139.702 | -25.785 |
| 2012-05-01 14:16:00 | 51360 | 7252005.01 | -452651.43 | -2719257.93 | 1279.44 | 6659.83 | 2323.48 | 226.215 | -67.603 | 136.619 | -26.273 |
| 2012-05-01 14:17:00 | 51420 | 7317591.95 | -52571.04 | -2575743.39 | 906.22 | 6672.77 | 2459.12 | 230.110 | -66.795 | 133.566 | -26.790 |
| 2012-05-01 14:18:00 | 51480 | 7360694.99 | 347670.85 | -2424314.69 | 530.17 | 6665.21 | 2587.22 | 233.791 | -65.921 | 130.542 | -27.332 |
| 2012-05-01 14:19:00 | 51540 | 7381178.10 | 746844.29 | -2265435.92 | 152.42 | 6637.16 | 2707.39 | 237.266 | -64.988 | 127.548 | -27.898 |
| 2012-05-01 14:20:00 | 51600 | 7378974.83 | 1143722.23 | -2099594.23 | -225.85 | 6588.71 | 2819.25 | 240.548 | -64.001 | 124.583 | -28.487 |
| 2012-05-01 14:21:00 | 51660 | 7354088.49 | 1537084.33 | -1927298.38 | -603.49 | 6520.00 | 2922.47 | 243.649 | -62.966 | 121.646 | -29.096 |
| 2012-05-01 14:22:00 | 51720 | 7306592.23 | 1925720.69 | -1749077.12 | -979.32 | 6431.23 | 3016.72 | 246.582 | -61.889 | 118.737 | -29.725 |
| 2012-05-01 14:23:00 | 51780 | 7236628.87 | 2308435.59 | -1565477.63 | -1352.20 | 6322.67 | 3101.70 | 249.362 | -60.773 | 115.855 | -30.371 |
| 2012-05-01 14:24:00 | 51840 | 7144410.44 | 2684051.26 | -1377063.77 | -1720.97 | 6194.64 | 3177.15 | 252.003 | -59.622 | 112.998 | -31.033 |
| 2012-05-01 14:25:00 | 51900 | 7030217.61 | 3051411.43 | -1184414.42 | -2084.49 | 6047.55 | 3242.85 | 254.516 | -58.441 | 110.166 | -31.709 |
| 2012-05-01 14:26:00 | 51960 | 6894398.85 | 3409385.01 | -988121.67 | -2441.65 | 5881.84 | 3298.57 | 256.916 | -57.231 | 107.356 | -32.398 |
| 2012-05-01 14:27:00 | 52020 | 6737369.38 | 3756869.54 | -788788.95 | -2791.33 | 5698.01 | 3344.15 | 259.212 | -55.995 | 104.568 | -33.099 |
| 2012-05-01 14:28:00 | 52080 | 6559609.91 | 4092794.66 | -587029.24 | -3132.47 | 5496.62 | 3379.45 | 261.416 | -54.736 | 101.799 | -33.809 |
| 2012-05-01 14:29:00 | 52140 | 6361665.19 | 4416125.39 | -383463.14 | -3464.00 | 5278.30 | 3404.35 | 263.539 | -53.455 | 99.047 | -34.529 |
| 2012-05-01 14:30:00 | 52200 | 6144142.36 | 4725865.41 | -178716.93 | -3784.90 | 5043.71 | 3418.77 | 265.588 | -52.155 | 96.311 | -35.255 |
| 2012-05-01 14:31:00 | 52260 | 5907709.08 | 5021060.13 | 26579.32 | -4094.19 | 4793.58 | 3422.68 | 267.573 | -50.837 | 93.589 | -35.987 |
| 2012-05-01 14:32:00 | 52320 | 5653091.46 | 5300799.67 | 231793.75 | -4390.89 | 4528.67 | 3416.05 | 269.502 | -49.501 | 90.878 | -36.724 |
| 2012-05-01 14:33:00 | 52380 | 5381071.90 | 5564221.70 | 436294.64 | -4674.10 | 4249.80 | 3398.90 | 271.383 | -48.150 | 88.177 | -37.465 |
| 2012-05-01 14:34:00 | 52440 | 5092486.59 | 5810514.13 | 639452.38 | -4942.94 | 3957.83 | 3371.29 | 273.221 | -46.784 | 85.483 | -38.207 |
| 2012-05-01 14:35:00 | 52500 | 4788222.98 | 6038917.65 | 840641.43 | -5196.58 | 3653.66 | 3333.29 | 275.024 | -45.404 | 82.795 | -38.951 |
| 2012-05-01 14:36:00 | 52560 | 4469217.04 | 6248728.09 | 1039242.24 | -5434.23 | 3338.22 | 3285.03 | 276.798 | -44.010 | 80.109 | -39.694 |
| 2012-05-01 14:37:00 | 52620 | 4136450.34 | 6439298.63 | 1234643.20 | -5655.15 | 3012.50 | 3226.66 | 278.550 | -42.603 | 77.425 | -40.437 |
| 2012-05-01 14:38:00 | 52680 | 3790947.03 | 6610041.82 | 1426242.56 | -5858.67 | 2677.48 | 3158.35 | 280.284 | -41.184 | 74.738 | -41.176 |
| 2012-05-01 14:39:00 | 52740 | 3433770.66 | 6760431.39 | 1613450.23 | -6044.15 | 2334.22 | 3080.31 | 282.006 | -39.753 | 72.048 | -41.912 |
| 2012-05-01 14:40:00 | 52800 | 3066020.87 | 6890003.92 | 1795689.69 | -6211.03 | 1983.76 | 2992.78 | 283.723 | -38.311 | 69.352 | -42.643 |
| 2012-05-01 14:41:00 | 52860 | 2688829.96 | 6998360.26 | 1972399.73 | -6358.78 | 1627.19 | 2896.04 | 285.439 | -36.856 | 66.648 | -43.368 |
| 2012-05-01 14:42:00 | 52920 | 2303359.44 | 7085166.81 | 2143036.24 | -6486.94 | 1265.62 | 2790.38 | 287.161 | -35.391 | 63.933 | -44.086 |
| 2012-05-01 14:43:00 | 52980 | 1910796.35 | 7150156.50 | 2307073.84 | -6595.13 | 900.15 | 2676.13 | 288.893 | -33.914 | 61.206 | -44.795 |
| 2012-05-01 14:44:00 | 53040 | 1512349.58 | 7193129.66 | 2464007.58 | -6683.02 | 531.92 | 2553.65 | 290.643 | -32.426 | 58.463 | -45.495 |
| 2012-05-01 14:45:00 | 53100 | 1109246.17 | 7213954.61 | 2613354.46 | -6750.32 | 162.07 | 2423.30 | 292.415 | -30.927 | 55.703 | -46.184 |
| 2012-05-01 14:46:00 | 53160 | 702727.42 | 7212568.10 | 2754654.92 | -6796.83 | -208.27 | 2285.50 | 294.217 | -29.417 | 52.924 | -46.861 |
| 2012-05-01 14:47:00 | 53220 | 294045.10 | 7188975.44 | 2887474.32 | -6822.42 | -577.95 | 2140.67 | 296.056 | -27.896 | 50.124 | -47.524 |
| 2012-05-01 14:48:00 | 53280 | -115542.49 | 7143250.51 | 3011404.23 | -6827.00 | -945.82 | 1989.26 | 297.938 | -26.364 | 47.301 | -48.173 |
| 2012-05-01 14:49:00 | 53340 | -524774.41 | 7075535.50 | 3126063.72 | -6810.57 | -1310.76 | 1831.74 | 299.873 | -24.821 | 44.452 | -48.806 |
| 2012-05-01 14:50:00 | 53400 | -932390.98 | 6986040.46 | 3231100.51 | -6773.17 | -1671.64 | 1668.59 | 301.867 | -23.267 | 41.577 | -49.421 |
| 2012-05-01 14:51:00 | 53460 | -1337137.69 | 6875042.63 | 3326192.08 | -6714.93 | -2027.34 | 1500.32 | 303.932 | -21.702 | 38.674 | -50.017 |
| 2012-05-01 14:52:00 | 53520 | -1737769.13 | 6742885.53 | 3411046.68 | -6636.03 | -2376.76 | 1327.45 | 306.077 | -20.128 | 35.741 | -50.593 |
| 2012-05-01 14:53:00 | 53580 | -2133052.79 | 6589977.92 | 3485404.14 | -6536.72 | -2718.84 | 1150.50 | 308.314 | -18.544 | 32.777 | -51.148 |
| 2012-05-01 14:54:00 | 53640 | -2521772.91 | 6416792.46 | 3549036.77 | -6417.30 | -3052.52 | 970.04 | 310.656 | -16.952 | 29.781 | -51.679 |
| 2012-05-01 14:55:00 | 53700 | -2902734.24 | 6223864.25 | 3601749.97 | -6278.16 | -3376.77 | 786.62 | 313.115 | -15.353 | 26.753 | -52.186 |
| 2012-05-01 14:56:00 | 53760 | -3274765.72 | 6011789.14 | 3643382.87 | -6119.72 | -3690.59 | 600.79 | 315.708 | -13.750 | 23.692 | -52.667 |
| 2012-05-01 14:57:00 | 53820 | -3636724.09 | 5781221.84 | 3673808.78 | -5942.47 | -3993.01 | 413.15 | 318.450 | -12.145 | 20.598 | -53.121 |
| 2012-05-01 14:58:00 | 53880 | -3987497.43 | 5532873.88 | 3692935.58 | -5746.98 | -4283.12 | 224.25 | 321.360 | -10.544 | 17.471 | -53.545 |
| 2012-05-01 14:59:00 | 53940 | -4326008.57 | 5267511.39 | 3700705.98 | -5533.84 | -4560.02 | 34.70 | 324.456 | -8.952 | 14.312 | -53.939 |
| 2012-05-01 15:00:00 | 54000 | -4651218.41 | 4985952.68 | 3697097.70 | -5303.72 | -4822.86 | -154.94 | 327.759 | -7.378 | 11.122 | -54.302 |
| 2012-05-01 15:01:00 | 54060 | -4962129.09 | 4689065.70 | 3682123.48 | -5057.32 | -5070.83 | -344.07 | 331.286 | -5.832 | 7.903 | -54.631 |
| 2012-05-01 15:02:00 | 54120 | -5257787.07 | 4377765.31 | 3655831.04 | -4795.42 | -5303.18 | -532.12 | 335.058 | -4.328 | 4.655 | -54.926 |
| 2012-05-01 15:03:00 | 54180 | -5537286.02 | 4053010.47 | 3618302.93 | -4518.83 | -5519.20 | -718.50 | 339.091 | -2.883 | 1.383 | -55.185 |
| 2012-05-01 15:04:00 | 54240 | -5799769.60 | 3715801.21 | 3569656.23 | -4228.39 | -5718.22 | -902.64 | 343.396 | -1.519 | 358.087 | -55.408 |
| 2012-05-01 15:05:00 | 54300 | -6044434.10 | 3367175.56 | 3510042.17 | -3925.01 | -5899.65 | -1083.98 | 347.978 | -0.260 | 354.771 | -55.593 |
| 2012-05-01 15:06:00 | 54360 | -6270530.79 | 3008206.30 | 3439645.65 | -3609.62 | -6062.92 | -1261.96 | 352.833 | 0.863 | 351.439 | -55.740 |
| 2012-05-01 15:07:00 | 54420 | -6477368.31 | 2639997.65 | 3358684.66 | -3283.20 | -6207.55 | -1436.04 | 357.943 | 1.822 | 348.094 | -55.847 |
| 2012-05-01 15:08:00 | 54480 | -6664314.63 | 2263681.87 | 3267409.56 | -2946.76 | -6333.09 | -1605.68 | 3.278 | 2.584 | 344.740 | -55.914 |
| 2012-05-01 15:09:00 | 54540 | -6830799.08 | 1880415.72 | 3166102.29 | -2601.32 | -6439.17 | -1770.36 | 8.788 | 3.121 | 341.382 | -55.940 |
| 2012-05-01 15:10:00 | 54600 | -6976313.97 | 1491376.92 | 3055075.54 | -2247.95 | -6525.47 | -1929.58 | 14.414 | 3.413 | 338.022 | -55.926 |
| 2012-05-01 15:11:00 | 54660 | -7100416.12 | 1097760.54 | 2934671.69 | -1887.74 | -6591.72 | -2082.85 | 20.085 | 3.445 | 334.667 | -55.871 |
| 2012-05-01 15:12:00 | 54720 | -7202728.24 | 700775.27 | 2805261.78 | -1521.80 | -6637.74 | -2229.71 | 25.726 | 3.218 | 331.319 | -55.775 |
| 2012-05-01 15:13:00 | 54780 | -7282939.97 | 301639.76 | 2667244.37 | -1151.25 | -6663.38 | -2369.69 | 31.263 | 2.739 | 327.984 | -55.638 |
| 2012-05-01 15:14:00 | 54840 | -7340808.80 | -98421.17 | 2521044.28 | -777.23 | -6668.58 | -2502.39 | 36.633 | 2.028 | 324.665 | -55.461 |
| 2012-05-01 15:15:00 | 54900 | -7376160.82 | -498180.25 | 2367111.24 | -400.88 | -6653.32 | -2627.39 | 41.784 | 1.112 | 321.366 | -55.244 |
| 2012-05-01 15:16:00 | 54960 | -7388891.13 | -896411.50 | 2205918.58 | -23.36 | -6617.66 | -2744.32 | 46.681 | 0.022 | 318.090 | -54.988 |
| 2012-05-01 15:17:00 | 55020 | -7378964.20 | -1291894.03 | 2037961.70 | 354.17 | -6561.72 | -2852.81 | 51.303 | -1.213 | 314.842 | -54.693 |
| 2012-05-01 15:18:00 | 55080 | -7346413.85 | -1683415.73 | 1863756.54 | 730.56 | -6485.67 | -2952.54 | 55.642 | -2.562 | 311.622 | -54.361 |
| 2012-05-01 15:19:00 | 55140 | -7291343.19 | -2069776.97 | 1683838.06 | 1104.65 | -6389.75 | -3043.21 | 59.701 | -4.000 | 308.435 | -53.992 |
| 2012-05-01 15:20:00 | 55200 | -7213924.19 | -2449794.27 | 1498758.51 | 1475.31 | -6274.26 | -3124.53 | 63.489 | -5.505 | 305.281 | -53.587 |
| 2012-05-01 15:21:00 | 55260 | -7114397.18 | -2822303.89 | 1309085.82 | 1841.40 | -6139.56 | -3196.27 | 67.022 | -7.057 | 302.163 | -53.148 |
| 2012-05-01 15:22:00 | 55320 | -6993070.02 | -3186165.34 | 1115401.77 | 2201.80 | -5986.06 | -3258.21 | 70.318 | -8.643 | 299.082 | -52.676 |
| 2012-05-01 15:23:00 | 55380 | -6850317.18 | -3540264.91 | 918300.29 | 2555.41 | -5814.25 | -3310.16 | 73.397 | -10.250 | 296.038 | -52.171 |
| 2012-05-01 15:24:00 | 55440 | -6686578.54 | -3883518.97 | 718385.61 | 2901.15 | -5624.64 | -3351.96 | 76.279 | -11.871 | 293.031 | -51.636 |
| 2012-05-01 15:25:00 | 55500 | -6502358.01 | -4214877.32 | 516270.41 | 3237.96 | -5417.82 | -3383.49 | 78.981 | -13.498 | 290.063 | -51.072 |
| 2012-05-01 15:26:00 | 55560 | -6298221.98 | -4533326.32 | 312573.96 | 3564.83 | -5194.44 | -3404.66 | 81.523 | -15.128 | 287.131 | -50.480 |
| 2012-05-01 15:27:00 | 55620 | -6074797.60 | -4837891.99 | 107920.24 | 3880.75 | -4955.16 | -3415.39 | 83.921 | -16.756 | 284.237 | -49.861 |
| 2012-05-01 15:28:00 | 55680 | -5832770.79 | -5127642.94 | -97063.96 | 4184.75 | -4700.74 | -3415.67 | 86.190 | -18.379 | 281.377 | -49.217 |
| 2012-05-01 15:29:00 | 55740 | -5572884.18 | -5401693.20 | -301750.97 | 4475.92 | -4431.94 | -3405.49 | 88.344 | -19.998 | 278.552 | -48.548 |
| 2012-05-01 15:30:00 | 55800 | -5295934.82 | -5659204.86 | -505514.10 | 4753.37 | -4149.59 | -3384.88 | 90.397 | -21.609 | 275.760 | -47.858 |
| 2012-05-01 15:31:00 | 55860 | -5002771.74 | -5899390.66 | -707729.59 | 5016.24 | -3854.56 | -3353.91 | 92.359 | -23.212 | 272.999 | -47.146 |
| 2012-05-01 15:32:00 | 55920 | -4694293.36 | -6121516.31 | -907778.46 | 5263.75 | -3547.74 | -3312.68 | 94.241 | -24.808 | 270.267 | -46.414 |
| 2012-05-01 15:33:00 | 55980 | -4371444.77 | -6324902.72 | -1105048.44 | 5495.13 | -3230.08 | -3261.31 | 96.054 | -26.395 | 267.562 | -45.663 |
| 2012-05-01 15:34:00 | 56040 | -4035214.80 | -6508928.07 | -1298935.78 | 5709.68 | -2902.54 | -3199.95 | 97.804 | -27.974 | 264.882 | -44.895 |
| 2012-05-01 15:35:00 | 56100 | -3686633.10 | -6673029.69 | -1488847.10 | 5906.75 | -2566.12 | -3128.81 | 99.502 | -29.544 | 262.224 | -44.111 |
| 2012-05-01 15:36:00 | 56160 | -3326766.92 | -6816705.73 | -1674201.18 | 6085.73 | -2221.86 | -3048.09 | 101.153 | -31.106 | 259.586 | -43.312 |
| 2012-05-01 15:37:00 | 56220 | -2956717.92 | -6939516.68 | -1854430.74 | 6246.09 | -1870.80 | -2958.03 | 102.765 | -32.659 | 256.966 | -42.499 |
| 2012-05-01 15:38:00 | 56280 | -2577618.82 | -7041086.74 | -2028984.15 | 6387.33 | -1514.01 | -2858.93 | 104.344 | -34.205 | 254.360 | -41.674 |
| 2012-05-01 15:39:00 | 56340 | -2190629.95 | -7121104.91 | -2197327.09 | 6509.01 | -1152.58 | -2751.07 | 105.897 | -35.742 | 251.766 | -40.838 |
| 2012-05-01 15:40:00 | 56400 | -1796935.75 | -7179325.96 | -2358944.19 | 6610.78 | -787.62 | -2634.79 | 107.429 | -37.271 | 249.181 | -39.991 |
| 2012-05-01 15:41:00 | 56460 | -1397741.15 | -7215571.19 | -2513340.59 | 6692.31 | -420.24 | -2510.44 | 108.945 | -38.793 | 246.602 | -39.136 |
| 2012-05-01 15:42:00 | 56520 | -994267.96 | -7229728.94 | -2660043.43 | 6753.36 | -51.56 | -2378.40 | 110.452 | -40.306 | 244.026 | -38.273 |
| 2012-05-01 15:43:00 | 56580 | -587751.11 | -7221754.95 | -2798603.32 | 6793.74 | 317.30 | -2239.08 | 111.954 | -41.812 | 241.451 | -37.405 |
| 2012-05-01 15:44:00 | 56640 | -179434.96 | -7191672.55 | -2928595.66 | 6813.32 | 685.20 | -2092.89 | 113.458 | -43.309 | 238.872 | -36.531 |
| 2012-05-01 15:45:00 | 56700 | 229430.53 | -7139572.54 | -3049621.98 | 6812.05 | 1051.03 | -1940.29 | 114.969 | -44.798 | 236.287 | -35.653 |
| 2012-05-01 15:46:00 | 56760 | 637593.55 | -7065612.97 | -3161311.12 | 6789.91 | 1413.67 | -1781.73 | 116.492 | -46.278 | 233.693 | -34.773 |
| 2012-05-01 15:47:00 | 56820 | 1043804.25 | -6970018.64 | -3263320.39 | 6746.99 | 1772.00 | -1617.71 | 118.034 | -47.750 | 231.087 | -33.892 |
| 2012-05-01 15:48:00 | 56880 | 1446818.56 | -6853080.50 | -3355336.61 | 6683.39 | 2124.95 | -1448.71 | 119.601 | -49.212 | 228.465 | -33.012 |
| 2012-05-01 15:49:00 | 56940 | 1845401.96 | -6715154.71 | -3437077.06 | 6599.33 | 2471.41 | -1275.27 | 121.200 | -50.665 | 225.824 | -32.134 |
| 2012-05-01 15:50:00 | 57000 | 2238333.21 | -6556661.67 | -3508290.38 | 6495.04 | 2810.35 | -1097.90 | 122.838 | -52.108 | 223.160 | -31.260 |
| 2012-05-01 15:51:00 | 57060 | 2624408.11 | -6378084.66 | -3568757.32 | 6370.84 | 3140.71 | -917.15 | 124.523 | -53.539 | 220.472 | -30.391 |
| 2012-05-01 15:52:00 | 57120 | 3002443.15 | -6179968.50 | -3618291.42 | 6227.11 | 3461.48 | -733.56 | 126.266 | -54.958 | 217.754 | -29.530 |
| 2012-05-01 15:53:00 | 57180 | 3371279.13 | -5962917.86 | -3656739.61 | 6064.28 | 3771.70 | -547.71 | 128.074 | -56.365 | 215.005 | -28.678 |
| 2012-05-01 15:54:00 | 57240 | 3729784.70 | -5727595.45 | -3683982.69 | 5882.85 | 4070.39 | -360.16 | 129.960 | -57.756 | 212.221 | -27.837 |
| 2012-05-01 15:55:00 | 57300 | 4076859.80 | -5474720.04 | -3699935.69 | 5683.36 | 4356.65 | -171.47 | 131.937 | -59.132 | 209.400 | -27.010 |
| 2012-05-01 15:56:00 | 57360 | 4411439.07 | -5205064.33 | -3704548.18 | 5466.43 | 4629.59 | 17.77 | 134.018 | -60.490 | 206.537 | -26.198 |
| 2012-05-01 15:57:00 | 57420 | 4732495.08 | -4919452.55 | -3697804.40 | 5232.70 | 4888.38 | 206.97 | 136.220 | -61.827 | 203.631 | -25.406 |
| 2012-05-01 15:58:00 | 57480 | 5039041.52 | -4618758.04 | -3679723.37 | 4982.90 | 5132.22 | 395.58 | 138.560 | -63.141 | 200.680 | -24.634 |
| 2012-05-01 15:59:00 | 57540 | 5330136.20 | -4303900.58 | -3650358.84 | 4717.77 | 5360.36 | 583.00 | 141.062 | -64.429 | 197.680 | -23.886 |
| 2012-05-01 16:00:00 | 57600 | 5604883.99 | -3975843.62 | -3609799.11 | 4438.14 | 5572.09 | 768.65 | 143.747 | -65.687 | 194.631 | -23.166 |
| 2012-05-01 16:01:00 | 57660 | 5862439.58 | -3635591.36 | -3558166.83 | 4144.85 | 5766.76 | 951.99 | 146.643 | -66.909 | 191.532 | -22.475 |
| 2012-05-01 16:02:00 | 57720 | 6102010.07 | -3284185.67 | -3495618.65 | 3838.79 | 5943.77 | 1132.43 | 149.779 | -68.091 | 188.381 | -21.818 |
| 2012-05-01 16:03:00 | 57780 | 6322857.45 | -2922703.00 | -3422344.69 | 3520.90 | 6102.57 | 1309.42 | 153.189 | -69.225 | 185.178 | -21.198 |
| 2012-05-01 16:04:00 | 57840 | 6524300.91 | -2552251.01 | -3338568.08 | 3192.16 | 6242.67 | 1482.42 | 156.908 | -70.303 | 181.924 | -20.619 |
| 2012-05-01 16:05:00 | 57900 | 6705718.94 | -2173965.29 | -3244544.22 | 2853.56 | 6363.63 | 1650.91 | 160.969 | -71.316 | 178.621 | -20.083 |
| 2012-05-01 16:06:00 | 57960 | 6866551.30 | -1789005.82 | -3140560.05 | 2506.14 | 6465.07 | 1814.35 | 165.408 | -72.252 | 175.269 | -19.595 |
| 2012-05-01 16:07:00 | 58020 | 7006300.74 | -1398553.45 | -3026933.18 | 2150.97 | 6546.68 | 1972.25 | 170.249 | -73.099 | 171.872 | -19.157 |
| 2012-05-01 16:08:00 | 58080 | 7124534.61 | -1003806.32 | -2904010.96 | 1789.14 | 6608.19 | 2124.12 | 175.507 | -73.842 | 168.433 | -18.774 |
| 2012-05-01 16:09:00 | 58140 | 7220886.23 | -605976.15 | -2772169.38 | 1421.75 | 6649.42 | 2269.48 | 181.176 | -74.467 | 164.958 | -18.448 |
| 2012-05-01 16:10:00 | 58200 | 7295056.04 | -206284.51 | -2631812.01 | 1049.93 | 6670.22 | 2407.90 | 187.224 | -74.958 | 161.450 | -18.182 |
| 2012-05-01 16:11:00 | 58260 | 7346812.56 | 194040.89 | -2483368.68 | 674.83 | 6670.54 | 2538.95 | 193.584 | -75.303 | 157.917 | -17.979 |
| 2012-05-01 16:12:00 | 58320 | 7375993.18 | 593769.98 | -2327294.27 | 297.60 | 6650.35 | 2662.21 | 200.157 | -75.490 | 154.365 | -17.841 |
| 2012-05-01 16:13:00 | 58380 | 7382504.69 | 991674.18 | -2164067.25 | -80.61 | 6609.72 | 2777.30 | 206.818 | -75.512 | 150.800 | -17.769 |
| 2012-05-01 16:14:00 | 58440 | 7366323.58 | 1386530.09 | -1994188.25 | -458.63 | 6548.77 | 2883.88 | 213.429 | -75.370 | 147.231 | -17.764 |
| 2012-05-01 16:15:00 | 58500 | 7327496.22 | 1777123.38 | -1818178.53 | -835.29 | 6467.67 | 2981.61 | 219.860 | -75.066 | 143.665 | -17.827 |
| 2012-05-01 16:16:00 | 58560 | 7266138.69 | 2162252.44 | -1636578.40 | -1209.44 | 6366.67 | 3070.18 | 226.001 | -74.611 | 140.110 | -17.957 |
| 2012-05-01 16:17:00 | 58620 | 7182436.52 | 2540732.16 | -1449945.52 | -1579.92 | 6246.08 | 3149.32 | 231.775 | -74.016 | 136.573 | -18.155 |
| 2012-05-01 16:18:00 | 58680 | 7076644.10 | 2911397.60 | -1258853.21 | -1945.59 | 6106.26 | 3218.79 | 237.138 | -73.297 | 133.062 | -18.418 |
| 2012-05-01 16:19:00 | 58740 | 6949083.98 | 3273107.58 | -1063888.69 | -2305.33 | 5947.64 | 3278.36 | 242.075 | -72.467 | 129.584 | -18.746 |
| 2012-05-01 16:20:00 | 58800 | 6800145.89 | 3624748.23 | -865651.26 | -2658.01 | 5770.70 | 3327.86 | 246.596 | -71.542 | 126.144 | -19.136 |
| 2012-05-01 16:21:00 | 58860 | 6630285.54 | 3965236.47 | -664750.45 | -3002.56 | 5575.99 | 3367.12 | 250.722 | -70.533 | 122.748 | -19.584 |
| 2012-05-01 16:22:00 | 58920 | 6440023.30 | 4293523.34 | -461804.18 | -3337.90 | 5364.10 | 3396.02 | 254.486 | -69.454 | 119.400 | -20.090 |
| 2012-05-01 16:23:00 | 58980 | 6229942.54 | 4608597.31 | -257436.75 | -3663.00 | 5135.67 | 3414.48 | 257.921 | -68.314 | 116.105 | -20.649 |
| 2012-05-01 16:24:00 | 59040 | 6000687.92 | 4909487.39 | -52277.03 | -3976.86 | 4891.42 | 3422.43 | 261.064 | -67.121 | 112.865 | -21.258 |
| 2012-05-01 16:25:00 | 59100 | 5752963.36 | 5195266.21 | 153043.60 | -4278.51 | 4632.09 | 3419.84 | 263.947 | -65.882 | 109.683 | -21.914 |
| 2012-05-01 16:26:00 | 59160 | 5487529.93 | 5465052.84 | 357893.11 | -4567.01 | 4358.48 | 3406.73 | 266.601 | -64.604 | 106.559 | -22.614 |
| 2012-05-01 16:27:00 | 59220 | 5205203.44 | 5718015.59 | 561640.86 | -4841.46 | 4071.44 | 3383.12 | 269.054 | -63.292 | 103.495 | -23.355 |
| 2012-05-01 16:28:00 | 59280 | 4906851.99 | 5953374.60 | 763659.52 | -5101.03 | 3771.84 | 3349.10 | 271.331 | -61.950 | 100.490 | -24.132 |
| 2012-05-01 16:29:00 | 59340 | 4593393.25 | 6170404.25 | 963327.00 | -5344.91 | 3460.62 | 3304.77 | 273.453 | -60.580 | 97.543 | -24.943 |
| 2012-05-01 16:30:00 | 59400 | 4265791.64 | 6368435.46 | 1160028.43 | -5572.34 | 3138.73 | 3250.26 | 275.440 | -59.187 | 94.653 | -25.785 |
| 2012-05-01 16:31:00 | 59460 | 3925055.32 | 6546857.75 | 1353158.03 | -5782.62 | 2807.16 | 3185.74 | 277.307 | -57.771 | 91.818 | -26.655 |
| 2012-05-01 16:32:00 | 59520 | 3572233.09 | 6705121.14 | 1542121.03 | -5975.10 | 2466.94 | 3111.41 | 279.071 | -56.336 | 89.036 | -27.550 |
| 2012-05-01 16:33:00 | 59580 | 3208411.13 | 6842737.91 | 1726335.48 | -6149.19 | 2119.11 | 3027.50 | 280.744 | -54.883 | 86.305 | -28.467 |
| 2012-05-01 16:34:00 | 59640 | 2834709.63 | 6959284.08 | 1905234.09 | -6304.34 | 1764.76 | 2934.26 | 282.337 | -53.412 | 83.622 | -29.405 |
| 2012-05-01 16:35:00 | 59700 | 2452279.28 | 7054400.72 | 2078265.96 | -6440.07 | 1404.98 | 2831.99 | 283.860 | -51.926 | 80.983 | -30.361 |
| 2012-05-01 16:36:00 | 59760 | 2062297.76 | 7127795.12 | 2244898.34 | -6555.98 | 1040.87 | 2721.00 | 285.323 | -50.424 | 78.385 | -31.332 |
| 2012-05-01 16:37:00 | 59820 | 1665966.02 | 7179241.65 | 2404618.24 | -6651.69 | 673.57 | 2601.63 | 286.734 | -48.908 | 75.826 | -32.316 |
| 2012-05-01 16:38:00 | 59880 | 1264504.56 | 7208582.46 | 2556934.04 | -6726.92 | 304.21 | 2474.26 | 288.099 | -47.378 | 73.300 | -33.312 |
| 2012-05-01 16:39:00 | 59940 | 859149.68 | 7215727.98 | 2701377.02 | -6781.44 | -66.08 | 2339.27 | 289.426 | -45.834 | 70.805 | -34.318 |
| 2012-05-01 16:40:00 | 60000 | 451149.58 | 7200657.19 | 2837502.81 | -6815.07 | -436.15 | 2197.09 | 290.721 | -44.275 | 68.337 | -35.333 |
| 2012-05-01 16:41:00 | 60060 | 41760.50 | 7163417.66 | 2964892.76 | -6827.73 | -804.85 | 2048.15 | 291.989 | -42.703 | 65.891 | -36.353 |
| 2012-05-01 16:42:00 | 60120 | -367757.15 | 7104125.39 | 3083155.22 | -6819.36 | -1171.05 | 1892.92 | 293.237 | -41.117 | 63.465 | -37.378 |
| 2012-05-01 16:43:00 | 60180 | -776142.75 | 7022964.45 | 3191926.79 | -6790.00 | -1533.62 | 1731.87 | 294.469 | -39.517 | 61.053 | -38.407 |
| 2012-05-01 16:44:00 | 60240 | -1182139.34 | 6920186.36 | 3290873.40 | -6739.75 | -1891.44 | 1565.51 | 295.690 | -37.902 | 58.652 | -39.438 |
| 2012-05-01 16:45:00 | 60300 | -1584497.57 | 6796109.31 | 3379691.37 | -6668.75 | -2243.40 | 1394.34 | 296.906 | -36.272 | 56.258 | -40.468 |
| 2012-05-01 16:46:00 | 60360 | -1981979.50 | 6651117.16 | 3458108.31 | -6577.25 | -2588.43 | 1218.89 | 298.122 | -34.626 | 53.867 | -41.498 |
| 2012-05-01 16:47:00 | 60420 | -2373362.51 | 6485658.19 | 3525883.96 | -6465.51 | -2925.45 | 1039.72 | 299.343 | -32.962 | 51.474 | -42.525 |
| 2012-05-01 16:48:00 | 60480 | -2757443.03 | 6300243.71 | 3582810.95 | -6333.89 | -3253.44 | 857.36 | 300.573 | -31.281 | 49.074 | -43.548 |
| 2012-05-01 16:49:00 | 60540 | -3133040.27 | 6095446.46 | 3628715.39 | -6182.81 | -3571.38 | 672.39 | 301.820 | -29.581 | 46.665 | -44.566 |
| 2012-05-01 16:50:00 | 60600 | -3498999.88 | 5871898.78 | 3663457.41 | -6012.72 | -3878.29 | 485.38 | 303.089 | -27.860 | 44.240 | -45.577 |
| 2012-05-01 16:51:00 | 60660 | -3854197.50 | 5630290.64 | 3686931.59 | -5824.17 | -4173.24 | 296.90 | 304.387 | -26.117 | 41.796 | -46.580 |
| 2012-05-01 16:52:00 | 60720 | -4197542.20 | 5371367.48 | 3699067.23 | -5617.73 | -4455.31 | 107.53 | 305.721 | -24.350 | 39.327 | -47.572 |
| 2012-05-01 16:53:00 | 60780 | -4527979.90 | 5095927.86 | 3699828.60 | -5394.04 | -4723.65 | -82.15 | 307.099 | -22.557 | 36.830 | -48.553 |
| 2012-05-01 16:54:00 | 60840 | -4844496.52 | 4804820.99 | 3689214.99 | -5153.81 | -4977.42 | -271.54 | 308.531 | -20.735 | 34.299 | -49.520 |
| 2012-05-01 16:55:00 | 60900 | -5146121.15 | 4498944.05 | 3667260.70 | -4897.77 | -5215.86 | -460.07 | 310.026 | -18.881 | 31.729 | -50.472 |
| 2012-05-01 16:56:00 | 60960 | -5431928.99 | 4179239.40 | 3634034.95 | -4626.72 | -5438.23 | -647.16 | 311.597 | -16.993 | 29.115 | -51.407 |
| 2012-05-01 16:57:00 | 61020 | -5701044.22 | 3846691.66 | 3589641.58 | -4341.50 | -5643.85 | -832.23 | 313.259 | -15.066 | 26.453 | -52.322 |
| 2012-05-01 16:58:00 | 61080 | -5952642.58 | 3502324.66 | 3534218.77 | -4042.98 | -5832.10 | -1014.72 | 315.027 | -13.098 | 23.737 | -53.216 |
| 2012-05-01 16:59:00 | 61140 | -6185953.98 | 3147198.21 | 3467938.56 | -3732.09 | -6002.41 | -1194.05 | 316.921 | -11.083 | 20.963 | -54.086 |
| 2012-05-01 17:00:00 | 61200 | -6400264.75 | 2782404.86 | 3391006.29 | -3409.79 | -6154.25 | -1369.69 | 318.965 | -9.017 | 18.126 | -54.931 |
| 2012-05-01 17:01:00 | 61260 | -6594919.87 | 2409066.52 | 3303660.00 | -3077.07 | -6287.17 | -1541.10 | 321.185 | -6.895 | 15.222 | -55.746 |
| 2012-05-01 17:02:00 | 61320 | -6769324.88 | 2028330.95 | 3206169.59 | -2734.96 | -6400.76 | -1707.74 | 323.616 | -4.714 | 12.245 | -56.530 |
| 2012-05-01 17:03:00 | 61380 | -6922947.71 | 1641368.27 | 3098836.07 | -2384.51 | -6494.69 | -1869.12 | 326.297 | -2.470 | 9.194 | -57.280 |
| 2012-05-01 17:04:00 | 61440 | -7055320.25 | 1249367.29 | 2981990.52 | -2026.79 | -6568.67 | -2024.73 | 329.277 | -0.161 | 6.064 | -57.993 |
| 2012-05-01 17:05:00 | 61500 | -7166039.77 | 853531.94 | 2855993.12 | -1662.92 | -6622.47 | -2174.10 | 332.613 | 2.211 | 2.853 | -58.665 |
| 2012-05-01 17:06:00 | 61560 | -7254770.09 | 455077.45 | 2721232.00 | -1294.01 | -6655.95 | -2316.78 | 336.372 | 4.636 | 359.561 | -59.293 |
| 2012-05-01 17:07:00 | 61620 | -7321242.54 | 55226.75 | 2578122.05 | -921.18 | -6669.00 | -2452.33 | 340.631 | 7.096 | 356.187 | -59.875 |
| 2012-05-01 17:08:00 | 61680 | -7365256.82 | -344793.40 | 2427103.60 | -545.59 | -6661.60 | -2580.33 | 345.476 | 9.556 | 352.732 | -60.407 |
| 2012-05-01 17:09:00 | 61740 | -7386681.47 | -743756.08 | 2268641.09 | -168.39 | -6633.76 | -2700.40 | 350.988 | 11.958 | 349.199 | -60.885 |
| 2012-05-01 17:10:00 | 61800 | -7385454.31 | -1140438.01 | 2103221.61 | 209.27 | -6585.59 | -2812.17 | 357.236 | 14.215 | 345.593 | -61.307 |
| 2012-05-01 17:11:00 | 61860 | -7361582.54 | -1533623.26 | 1931353.40 | 586.24 | -6517.24 | -2915.30 | 4.245 | 16.206 | 341.919 | -61.670 |
| 2012-05-01 17:12:00 | 61920 | -7315142.69 | -1922106.98 | 1753564.32 | 961.35 | -6428.91 | -3009.48 | 11.971 | 17.786 | 338.185 | -61.970 |
| 2012-05-01 17:13:00 | 61980 | -7246280.35 | -2304699.06 | 1570400.17 | 1333.46 | -6320.90 | -3094.43 | 20.264 | 18.805 | 334.401 | -62.206 |
| 2012-05-01 17:14:00 | 62040 | -7155209.68 | -2680227.74 | 1382423.07 | 1701.44 | -6193.53 | -3169.88 | 28.870 | 19.150 | 330.578 | -62.375 |
| 2012-05-01 17:15:00 | 62100 | -7042212.70 | -3047543.21 | 1190209.68 | 2064.16 | -6047.20 | -3235.60 | 37.466 | 18.779 | 326.729 | -62.476 |
| 2012-05-01 17:16:00 | 62160 | -6907638.45 | -3405521.06 | 994349.48 | 2420.50 | -5882.36 | -3291.40 | 45.729 | 17.736 | 322.866 | -62.508 |
| 2012-05-01 17:17:00 | 62220 | -6751901.83 | -3753065.71 | 795442.97 | 2769.39 | -5699.51 | -3337.12 | 53.409 | 16.137 | 319.004 | -62.469 |
| 2012-05-01 17:18:00 | 62280 | -6575482.34 | -4089113.76 | 594099.79 | 3109.75 | -5499.23 | -3372.61 | 60.360 | 14.130 | 315.157 | -62.360 |
| 2012-05-01 17:19:00 | 62340 | -6378922.59 | -4412637.15 | 390936.92 | 3440.56 | -5282.13 | -3397.76 | 66.541 | 11.862 | 311.339 | -62.181 |
| 2012-05-01 17:20:00 | 62400 | -6162826.61 | -4722646.32 | 186576.75 | 3760.80 | -5048.87 | -3412.51 | 71.980 | 9.449 | 307.562 | -61.934 |
| 2012-05-01 17:21:00 | 62460 | -5927857.99 | -5018193.18 | -18354.80 | 4069.49 | -4800.18 | -3416.80 | 76.744 | 6.980 | 303.839 | -61.619 |
| 2012-05-01 17:22:00 | 62520 | -5674737.84 | -5298374.00 | -223230.15 | 4365.69 | -4536.80 | -3410.63 | 80.918 | 4.510 | 300.180 | -61.239 |
| 2012-05-01 17:23:00 | 62580 | -5404242.61 | -5562332.11 | -427422.01 | 4648.51 | -4259.56 | -3394.02 | 84.585 | 2.075 | 296.593 | -60.796 |
| 2012-05-01 17:24:00 | 62640 | -5117201.65 | -5809260.51 | -630305.28 | 4917.08 | -3969.29 | -3367.03 | 87.822 | -0.309 | 293.086 | -60.293 |
| 2012-05-01 17:25:00 | 62700 | -4814494.74 | -6038404.29 | -831258.92 | 5170.58 | -3666.89 | -3329.72 | 90.696 | -2.631 | 289.665 | -59.731 |
| 2012-05-01 17:26:00 | 62760 | -4497049.39 | -6249062.92 | -1029667.86 | 5408.23 | -3353.28 | -3282.22 | 93.264 | -4.891 | 286.332 | -59.115 |
| 2012-05-01 17:27:00 | 62820 | -4165837.98 | -6440592.36 | -1224924.86 | 5629.33 | -3029.41 | -3224.68 | 95.573 | -7.090 | 283.090 | -58.446 |
| 2012-05-01 17:28:00 | 62880 | -3821874.84 | -6612407.02 | -1416432.35 | 5833.18 | -2696.28 | -3157.27 | 97.663 | -9.231 | 279.940 | -57.729 |
| 2012-05-01 17:29:00 | 62940 | -3466213.17 | -6763981.49 | -1603604.22 | 6019.18 | -2354.91 | -3080.20 | 99.567 | -11.319 | 276.880 | -56.965 |
| 2012-05-01 17:30:00 | 63000 | -3099941.81 | -6894852.18 | -1785867.63 | 6186.75 | -2006.33 | -2993.70 | 101.313 | -13.358 | 273.909 | -56.158 |
| 2012-05-01 17:31:00 | 63060 | -2724181.94 | -7004618.66 | -1962664.75 | 6335.38 | -1651.62 | -2898.04 | 102.922 | -15.353 | 271.024 | -55.311 |
| 2012-05-01 17:32:00 | 63120 | -2340083.72 | -7092944.93 | -2133454.41 | 6464.63 | -1291.84 | -2793.50 | 104.414 | -17.308 | 268.222 | -54.426 |
| 2012-05-01 17:33:00 | 63180 | -1948822.74 | -7159560.42 | -2297713.79 | 6574.08 | -928.11 | -2680.41 | 105.806 | -19.228 | 265.499 | -53.506 |
| 2012-05-01 17:34:00 | 63240 | -1551596.51 | -7204260.79 | -2454939.95 | 6663.42 | -561.52 | -2559.12 | 107.110 | -21.115 | 262.850 | -52.554 |
| 2012-05-01 17:35:00 | 63300 | -1149620.77 | -7226908.58 | -2604651.43 | 6732.36 | -193.21 | -2429.99 | 108.339 | -22.973 | 260.272 | -51.571 |
| 2012-05-01 17:36:00 | 63360 | -744125.86 | -7227433.65 | -2746389.65 | 6780.69 | 175.71 | -2293.41 | 109.502 | -24.805 | 257.758 | -50.559 |
| 2012-05-01 17:37:00 | 63420 | -336352.98 | -7205833.36 | -2879720.34 | 6808.27 | 544.12 | -2149.81 | 110.609 | -26.613 | 255.304 | -49.521 |
| 2012-05-01 17:38:00 | 63480 | 72449.63 | -7162172.62 | -3004234.84 | 6815.01 | 910.87 | -1999.61 | 111.666 | -28.400 | 252.904 | -48.459 |
| 2012-05-01 17:39:00 | 63540 | 481030.41 | -7096583.69 | -3119551.37 | 6800.88 | 1274.87 | -1843.29 | 112.680 | -30.168 | 250.554 | -47.373 |
| 2012-05-01 17:40:00 | 63600 | 888138.31 | -7009265.82 | -3225316.18 | 6765.92 | 1634.99 | -1681.30 | 113.659 | -31.918 | 248.249 | -46.267 |
| 2012-05-01 17:41:00 | 63660 | 1292526.60 | -6900484.66 | -3321204.64 | 6710.25 | 1990.13 | -1514.16 | 114.605 | -33.653 | 245.982 | -45.141 |
| 2012-05-01 17:42:00 | 63720 | 1692956.63 | -6770571.43 | -3406922.22 | 6634.01 | 2339.21 | -1342.36 | 115.526 | -35.372 | 243.749 | -43.997 |
| 2012-05-01 17:43:00 | 63780 | 2088201.63 | -6619922.00 | -3482205.44 | 6537.46 | 2681.16 | -1166.44 | 116.426 | -37.079 | 241.545 | -42.836 |
| 2012-05-01 17:44:00 | 63840 | 2477050.42 | -6448995.67 | -3546822.61 | 6420.86 | 3014.93 | -986.92 | 117.308 | -38.773 | 239.364 | -41.659 |
| 2012-05-01 17:45:00 | 63900 | 2858311.11 | -6258313.83 | -3600574.61 | 6284.58 | 3339.51 | -804.36 | 118.176 | -40.455 | 237.202 | -40.467 |
| 2012-05-01 17:46:00 | 63960 | 3230814.74 | -6048458.34 | -3643295.50 | 6129.03 | 3653.89 | -619.31 | 119.036 | -42.128 | 235.053 | -39.263 |
| 2012-05-01 17:47:00 | 64020 | 3593418.86 | -5820069.87 | -3674853.02 | 5954.68 | 3957.12 | -432.34 | 119.891 | -43.790 | 232.911 | -38.045 |
| 2012-05-01 17:48:00 | 64080 | 3945010.98 | -5573845.92 | -3695149.02 | 5762.06 | 4248.26 | -244.02 | 120.745 | -45.444 | 230.772 | -36.817 |
| 2012-05-01 17:49:00 | 64140 | 4284512.04 | -5310538.73 | -3704119.76 | 5551.75 | 4526.41 | -54.93 | 121.601 | -47.088 | 228.630 | -35.579 |
| 2012-05-01 17:50:00 | 64200 | 4610879.65 | -5030953.04 | -3701736.18 | 5324.39 | 4790.73 | 134.36 | 122.465 | -48.725 | 226.480 | -34.331 |
| 2012-05-01 17:51:00 | 64260 | 4923111.35 | -4735943.63 | -3688003.94 | 5080.67 | 5040.41 | 323.27 | 123.341 | -50.354 | 224.316 | -33.076 |
| 2012-05-01 17:52:00 | 64320 | 5220247.66 | -4426412.79 | -3662963.45 | 4821.34 | 5274.66 | 511.21 | 124.234 | -51.975 | 222.132 | -31.813 |
| 2012-05-01 17:53:00 | 64380 | 5501375.05 | -4103307.51 | -3626689.77 | 4547.17 | 5492.77 | 697.61 | 125.150 | -53.590 | 219.921 | -30.545 |
| 2012-05-01 17:54:00 | 64440 | 5765628.75 | -3767616.71 | -3579292.42 | 4259.02 | 5694.07 | 881.90 | 126.094 | -55.196 | 217.679 | -29.273 |
| 2012-05-01 17:55:00 | 64500 | 6012195.43 | -3420368.17 | -3520915.01 | 3957.76 | 5877.93 | 1063.52 | 127.074 | -56.796 | 215.397 | -27.998 |
| 2012-05-01 17:56:00 | 64560 | 6240315.75 | -3062625.43 | -3451734.89 | 3644.30 | 6043.78 | 1241.90 | 128.098 | -58.389 | 213.071 | -26.721 |
| 2012-05-01 17:57:00 | 64620 | 6449286.65 | -2695484.58 | -3371962.59 | 3319.61 | 6191.12 | 1416.50 | 129.176 | -59.973 | 210.691 | -25.445 |
| 2012-05-01 17:58:00 | 64680 | 6638463.62 | -2320070.87 | -3281841.19 | 2984.67 | 6319.47 | 1586.78 | 130.318 | -61.550 | 208.251 | -24.171 |
| 2012-05-01 17:59:00 | 64740 | 6807262.67 | -1937535.34 | -3181645.63 | 2640.52 | 6428.45 | 1752.22 | 131.540 | -63.119 | 205.744 | -22.902 |
| 2012-05-01 18:00:00 | 64800 | 6955162.17 | -1549051.26 | -3071681.84 | 2288.20 | 6517.71 | 1912.31 | 132.856 | -64.678 | 203.161 | -21.640 |
| 2012-05-01 18:01:00 | 64860 | 7081704.51 | -1155810.55 | -2952285.87 | 1928.79 | 6586.96 | 2066.55 | 134.289 | -66.227 | 200.494 | -20.390 |
| 2012-05-01 18:02:00 | 64920 | 7186497.50 | -759020.16 | -2823822.86 | 1563.40 | 6636.00 | 2214.46 | 135.864 | -67.765 | 197.735 | -19.153 |
| 2012-05-01 18:03:00 | 64980 | 7269215.70 | -359898.31 | -2686685.90 | 1193.15 | 6664.66 | 2355.60 | 137.613 | -69.288 | 194.875 | -17.935 |
| 2012-05-01 18:04:00 | 65040 | 7329601.37 | 40329.19 | -2541294.92 | 819.18 | 6672.84 | 2489.53 | 139.579 | -70.795 | 191.906 | -16.741 |
| 2012-05-01 18:05:00 | 65100 | 7367465.41 | 440432.74 | -2388095.32 | 442.62 | 6660.53 | 2615.82 | 141.816 | -72.282 | 188.821 | -15.576 |
| 2012-05-01 18:06:00 | 65160 | 7382687.88 | 839182.72 | -2227556.68 | 64.65 | 6627.74 | 2734.10 | 144.395 | -73.745 | 185.612 | -14.447 |
| 2012-05-01 18:07:00 | 65220 | 7375218.50 | 1235353.31 | -2060171.30 | -313.58 | 6574.57 | 2843.99 | 147.412 | -75.175 | 182.274 | -13.362 |
| 2012-05-01 18:08:00 | 65280 | 7345076.80 | 1627726.25 | -1886452.73 | -690.90 | 6501.18 | 2945.15 | 150.995 | -76.564 | 178.801 | -12.329 |
| 2012-05-01 18:09:00 | 65340 | 7292352.12 | 2015094.59 | -1706934.13 | -1066.15 | 6407.79 | 3037.27 | 155.315 | -77.898 | 175.191 | -11.358 |
| 2012-05-01 18:10:00 | 65400 | 7217203.37 | 2396266.46 | -1522166.69 | -1438.18 | 6294.68 | 3120.07 | 160.597 | -79.157 | 171.444 | -10.460 |
| 2012-05-01 18:11:00 | 65460 | 7119858.59 | 2770068.75 | -1332717.95 | -1805.83 | 6162.20 | 3193.28 | 167.127 | -80.311 | 167.563 | -9.644 |
| 2012-05-01 18:12:00 | 65520 | 7000614.26 | 3135350.71 | -1139169.99 | -2167.97 | 6010.74 | 3256.67 | 175.222 | -81.319 | 163.556 | -8.923 |
| 2012-05-01 18:13:00 | 65580 | 6859834.43 | 3490987.59 | -942117.69 | -2523.49 | 5840.78 | 3310.06 | 185.134 | -82.125 | 159.433 | -8.309 |
| 2012-05-01 18:14:00 | 65640 | 6697949.63 | 3835884.07 | -742166.88 | -2871.29 | 5652.82 | 3353.26 | 196.834 | -82.662 | 155.209 | -7.811 |
| 2012-05-01 18:15:00 | 65700 | 6515455.58 | 4168977.71 | -539932.47 | -3210.29 | 5447.45 | 3386.16 | 209.746 | -82.869 | 150.904 | -7.439 |
| 2012-05-01 18:16:00 | 65760 | 6312911.66 | 4489242.22 | -336036.55 | -3539.45 | 5225.29 | 3408.63 | 222.753 | -82.716 | 146.541 | -7.201 |
| 2012-05-01 18:17:00 | 65820 | 6090939.23 | 4795690.68 | -131106.46 | -3857.74 | 4987.03 | 3420.62 | 234.690 | -82.225 | 142.146 | -7.103 |
| 2012-05-01 18:18:00 | 65880 | 5850219.69 | 5087378.62 | 74227.15 | -4164.19 | 4733.40 | 3422.08 | 244.880 | -81.453 | 137.746 | -7.146 |
| 2012-05-01 18:19:00 | 65940 | 5591492.43 | 5363406.95 | 279332.27 | -4457.84 | 4465.18 | 3413.00 | 253.222 | -80.468 | 133.369 | -7.331 |
| 2012-05-01 18:20:00 | 66000 | 5315552.53 | 5622924.75 | 483577.49 | -4737.80 | 4183.19 | 3393.42 | 259.941 | -79.328 | 129.043 | -7.655 |
| 2012-05-01 18:21:00 | 66060 | 5023248.34 | 5865131.96 | 686333.99 | -5003.18 | 3888.31 | 3363.39 | 265.353 | -78.077 | 124.793 | -8.111 |
| 2012-05-01 18:22:00 | 66120 | 4715478.81 | 6089281.86 | 886977.42 | -5253.17 | 3581.44 | 3323.01 | 269.751 | -76.744 | 120.641 | -8.691 |
| 2012-05-01 18:23:00 | 66180 | 4393190.74 | 6294683.41 | 1084889.92 | -5487.01 | 3263.52 | 3272.38 | 273.370 | -75.353 | 116.604 | -9.386 |
| 2012-05-01 18:24:00 | 66240 | 4057375.89 | 6480703.38 | 1279462.00 | -5703.95 | 2935.55 | 3211.69 | 276.390 | -73.917 | 112.697 | -10.184 |
| 2012-05-01 18:25:00 | 66300 | 3709067.80 | 6646768.36 | 1470094.44 | -5903.34 | 2598.53 | 3141.10 | 278.945 | -72.446 | 108.930 | -11.076 |
| 2012-05-01 18:26:00 | 66360 | 3349338.72 | 6792366.53 | 1656200.14 | -6084.55 | 2253.50 | 3060.83 | 281.133 | -70.948 | 105.307 | -12.049 |
| 2012-05-01 18:27:00 | 66420 | 2979296.17 | 6917049.25 | 1837205.98 | -6247.03 | 1901.53 | 2971.15 | 283.030 | -69.428 | 101.831 | -13.094 |
| 2012-05-01 18:28:00 | 66480 | 2600079.55 | 7020432.50 | 2012554.56 | -6390.28 | 1543.70 | 2872.31 | 284.692 | -67.889 | 98.501 | -14.200 |
| 2012-05-01 18:29:00 | 66540 | 2212856.60 | 7102198.00 | 2181705.95 | -6513.84 | 1181.12 | 2764.63 | 286.164 | -66.335 | 95.315 | -15.358 |
| 2012-05-01 18:30:00 | 66600 | 1818819.78 | 7162094.28 | 2344139.40 | -6617.35 | 814.91 | 2648.43 | 287.478 | -64.767 | 92.266 | -16.560 |
| 2012-05-01 18:31:00 | 66660 | 1419182.53 | 7199937.38 | 2499354.89 | -6700.48 | 446.20 | 2524.09 | 288.661 | -63.187 | 89.349 | -17.799 |
| 2012-05-01 18:32:00 | 66720 | 1015175.54 | 7215611.50 | 2646874.74 | -6762.97 | 76.14 | 2391.98 | 289.734 | -61.596 | 86.556 | -19.069 |
| 2012-05-01 18:33:00 | 66780 | 608042.89 | 7209069.25 | 2786245.05 | -6804.64 | -294.15 | 2252.51 | 290.714 | -59.994 | 83.879 | -20.365 |
| 2012-05-01 18:34:00 | 66840 | 199038.19 | 7180331.89 | 2917037.13 | -6825.35 | -663.51 | 2106.11 | 291.615 | -58.383 | 81.310 | -21.681 |
| 2012-05-01 18:35:00 | 66900 | -210579.27 | 7129489.17 | 3038848.84 | -6825.06 | -1030.81 | 1953.24 | 292.448 | -56.762 | 78.840 | -23.015 |
| 2012-05-01 18:36:00 | 66960 | -619548.51 | 7056699.09 | 3151305.77 | -6803.75 | -1394.90 | 1794.37 | 293.222 | -55.131 | 76.463 | -24.362 |
| 2012-05-01 18:37:00 | 67020 | -1026610.69 | 6962187.33 | 3254062.49 | -6761.51 | -1754.68 | 1629.98 | 293.946 | -53.491 | 74.168 | -25.721 |
| 2012-05-01 18:38:00 | 67080 | -1430513.08 | 6846246.62 | 3346803.51 | -6698.45 | -2109.02 | 1460.60 | 294.625 | -51.842 | 71.949 | -27.087 |
| 2012-05-01 18:39:00 | 67140 | -1830012.90 | 6709235.72 | 3429244.32 | -6614.79 | -2456.84 | 1286.73 | 295.267 | -50.182 | 69.798 | -28.461 |
| 2012-05-01 18:40:00 | 67200 | -2223881.21 | 6551578.32 | 3501132.26 | -6510.79 | -2797.06 | 1108.92 | 295.874 | -48.513 | 67.707 | -29.838 |
| 2012-05-01 18:41:00 | 67260 | -2610906.66 | 6373761.74 | 3562247.22 | -6386.76 | -3128.64 | 927.72 | 296.453 | -46.832 | 65.669 | -31.219 |
| 2012-05-01 18:42:00 | 67320 | -2989899.29 | 6176335.31 | 3612402.40 | -6243.09 | -3450.55 | 743.69 | 297.006 | -45.141 | 63.679 | -32.602 |
| 2012-05-01 18:43:00 | 67380 | -3359694.17 | 5959908.73 | 3651444.81 | -6080.25 | -3761.81 | 557.39 | 297.537 | -43.437 | 61.728 | -33.985 |
| 2012-05-01 18:44:00 | 67440 | -3719155.01 | 5725150.07 | 3679255.77 | -5898.72 | -4061.46 | 369.41 | 298.049 | -41.721 | 59.812 | -35.368 |
| 2012-05-01 18:45:00 | 67500 | -4067177.64 | 5472783.74 | 3695751.20 | -5699.07 | -4348.58 | 180.31 | 298.545 | -39.991 | 57.922 | -36.749 |
| 2012-05-01 18:46:00 | 67560 | -4402693.44 | 5203588.20 | 3700881.94 | -5481.93 | -4622.29 | -9.32 | 299.028 | -38.245 | 56.055 | -38.128 |
| 2012-05-01 18:47:00 | 67620 | -4724672.59 | 4918393.51 | 3694633.83 | -5247.97 | -4881.75 | -198.89 | 299.499 | -36.484 | 54.202 | -39.504 |
| 2012-05-01 18:48:00 | 67680 | -5032127.26 | 4618078.76 | 3677027.73 | -4997.90 | -5126.17 | -387.82 | 299.961 | -34.705 | 52.359 | -40.875 |
| 2012-05-01 18:49:00 | 67740 | -5324114.60 | 4303569.30 | 3648119.48 | -4732.52 | -5354.79 | -575.53 | 300.418 | -32.906 | 50.520 | -42.241 |
| 2012-05-01 18:50:00 | 67800 | -5599739.64 | 3975833.88 | 3607999.65 | -4452.64 | -5566.92 | -761.45 | 300.870 | -31.085 | 48.678 | -43.602 |
| 2012-05-01 18:51:00 | 67860 | -5858158.03 | 3635881.65 | 3556793.29 | -4159.11 | -5761.91 | -944.99 | 301.320 | -29.241 | 46.827 | -44.955 |
| 2012-05-01 18:52:00 | 67920 | -6098578.60 | 3284758.97 | 3494659.50 | -3852.86 | -5939.17 | -1125.60 | 301.772 | -27.369 | 44.960 | -46.301 |
| 2012-05-01 18:53:00 | 67980 | -6320265.75 | 2923546.23 | 3421790.90 | -3534.83 | -6098.16 | -1302.73 | 302.227 | -25.467 | 43.072 | -47.638 |
| 2012-05-01 18:54:00 | 68040 | -6522541.68 | 2553354.47 | 3338413.06 | -3205.99 | -6238.40 | -1475.82 | 302.688 | -23.532 | 41.155 | -48.965 |
| 2012-05-01 18:55:00 | 68100 | -6704788.47 | 2175321.94 | 3244783.75 | -2867.36 | -6359.46 | -1644.35 | 303.160 | -21.557 | 39.202 | -50.280 |
| 2012-05-01 18:56:00 | 68160 | -6866449.89 | 1790610.59 | 3141192.12 | -2519.99 | -6460.97 | -1807.81 | 303.644 | -19.539 | 37.204 | -51.583 |
| 2012-05-01 18:57:00 | 68220 | -7007033.10 | 1400402.50 | 3027957.85 | -2164.93 | -6542.64 | -1965.69 | 304.147 | -17.471 | 35.154 | -52.872 |
| 2012-05-01 18:58:00 | 68280 | -7126110.11 | 1005896.21 | 2905430.05 | -1803.30 | -6604.21 | -2117.52 | 304.673 | -15.344 | 33.042 | -54.145 |
| 2012-05-01 18:59:00 | 68340 | -7223319.05 | 608303.07 | 2773986.28 | -1436.18 | -6645.51 | -2262.82 | 305.228 | -13.150 | 30.860 | -55.401 |
| 2012-05-01 19:00:00 | 68400 | -7298365.25 | 208843.49 | 2634031.29 | -1064.73 | -6666.41 | -2401.15 | 305.820 | -10.877 | 28.596 | -56.636 |
| 2012-05-01 19:01:00 | 68460 | -7351022.05 | -191256.80 | 2485995.80 | -690.06 | -6666.86 | -2532.10 | 306.457 | -8.511 | 26.240 | -57.849 |
| 2012-05-01 19:02:00 | 68520 | -7381131.53 | -590770.49 | 2330335.15 | -313.34 | -6646.86 | -2655.26 | 307.152 | -6.033 | 23.779 | -59.037 |
| 2012-05-01 19:03:00 | 68580 | -7388604.86 | -988472.46 | 2167527.92 | 64.28 | -6606.48 | -2770.26 | 307.920 | -3.422 | 21.200 | -60.197 |
| 2012-05-01 19:04:00 | 68640 | -7373422.61 | -1383143.52 | 1998074.41 | 441.65 | -6545.86 | -2876.75 | 308.780 | -0.650 | 18.491 | -61.324 |
| 2012-05-01 19:05:00 | 68700 | -7335634.70 | -1773574.13 | 1822495.14 | 817.61 | -6465.17 | -2974.40 | 309.760 | 2.321 | 15.636 | -62.416 |
| 2012-05-01 19:06:00 | 68760 | -7275360.25 | -2158568.08 | 1641329.20 | 1191.01 | -6364.68 | -3062.92 | 310.897 | 5.537 | 12.622 | -63.467 |
| 2012-05-01 19:07:00 | 68820 | -7192787.16 | -2536946.16 | 1455132.66 | 1560.71 | -6244.70 | -3142.04 | 312.242 | 9.060 | 9.433 | -64.473 |
| 2012-05-01 19:08:00 | 68880 | -7088171.50 | -2907549.68 | 1264476.79 | 1925.58 | -6105.60 | -3211.53 | 313.874 | 12.973 | 6.055 | -65.426 |
| 2012-05-01 19:09:00 | 68940 | -6961836.69 | -3269244.10 | 1069946.38 | 2284.50 | -5947.81 | -3271.16 | 315.909 | 17.380 | 2.476 | -66.322 |
| 2012-05-01 19:10:00 | 69000 | -6814172.48 | -3620922.37 | 872137.90 | 2636.38 | -5771.81 | -3320.77 | 318.535 | 22.418 | 358.685 | -67.152 |
| 2012-05-01 19:11:00 | 69060 | -6645633.74 | -3961508.32 | 671657.70 | 2980.14 | -5578.16 | -3360.20 | 322.067 | 28.247 | 354.676 | -67.910 |
| 2012-05-01 19:12:00 | 69120 | -6456739.04 | -4289959.97 | 469120.16 | 3314.74 | -5367.44 | -3389.33 | 327.067 | 35.019 | 350.448 | -68.587 |
| 2012-05-01 19:13:00 | 69180 | -6248069.02 | -4605272.61 | 265145.82 | 3639.15 | -5140.30 | -3408.08 | 334.602 | 42.754 | 346.007 | -69.176 |
| 2012-05-01 19:14:00 | 69240 | -6020264.64 | -4906481.88 | 60359.48 | 3952.39 | -4897.45 | -3416.39 | 346.735 | 50.984 | 341.368 | -69.668 |
| 2012-05-01 19:15:00 | 69300 | -5774025.19 | -5192666.69 | -144611.70 | 4253.50 | -4639.62 | -3414.24 | 6.780 | 57.985 | 336.554 | -70.055 |
| 2012-05-01 19:16:00 | 69360 | -5510106.12 | -5462951.99 | -349140.10 | 4541.56 | -4367.60 | -3401.63 | 35.141 | 60.355 | 331.602 | -70.332 |
| 2012-05-01 19:17:00 | 69420 | -5229316.78 | -5716511.39 | -552599.54 | 4815.70 | -4082.23 | -3378.62 | 61.737 | 56.340 | 326.553 | -70.493 |
| 2012-05-01 19:18:00 | 69480 | -4932517.89 | -5952569.72 | -754367.22 | 5075.07 | -3784.38 | -3345.26 | 79.220 | 48.722 | 321.458 | -70.535 |
| 2012-05-01 19:19:00 | 69540 | -4620618.95 | -6170405.31 | -953825.53 | 5318.90 | -3474.96 | -3301.66 | 89.755 | 40.539 | 316.372 | -70.457 |
| 2012-05-01 19:20:00 | 69600 | -4294575.46 | -6369352.19 | -1150364.02 | 5546.44 | -3154.91 | -3247.95 | 96.395 | 33.065 | 311.348 | -70.260 |
| 2012-05-01 19:21:00 | 69660 | -3955386.02 | -6548802.10 | -1343381.17 | 5756.99 | -2825.22 | -3184.31 | 100.863 | 26.573 | 306.435 | -69.947 |
| 2012-05-01 19:22:00 | 69720 | -3604089.26 | -6708206.33 | -1532286.28 | 5949.91 | -2486.90 | -3110.92 | 104.048 | 20.985 | 301.676 | -69.523 |
| 2012-05-01 19:23:00 | 69780 | -3241760.71 | -6847077.37 | -1716501.21 | 6124.62 | -2140.96 | -3028.01 | 106.422 | 16.141 | 297.104 | -68.995 |
| 2012-05-01 19:24:00 | 69840 | -2869509.55 | -6964990.40 | -1895462.15 | 6280.58 | -1788.47 | -2935.83 | 108.259 | 11.884 | 292.743 | -68.369 |
| 2012-05-01 19:25:00 | 69900 | -2488475.22 | -7061584.54 | -2068621.34 | 6417.32 | -1430.51 | -2834.67 | 109.720 | 8.089 | 288.605 | -67.653 |
| 2012-05-01 19:26:00 | 69960 | -2099823.96 | -7136563.99 | -2235448.72 | 6534.42 | -1068.17 | -2724.83 | 110.910 | 4.657 | 284.696 | -66.855 |
| 2012-05-01 19:27:00 | 70020 | -1704745.28 | -7189698.89 | -2395433.53 | 6631.51 | -702.54 | -2606.64 | 111.897 | 1.512 | 281.014 | -65.983 |
| 2012-05-01 19:28:00 | 70080 | -1304448.36 | -7220826.06 | -2548085.90 | 6708.31 | -334.76 | -2480.47 | 112.730 | -1.403 | 277.552 | -65.045 |
| 2012-05-01 19:29:00 | 70140 | -900158.39 | -7229849.45 | -2692938.29 | 6764.58 | 34.06 | -2346.71 | 113.440 | -4.131 | 274.300 | -64.046 |
| 2012-05-01 19:30:00 | 70200 | -493112.82 | -7216740.50 | -2829546.93 | 6800.14 | 402.80 | -2205.75 | 114.054 | -6.707 | 271.244 | -62.993 |
| 2012-05-01 19:31:00 | 70260 | -84557.64 | -7181538.19 | -2957493.17 | 6814.89 | 770.32 | -2058.03 | 114.588 | -9.157 | 268.371 | -61.892 |
| 2012-05-01 19:32:00 | 70320 | 324256.40 | -7124348.95 | -3076384.78 | 6808.77 | 1135.51 | -1904.01 | 115.056 | -11.502 | 265.666 | -60.747 |
| 2012-05-01 19:33:00 | 70380 | 732077.62 | -7045346.39 | -3185857.11 | 6781.80 | 1497.24 | -1744.14 | 115.470 | -13.759 | 263.114 | -59.563 |
| 2012-05-01 19:34:00 | 70440 | 1137657.17 | -6944770.71 | -3285574.22 | 6734.07 | 1854.43 | -1578.92 | 115.837 | -15.941 | 260.702 | -58.344 |
| 2012-05-01 19:35:00 | 70500 | 1539752.86 | -6822928.09 | -3375229.93 | 6665.70 | 2205.96 | -1408.84 | 116.165 | -18.058 | 258.416 | -57.093 |
| 2012-05-01 19:36:00 | 70560 | 1937132.91 | -6680189.70 | -3454548.71 | 6576.92 | 2550.78 | -1234.44 | 116.457 | -20.121 | 256.243 | -55.813 |
| 2012-05-01 19:37:00 | 70620 | 2328579.71 | -6516990.63 | -3523286.58 | 6467.98 | 2887.81 | -1056.24 | 116.720 | -22.136 | 254.172 | -54.507 |
| 2012-05-01 19:38:00 | 70680 | 2712893.55 | -6333828.60 | -3581231.85 | 6339.21 | 3216.04 | -874.78 | 116.955 | -24.110 | 252.193 | -53.176 |
| 2012-05-01 19:39:00 | 70740 | 3088896.22 | -6131262.46 | -3628205.77 | 6191.01 | 3534.45 | -690.62 | 117.167 | -26.047 | 250.294 | -51.823 |
| 2012-05-01 19:40:00 | 70800 | 3455434.67 | -5909910.52 | -3664063.08 | 6023.82 | 3842.07 | -504.32 | 117.357 | -27.952 | 248.466 | -50.449 |
| 2012-05-01 19:41:00 | 70860 | 3811384.52 | -5670448.68 | -3688692.49 | 5838.14 | 4137.96 | -316.45 | 117.528 | -29.828 | 246.702 | -49.055 |
| 2012-05-01 19:42:00 | 70920 | 4155653.45 | -5413608.43 | -3702017.04 | 5634.55 | 4421.20 | -127.58 | 117.681 | -31.680 | 244.992 | -47.643 |
| 2012-05-01 19:43:00 | 70980 | 4487184.62 | -5140174.63 | -3703994.31 | 5413.66 | 4690.93 | 61.70 | 117.818 | -33.509 | 243.329 | -46.214 |
| 2012-05-01 19:44:00 | 71040 | 4804959.87 | -4850983.15 | -3694616.64 | 5176.14 | 4946.32 | 250.82 | 117.940 | -35.319 | 241.706 | -44.768 |
| 2012-05-01 19:45:00 | 71100 | 5108002.86 | -4546918.34 | -3673911.09 | 4922.71 | 5186.59 | 439.20 | 118.047 | -37.110 | 240.117 | -43.306 |
| 2012-05-01 19:46:00 | 71160 | 5395382.06 | -4228910.37 | -3641939.44 | 4654.14 | 5410.98 | 626.26 | 118.141 | -38.885 | 238.554 | -41.829 |
| 2012-05-01 19:47:00 | 71220 | 5666213.67 | -3897932.41 | -3598798.01 | 4371.26 | 5618.81 | 811.43 | 118.223 | -40.646 | 237.012 | -40.337 |
| 2012-05-01 19:48:00 | 71280 | 5919664.30 | -3554997.68 | -3544617.37 | 4074.92 | 5809.43 | 994.14 | 118.293 | -42.393 | 235.483 | -38.830 |
| 2012-05-01 19:49:00 | 71340 | 6154953.58 | -3201156.36 | -3479561.97 | 3766.04 | 5982.27 | 1173.83 | 118.350 | -44.128 | 233.963 | -37.308 |
| 2012-05-01 19:50:00 | 71400 | 6371356.58 | -2837492.44 | -3403829.67 | 3445.54 | 6136.77 | 1349.94 | 118.397 | -45.853 | 232.445 | -35.772 |
| 2012-05-01 19:51:00 | 71460 | 6568206.06 | -2465120.37 | -3317651.12 | 3114.42 | 6272.46 | 1521.95 | 118.433 | -47.568 | 230.923 | -34.222 |
| 2012-05-01 19:52:00 | 71520 | 6744894.57 | -2085181.70 | -3221289.12 | 2773.68 | 6388.92 | 1689.30 | 118.457 | -49.273 | 229.390 | -32.656 |
| 2012-05-01 19:53:00 | 71580 | 6900876.33 | -1698841.58 | -3115037.81 | 2424.37 | 6485.79 | 1851.50 | 118.471 | -50.971 | 227.840 | -31.076 |
| 2012-05-01 19:54:00 | 71640 | 7035668.96 | -1307285.21 | -2999221.81 | 2067.56 | 6562.75 | 2008.05 | 118.474 | -52.661 | 226.265 | -29.480 |
| 2012-05-01 19:55:00 | 71700 | 7148854.99 | -911714.19 | -2874195.18 | 1704.34 | 6619.57 | 2158.45 | 118.465 | -54.344 | 224.659 | -27.869 |
| 2012-05-01 19:56:00 | 71760 | 7240083.23 | -513342.87 | -2740340.44 | 1335.82 | 6656.07 | 2302.24 | 118.445 | -56.021 | 223.012 | -26.242 |
| 2012-05-01 19:57:00 | 71820 | 7309069.83 | -113394.58 | -2598067.35 | 963.14 | 6672.13 | 2438.99 | 118.413 | -57.692 | 221.317 | -24.598 |
| 2012-05-01 19:58:00 | 71880 | 7355599.24 | 286902.08 | -2447811.69 | 587.44 | 6667.68 | 2568.26 | 118.367 | -59.358 | 219.563 | -22.937 |
| 2012-05-01 19:59:00 | 71940 | 7379524.91 | 686317.06 | -2290033.92 | 209.87 | 6642.74 | 2689.66 | 118.308 | -61.019 | 217.741 | -21.259 |
| 2012-05-01 20:00:00 | 72000 | 7380769.79 | 1083622.63 | -2125217.80 | -168.40 | 6597.38 | 2802.81 | 118.234 | -62.675 | 215.837 | -19.563 |
| 2012-05-01 20:01:00 | 72060 | 7359326.58 | 1477597.17 | -1953868.89 | -546.20 | 6531.74 | 2907.36 | 118.142 | -64.328 | 213.839 | -17.849 |
| 2012-05-01 20:02:00 | 72120 | 7315257.84 | 1867028.95 | -1776513.01 | -922.39 | 6446.00 | 3002.99 | 118.032 | -65.977 | 211.731 | -16.116 |
| 2012-05-01 20:03:00 | 72180 | 7248695.81 | 2250719.87 | -1593694.64 | -1295.79 | 6340.42 | 3089.40 | 117.900 | -67.622 | 209.498 | -14.367 |
| 2012-05-01 20:04:00 | 72240 | 7159842.04 | 2627489.19 | -1405975.24 | -1665.25 | 6215.33 | 3166.31 | 117.742 | -69.264 | 207.119 | -12.600 |
| 2012-05-01 20:05:00 | 72300 | 7048966.84 | 2996177.14 | -1213931.50 | -2029.65 | 6071.11 | 3233.50 | 117.552 | -70.903 | 204.574 | -10.820 |
| 2012-05-01 20:06:00 | 72360 | 6916408.43 | 3355648.57 | -1018153.62 | -2387.84 | 5908.19 | 3290.75 | 117.325 | -72.539 | 201.838 | -9.028 |
| 2012-05-01 20:07:00 | 72420 | 6762571.96 | 3704796.46 | -819243.44 | -2738.73 | 5727.08 | 3337.89 | 117.049 | -74.173 | 198.886 | -7.230 |
| 2012-05-01 20:08:00 | 72480 | 6587928.28 | 4042545.33 | -617812.61 | -3081.24 | 5528.32 | 3374.75 | 116.709 | -75.805 | 195.689 | -5.433 |
| 2012-05-01 20:09:00 | 72540 | 6393012.52 | 4367854.64 | -414480.69 | -3414.29 | 5312.53 | 3401.24 | 116.283 | -77.434 | 192.217 | -3.650 |
| 2012-05-01 20:10:00 | 72600 | 6178422.46 | 4679721.99 | -209873.22 | -3736.88 | 5080.37 | 3417.26 | 115.734 | -79.061 | 188.438 | -1.896 |
| 2012-05-01 20:11:00 | 72660 | 5944816.68 | 4977186.24 | -4619.82 | -4047.99 | 4832.55 | 3422.76 | 115.000 | -80.686 | 184.325 | -0.193 |
| 2012-05-01 20:12:00 | 72720 | 5692912.57 | 5259330.53 | 200647.79 | -4346.67 | 4569.84 | 3417.73 | 113.968 | -82.307 | 179.851 | 1.430 |
| 2012-05-01 20:13:00 | 72780 | 5423484.11 | 5525285.14 | 405297.74 | -4631.98 | 4293.04 | 3402.18 | 112.404 | -83.924 | 175.003 | 2.938 |
| 2012-05-01 20:14:00 | 72840 | 5137359.49 | 5774230.17 | 608699.98 | -4903.06 | 4003.00 | 3376.15 | 109.739 | -85.533 | 169.779 | 4.286 |
| 2012-05-01 20:15:00 | 72900 | 4835418.55 | 6005398.14 | 810228.22 | -5159.06 | 3700.62 | 3339.73 | 104.160 | -87.121 | 164.201 | 5.425 |
| 2012-05-01 20:16:00 | 72960 | 4518590.07 | 6218076.36 | 1009261.87 | -5399.18 | 3386.83 | 3293.02 | 85.948 | -88.620 | 158.314 | 6.307 |
| 2012-05-01 20:17:00 | 73020 | 4187848.88 | 6411609.16 | 1205188.01 | -5622.70 | 3062.60 | 3236.17 | 357.498 | -89.103 | 152.193 | 6.884 |
| 2012-05-01 20:18:00 | 73080 | 3844212.88 | 6585399.93 | 1397403.25 | -5828.90 | 2728.93 | 3169.36 | 319.268 | -87.767 | 145.938 | 7.126 |
| 2012-05-01 20:19:00 | 73140 | 3488739.82 | 6738913.02 | 1585315.61 | -6017.16 | 2386.85 | 3092.78 | 310.810 | -86.199 | 139.663 | 7.015 |
| 2012-05-01 20:20:00 | 73200 | 3122524.08 | 6871675.35 | 1768346.41 | -6186.90 | 2037.42 | 3006.68 | 307.278 | -84.596 | 133.487 | 6.560 |
| 2012-05-01 20:21:00 | 73260 | 2746693.25 | 6983277.93 | 1945931.98 | -6337.58 | 1681.71 | 2911.32 | 305.327 | -82.983 | 127.517 | 5.785 |
| 2012-05-01 20:22:00 | 73320 | 2362404.61 | 7073377.12 | 2117525.50 | -6468.75 | 1320.83 | 2807.00 | 304.071 | -81.364 | 121.838 | 4.732 |
| 2012-05-01 20:23:00 | 73380 | 1970841.56 | 7141695.68 | 2282598.66 | -6580.00 | 955.88 | 2694.03 | 303.180 | -79.741 | 116.506 | 3.450 |
| 2012-05-01 20:24:00 | 73440 | 1573209.92 | 7188023.67 | 2440643.29 | -6670.99 | 587.99 | 2572.77 | 302.504 | -78.116 | 111.554 | 1.987 |
| 2012-05-01 20:25:00 | 73500 | 1170734.21 | 7212219.05 | 2591172.94 | -6741.43 | 218.31 | 2443.60 | 301.962 | -76.489 | 106.987 | 0.390 |
| 2012-05-01 20:26:00 | 73560 | 764653.83 | 7214208.14 | 2733724.42 | -6791.11 | -152.02 | 2306.90 | 301.511 | -74.860 | 102.797 | -1.303 |
| 2012-05-01 20:27:00 | 73620 | 356219.19 | 7193985.85 | 2867859.21 | -6819.88 | -521.88 | 2163.11 | 301.123 | -73.229 | 98.961 | -3.058 |
| 2012-05-01 20:28:00 | 73680 | -53312.15 | 7151615.63 | 2993164.81 | -6827.66 | -890.10 | 2012.67 | 300.779 | -71.595 | 95.453 | -4.853 |
| 2012-05-01 20:29:00 | 73740 | -462679.38 | 7087229.35 | 3109256.03 | -6814.42 | -1255.56 | 1856.05 | 300.467 | -69.959 | 92.242 | -6.670 |
| 2012-05-01 20:30:00 | 73800 | -870622.38 | 7001026.76 | 3215776.17 | -6780.20 | -1617.12 | 1693.72 | 300.179 | -68.320 | 89.297 | -8.494 |
| 2012-05-01 20:31:00 | 73860 | -1275885.61 | 6893274.96 | 3312398.13 | -6725.11 | -1973.68 | 1526.19 | 299.908 | -66.678 | 86.589 | -10.318 |
| 2012-05-01 20:32:00 | 73920 | -1677222.03 | 6764307.47 | 3398825.41 | -6649.34 | -2324.13 | 1353.98 | 299.651 | -65.034 | 84.091 | -12.134 |
| 2012-05-01 20:33:00 | 73980 | -2073396.91 | 6614523.20 | 3474793.00 | -6553.11 | -2667.39 | 1177.62 | 299.402 | -63.385 | 81.779 | -13.940 |
| 2012-05-01 20:34:00 | 74040 | -2463191.71 | 6444385.21 | 3540068.22 | -6436.72 | -3002.41 | 997.66 | 299.160 | -61.733 | 79.630 | -15.733 |
| 2012-05-01 20:35:00 | 74100 | -2845407.81 | 6254419.21 | 3594451.42 | -6300.55 | -3328.16 | 814.65 | 298.922 | -60.077 | 77.626 | -17.511 |
| 2012-05-01 20:36:00 | 74160 | -3218870.26 | 6045211.94 | 3637776.55 | -6145.01 | -3643.62 | 629.15 | 298.686 | -58.417 | 75.747 | -19.275 |
| 2012-05-01 20:37:00 | 74220 | -3582431.33 | 5817409.31 | 3669911.71 | -5970.59 | -3947.84 | 441.75 | 298.451 | -56.752 | 73.980 | -21.023 |
| 2012-05-01 20:38:00 | 74280 | -3934974.12 | 5571714.34 | 3690759.51 | -5777.83 | -4239.88 | 253.01 | 298.215 | -55.081 | 72.309 | -22.756 |
| 2012-05-01 20:39:00 | 74340 | -4275415.97 | 5308885.03 | 3700257.37 | -5567.33 | -4518.84 | 63.51 | 297.977 | -53.405 | 70.724 | -24.475 |
| 2012-05-01 20:40:00 | 74400 | -4602711.78 | 5029731.92 | 3698377.65 | -5339.74 | -4783.87 | -126.15 | 297.736 | -51.722 | 69.214 | -26.180 |
| 2012-05-01 20:41:00 | 74460 | -4915857.22 | 4735115.58 | 3685127.80 | -5095.77 | -5034.15 | -315.39 | 297.491 | -50.032 | 67.768 | -27.872 |
| 2012-05-01 20:42:00 | 74520 | -5213891.83 | 4425943.95 | 3660550.23 | -4836.18 | -5268.92 | -503.64 | 297.241 | -48.335 | 66.377 | -29.551 |
| 2012-05-01 20:43:00 | 74580 | -5495901.93 | 4103169.46 | 3624722.20 | -4561.76 | -5487.47 | -690.31 | 296.985 | -46.629 | 65.035 | -31.218 |
| 2012-05-01 20:44:00 | 74640 | -5761023.41 | 3767786.11 | 3577755.59 | -4273.37 | -5689.11 | -874.83 | 296.722 | -44.914 | 63.734 | -32.874 |
| 2012-05-01 20:45:00 | 74700 | -6008444.40 | 3420826.35 | 3519796.45 | -3971.89 | -5873.25 | -1056.64 | 296.450 | -43.189 | 62.466 | -34.519 |
| 2012-05-01 20:46:00 | 74760 | -6237407.68 | 3063357.91 | 3451024.62 | -3658.27 | -6039.31 | -1235.17 | 296.170 | -41.453 | 61.226 | -36.154 |
| 2012-05-01 20:47:00 | 74820 | -6447213.01 | 2696480.44 | 3371653.09 | -3333.46 | -6186.80 | -1409.87 | 295.879 | -39.705 | 60.007 | -37.778 |
| 2012-05-01 20:48:00 | 74880 | -6637219.24 | 2321322.17 | 3281927.35 | -2998.47 | -6315.27 | -1580.22 | 295.577 | -37.944 | 58.804 | -39.393 |
| 2012-05-01 20:49:00 | 74940 | -6806846.26 | 1939036.37 | 3182124.62 | -2654.33 | -6424.33 | -1745.68 | 295.262 | -36.167 | 57.611 | -40.999 |
| 2012-05-01 20:50:00 | 75000 | -6955576.70 | 1550797.82 | 3072552.95 | -2302.10 | -6513.65 | -1905.77 | 294.933 | -34.374 | 56.423 | -42.595 |
| 2012-05-01 20:51:00 | 75060 | -7082957.51 | 1157799.17 | 2953550.30 | -1942.86 | -6582.96 | -2059.97 | 294.588 | -32.563 | 55.234 | -44.182 |
| 2012-05-01 20:52:00 | 75120 | -7188601.29 | 761247.30 | 2825483.42 | -1577.71 | -6632.06 | -2207.83 | 294.225 | -30.731 | 54.037 | -45.761 |
| 2012-05-01 20:53:00 | 75180 | -7272187.46 | 362359.57 | 2688746.78 | -1207.79 | -6660.80 | -2348.89 | 293.843 | -28.876 | 52.828 | -47.330 |
| 2012-05-01 20:54:00 | 75240 | -7333463.16 | -37639.88 | 2543761.26 | -834.22 | -6669.11 | -2482.72 | 293.438 | -26.995 | 51.600 | -48.890 |
| 2012-05-01 20:55:00 | 75300 | -7372244.02 | -437523.91 | 2390972.94 | -458.15 | -6656.95 | -2608.92 | 293.009 | -25.085 | 50.345 | -50.441 |
| 2012-05-01 20:56:00 | 75360 | -7388414.62 | -836066.11 | 2230851.63 | -80.74 | -6624.39 | -2727.09 | 292.551 | -23.142 | 49.057 | -51.983 |
| 2012-05-01 20:57:00 | 75420 | -7381928.90 | -1232044.57 | 2063889.47 | 296.87 | -6571.52 | -2836.89 | 292.061 | -21.162 | 47.728 | -53.514 |
| 2012-05-01 20:58:00 | 75480 | -7352810.15 | -1624245.61 | 1890599.40 | 673.50 | -6498.51 | -2937.97 | 291.535 | -19.138 | 46.348 | -55.035 |
| 2012-05-01 20:59:00 | 75540 | -7301150.95 | -2011467.46 | 1711513.59 | 1048.02 | -6405.59 | -3030.03 | 290.966 | -17.066 | 44.908 | -56.544 |
| 2012-05-01 21:00:00 | 75600 | -7227112.87 | -2392523.96 | 1527181.77 | 1419.28 | -6293.05 | -3112.79 | 290.348 | -14.936 | 43.395 | -58.040 |
| 2012-05-01 21:01:00 | 75660 | -7130925.88 | -2766248.12 | 1338169.59 | 1786.13 | -6161.24 | -3186.00 | 289.673 | -12.740 | 41.797 | -59.523 |
| 2012-05-01 21:02:00 | 75720 | -7012887.65 | -3131495.72 | 1145056.86 | 2147.46 | -6010.57 | -3249.44 | 288.931 | -10.467 | 40.098 | -60.991 |
| 2012-05-01 21:03:00 | 75780 | -6873362.61 | -3487148.75 | 948435.78 | 2502.18 | -5841.51 | -3302.92 | 288.108 | -8.103 | 38.281 | -62.441 |
| 2012-05-01 21:04:00 | 75840 | -6712780.77 | -3832118.79 | 748909.11 | 2849.18 | -5654.56 | -3346.27 | 287.188 | -5.630 | 36.324 | -63.872 |
| 2012-05-01 21:05:00 | 75900 | -6531636.42 | -4165350.35 | 547088.39 | 3187.42 | -5450.32 | -3379.37 | 286.150 | -3.029 | 34.202 | -65.281 |
| 2012-05-01 21:06:00 | 75960 | -6330486.58 | -4485824.05 | 343592.00 | 3515.86 | -5229.41 | -3402.11 | 284.966 | -0.271 | 31.888 | -66.664 |
| 2012-05-01 21:07:00 | 76020 | -6109949.27 | -4792559.69 | 139043.34 | 3833.50 | -4992.50 | -3414.44 | 283.599 | 2.675 | 29.345 | -68.016 |
| 2012-05-01 21:08:00 | 76080 | -5870701.62 | -5084619.22 | -65931.13 | 4139.38 | -4740.33 | -3416.30 | 281.996 | 5.852 | 26.534 | -69.332 |
| 2012-05-01 21:09:00 | 76140 | -5613477.80 | -5361109.59 | -270703.72 | 4432.56 | -4473.67 | -3407.71 | 280.085 | 9.313 | 23.407 | -70.605 |
| 2012-05-01 21:10:00 | 76200 | -5339066.74 | -5621185.44 | -474647.49 | 4712.14 | -4193.32 | -3388.68 | 277.762 | 13.124 | 19.910 | -71.826 |
| 2012-05-01 21:11:00 | 76260 | -5048309.73 | -5864051.63 | -677138.09 | 4977.28 | -3900.16 | -3359.28 | 274.874 | 17.365 | 15.981 | -72.984 |
| 2012-05-01 21:12:00 | 76320 | -4742097.88 | -6088965.69 | -877555.72 | 5227.17 | -3595.07 | -3319.60 | 271.181 | 22.120 | 11.556 | -74.067 |
| 2012-05-01 21:13:00 | 76380 | -4421369.36 | -6295240.02 | -1075286.94 | 5461.05 | -3278.99 | -3269.76 | 266.314 | 27.462 | 6.571 | -75.056 |
| 2012-05-01 21:14:00 | 76440 | -4087106.57 | -6482243.97 | -1269726.61 | 5678.20 | -2952.89 | -3209.91 | 259.674 | 33.385 | 0.972 | -75.934 |
| 2012-05-01 21:15:00 | 76500 | -3740333.12 | -6649405.76 | -1460279.65 | 5877.96 | -2617.75 | -3140.24 | 250.329 | 39.655 | 354.735 | -76.678 |
| 2012-05-01 21:16:00 | 76560 | -3382110.79 | -6796214.20 | -1646362.88 | 6059.73 | -2274.61 | -3060.95 | 237.034 | 45.540 | 347.882 | -77.264 |
| 2012-05-01 21:17:00 | 76620 | -3013536.21 | -6922220.23 | -1827406.78 | 6222.95 | -1924.52 | -2972.30 | 219.043 | 49.569 | 340.504 | -77.669 |
| 2012-05-01 21:18:00 | 76680 | -2635737.61 | -7027038.29 | -2002857.24 | 6367.12 | -1568.53 | -2874.55 | 198.257 | 50.069 | 332.768 | -77.877 |
| 2012-05-01 21:19:00 | 76740 | -2249871.35 | -7110347.48 | -2172177.21 | 6491.81 | -1207.74 | -2768.01 | 179.249 | 46.790 | 324.906 | -77.875 |
| 2012-05-01 21:20:00 | 76800 | -1857118.44 | -7171892.52 | -2334848.34 | 6596.62 | -843.24 | -2652.98 | 164.795 | 41.210 | 317.175 | -77.665 |
| 2012-05-01 21:21:00 | 76860 | -1458680.93 | -7211484.55 | -2490372.54 | 6681.24 | -476.16 | -2529.84 | 154.577 | 34.953 | 309.806 | -77.255 |
| 2012-05-01 21:22:00 | 76920 | -1055778.30 | -7229001.72 | -2638273.53 | 6745.42 | -107.60 | -2398.94 | 147.343 | 28.922 | 302.968 | -76.664 |
| 2012-05-01 21:23:00 | 76980 | -649643.75 | -7224389.51 | -2778098.26 | 6788.95 | 261.30 | -2260.69 | 142.067 | 23.442 | 296.754 | -75.914 |
| 2012-05-01 21:24:00 | 77040 | -241520.42 | -7197660.99 | -2909418.27 | 6811.69 | 629.42 | -2115.52 | 138.081 | 18.558 | 291.185 | -75.029 |
| 2012-05-01 21:25:00 | 77100 | 167342.31 | -7148896.71 | -3031831.05 | 6813.59 | 995.64 | -1963.86 | 134.966 | 14.208 | 286.238 | -74.031 |
| 2012-05-01 21:26:00 | 77160 | 575692.68 | -7078244.53 | -3144961.22 | 6794.62 | 1358.84 | -1806.18 | 132.458 | 10.309 | 281.860 | -72.938 |
| 2012-05-01 21:27:00 | 77220 | 982280.29 | -6985919.16 | -3248461.68 | 6754.84 | 1717.90 | -1642.95 | 130.386 | 6.777 | 277.988 | -71.767 |
| 2012-05-01 21:28:00 | 77280 | 1385859.94 | -6872201.55 | -3342014.72 | 6694.38 | 2071.73 | -1474.68 | 128.636 | 3.545 | 274.556 | -70.532 |
| 2012-05-01 21:29:00 | 77340 | 1785195.42 | -6737438.03 | -3425332.93 | 6613.41 | 2419.25 | -1301.88 | 127.127 | 0.555 | 271.502 | -69.243 |
| 2012-05-01 21:30:00 | 77400 | 2179063.24 | -6582039.30 | -3498160.13 | 6512.17 | 2759.39 | -1125.07 | 125.805 | -2.235 | 268.774 | -67.909 |
| 2012-05-01 21:31:00 | 77460 | 2566256.37 | -6406479.19 | -3560272.14 | 6390.97 | 3091.12 | -944.80 | 124.628 | -4.861 | 266.323 | -66.538 |
| 2012-05-01 21:32:00 | 77520 | 2945587.92 | -6211293.27 | -3611477.50 | 6250.18 | 3413.42 | -761.61 | 123.566 | -7.352 | 264.109 | -65.134 |
| 2012-05-01 21:33:00 | 77580 | 3315894.76 | -5997077.23 | -3651618.02 | 6090.22 | 3725.30 | -576.06 | 122.596 | -9.730 | 262.098 | -63.702 |
| 2012-05-01 21:34:00 | 77640 | 3676041.09 | -5764485.08 | -3680569.33 | 5911.58 | 4025.80 | -388.73 | 121.700 | -12.012 | 260.260 | -62.245 |
| 2012-05-01 21:35:00 | 77700 | 4024921.88 | -5514227.24 | -3698241.24 | 5714.80 | 4314.00 | -200.18 | 120.865 | -14.212 | 258.572 | -60.767 |
| 2012-05-01 21:36:00 | 77760 | 4361466.28 | -5247068.34 | -3704578.07 | 5500.47 | 4589.02 | -10.99 | 120.079 | -16.343 | 257.012 | -59.270 |
| 2012-05-01 21:37:00 | 77820 | 4684640.90 | -4963824.95 | -3699558.78 | 5269.25 | 4850.02 | 178.26 | 119.332 | -18.414 | 255.563 | -57.755 |
| 2012-05-01 21:38:00 | 77880 | 4993452.96 | -4665363.12 | -3683197.10 | 5021.84 | 5096.18 | 366.99 | 118.619 | -20.434 | 254.210 | -56.223 |
| 2012-05-01 21:39:00 | 77940 | 5286953.42 | -4352595.75 | -3655541.51 | 4759.00 | 5326.74 | 554.63 | 117.931 | -22.407 | 252.940 | -54.677 |
| 2012-05-01 21:40:00 | 78000 | 5564239.79 | -4026479.84 | -3616675.10 | 4481.51 | 5541.01 | 740.59 | 117.265 | -24.342 | 251.741 | -53.116 |
| 2012-05-01 21:41:00 | 78060 | 5824459.02 | -3688013.59 | -3566715.30 | 4190.24 | 5738.32 | 924.31 | 116.615 | -26.241 | 250.604 | -51.542 |
| 2012-05-01 21:42:00 | 78120 | 6066810.09 | -3338233.36 | -3505813.63 | 3886.06 | 5918.05 | 1105.23 | 115.977 | -28.109 | 249.521 | -49.954 |
| 2012-05-01 21:43:00 | 78180 | 6290546.50 | -2978210.56 | -3434155.16 | 3569.91 | 6079.65 | 1282.78 | 115.349 | -29.949 | 248.483 | -48.354 |
| 2012-05-01 21:44:00 | 78240 | 6494978.63 | -2609048.32 | -3351958.05 | 3242.75 | 6222.62 | 1456.43 | 114.725 | -31.764 | 247.484 | -46.740 |
| 2012-05-01 21:45:00 | 78300 | 6679475.80 | -2231878.21 | -3259472.84 | 2905.58 | 6346.51 | 1625.63 | 114.104 | -33.557 | 246.517 | -45.113 |
| 2012-05-01 21:46:00 | 78360 | 6843468.37 | -1847856.72 | -3156981.74 | 2559.43 | 6450.94 | 1789.87 | 113.482 | -35.330 | 245.578 | -43.473 |
| 2012-05-01 21:47:00 | 78420 | 6986449.40 | -1458161.80 | -3044797.75 | 2205.37 | 6535.57 | 1948.64 | 112.857 | -37.085 | 244.660 | -41.819 |
| 2012-05-01 21:48:00 | 78480 | 7107976.36 | -1063989.16 | -2923263.78 | 1844.48 | 6600.15 | 2101.46 | 112.226 | -38.822 | 243.758 | -40.151 |
| 2012-05-01 21:49:00 | 78540 | 7207672.45 | -666548.71 | -2792751.54 | 1477.86 | 6644.47 | 2247.84 | 111.586 | -40.544 | 242.868 | -38.468 |
| 2012-05-01 21:50:00 | 78600 | 7285227.84 | -267060.78 | -2653660.48 | 1106.65 | 6668.38 | 2387.34 | 110.935 | -42.252 | 241.985 | -36.769 |
| 2012-05-01 21:51:00 | 78660 | 7340400.67 | 133247.62 | -2506416.54 | 731.97 | 6671.81 | 2519.53 | 110.269 | -43.947 | 241.104 | -35.053 |
| 2012-05-01 21:52:00 | 78720 | 7373017.83 | 533146.54 | -2351470.86 | 354.98 | 6654.74 | 2644.00 | 109.585 | -45.630 | 240.221 | -33.320 |
| 2012-05-01 21:53:00 | 78780 | 7382975.54 | 931406.91 | -2189298.43 | -23.15 | 6617.21 | 2760.36 | 108.881 | -47.301 | 239.330 | -31.567 |
| 2012-05-01 21:54:00 | 78840 | 7370239.69 | 1326804.33 | -2020396.62 | -401.28 | 6559.33 | 2868.26 | 108.151 | -48.961 | 238.426 | -29.793 |
| 2012-05-01 21:55:00 | 78900 | 7334846.04 | 1718122.81 | -1845283.66 | -778.22 | 6481.28 | 2967.35 | 107.393 | -50.610 | 237.504 | -27.996 |
| 2012-05-01 21:56:00 | 78960 | 7276900.10 | 2104158.58 | -1664497.06 | -1152.83 | 6383.28 | 3057.33 | 106.601 | -52.250 | 236.558 | -26.174 |
| 2012-05-01 21:57:00 | 79020 | 7196576.87 | 2483723.81 | -1478591.96 | -1523.94 | 6265.65 | 3137.92 | 105.771 | -53.880 | 235.581 | -24.325 |
| 2012-05-01 21:58:00 | 79080 | 7094120.33 | 2855650.23 | -1288139.41 | -1890.41 | 6128.72 | 3208.87 | 104.895 | -55.500 | 234.565 | -22.444 |
| 2012-05-01 21:59:00 | 79140 | 6969842.75 | 3218792.83 | -1093724.64 | -2251.12 | 5972.93 | 3269.96 | 103.967 | -57.111 | 233.501 | -20.530 |
| 2012-05-01 22:00:00 | 79200 | 6824123.71 | 3572033.38 | -895945.23 | -2604.94 | 5798.74 | 3321.00 | 102.979 | -58.713 | 232.381 | -18.577 |
| 2012-05-01 22:01:00 | 79260 | 6657408.99 | 3914283.88 | -695409.30 | -2950.79 | 5606.68 | 3361.82 | 101.920 | -60.304 | 231.191 | -16.581 |
| 2012-05-01 22:02:00 | 79320 | 6470209.23 | 4244489.98 | -492733.58 | -3287.60 | 5397.36 | 3392.30 | 100.779 | -61.885 | 229.920 | -14.536 |
| 2012-05-01 22:03:00 | 79380 | 6263098.35 | 4561634.27 | -288541.55 | -3614.33 | 5171.40 | 3412.35 | 99.540 | -63.456 | 228.549 | -12.437 |
| 2012-05-01 22:04:00 | 79440 | 6036711.83 | 4864739.41 | -83461.51 | -3929.96 | 4929.51 | 3421.90 | 98.187 | -65.014 | 227.060 | -10.275 |
| 2012-05-01 22:05:00 | 79500 | 5791744.72 | 5152871.19 | 121875.42 | -4233.52 | 4672.42 | 3420.91 | 96.697 | -66.560 | 225.429 | -8.042 |
| 2012-05-01 22:06:00 | 79560 | 5528949.54 | 5425141.46 | 326837.18 | -4524.07 | 4400.93 | 3409.39 | 95.041 | -68.091 | 223.625 | -5.728 |
| 2012-05-01 22:07:00 | 79620 | 5249133.98 | 5680710.88 | 530792.80 | -4800.72 | 4115.87 | 3387.38 | 93.185 | -69.605 | 221.611 | -3.323 |
| 2012-05-01 22:08:00 | 79680 | 4953158.34 | 5918791.55 | 733114.30 | -5062.60 | 3818.12 | 3354.94 | 91.084 | -71.099 | 219.343 | -0.816 |
| 2012-05-01 22:09:00 | 79740 | 4641932.95 | 6138649.47 | 933178.68 | -5308.92 | 3508.60 | 3312.16 | 88.679 | -72.568 | 216.760 | 1.807 |
| 2012-05-01 22:10:00 | 79800 | 4316415.32 | 6339606.82 | 1130369.86 | -5538.89 | 3188.26 | 3259.19 | 85.893 | -74.008 | 213.789 | 4.555 |
| 2012-05-01 22:11:00 | 79860 | 3977607.16 | 6521044.11 | 1324080.55 | -5751.82 | 2858.10 | 3196.18 | 82.625 | -75.410 | 210.336 | 7.432 |
| 2012-05-01 22:12:00 | 79920 | 3626551.32 | 6682402.05 | 1513714.18 | -5947.04 | 2519.12 | 3123.32 | 78.741 | -76.763 | 206.284 | 10.435 |
| 2012-05-01 22:13:00 | 79980 | 3264328.51 | 6823183.35 | 1698686.73 | -6123.95 | 2172.38 | 3040.85 | 74.068 | -78.052 | 201.490 | 13.536 |
| 2012-05-01 22:14:00 | 80040 | 2892053.97 | 6942954.27 | 1878428.59 | -6282.01 | 1818.96 | 2949.01 | 68.384 | -79.254 | 195.789 | 16.673 |
| 2012-05-01 22:15:00 | 80100 | 2510874.00 | 7041345.93 | 2052386.24 | -6420.73 | 1459.93 | 2848.09 | 61.426 | -80.337 | 189.014 | 19.727 |
| 2012-05-01 22:16:00 | 80160 | 2121962.41 | 7118055.49 | 2220024.08 | -6539.67 | 1096.40 | 2738.40 | 52.938 | -81.259 | 181.041 | 22.501 |
| 2012-05-01 22:17:00 | 80220 | 1726516.85 | 7172847.08 | 2380826.00 | -6638.47 | 729.51 | 2620.29 | 42.783 | -81.963 | 171.872 | 24.721 |
| 2012-05-01 22:18:00 | 80280 | 1325755.11 | 7205552.52 | 2534297.04 | -6716.83 | 360.39 | 2494.10 | 31.150 | -82.391 | 161.738 | 26.079 |
| 2012-05-01 22:19:00 | 80340 | 920911.33 | 7216071.85 | 2679964.91 | -6774.50 | -9.83 | 2360.25 | 18.704 | -82.496 | 151.136 | 26.346 |
| 2012-05-01 22:20:00 | 80400 | 513232.14 | 7204373.64 | 2817381.45 | -6811.32 | -380.01 | 2219.13 | 6.462 | -82.264 | 140.722 | 25.470 |
| 2012-05-01 22:21:00 | 80460 | 103972.83 | 7170495.03 | 2946124.01 | -6827.16 | -748.99 | 2071.19 | 355.344 | -81.724 | 131.088 | 23.612 |
| 2012-05-01 22:22:00 | 80520 | -305606.57 | 7114541.67 | 3065796.77 | -6821.98 | -1115.64 | 1916.88 | 345.825 | -80.930 | 122.586 | 21.059 |
| 2012-05-01 22:23:00 | 80580 | -714245.23 | 7036687.30 | 3176031.96 | -6795.81 | -1478.83 | 1756.68 | 337.938 | -79.944 | 115.309 | 18.110 |
| 2012-05-01 22:24:00 | 80640 | -1120685.39 | 6937173.26 | 3276490.99 | -6748.72 | -1837.44 | 1591.09 | 331.480 | -78.815 | 109.178 | 14.997 |
| 2012-05-01 22:25:00 | 80700 | -1523676.26 | 6816307.67 | 3366865.49 | -6680.86 | -2190.37 | 1420.62 | 326.183 | -77.582 | 104.037 | 11.873 |
| 2012-05-01 22:26:00 | 80760 | -1921977.96 | 6674464.48 | 3446878.27 | -6592.46 | -2536.52 | 1245.79 | 321.799 | -76.274 | 99.717 | 8.823 |
| 2012-05-01 22:27:00 | 80820 | -2314365.26 | 6512082.29 | 3516284.15 | -6483.77 | -2874.83 | 1067.15 | 318.124 | -74.908 | 96.063 | 5.888 |
| 2012-05-01 22:28:00 | 80880 | -2699631.48 | 6329662.94 | 3574870.70 | -6355.15 | -3204.25 | 885.24 | 315.001 | -73.500 | 92.948 | 3.080 |
| 2012-05-01 22:29:00 | 80940 | -3076592.11 | 6127769.94 | 3622458.93 | -6206.99 | -3523.78 | 700.63 | 312.310 | -72.058 | 90.267 | 0.398 |
| 2012-05-01 22:30:00 | 81000 | -3444088.57 | 5907026.70 | 3658903.77 | -6039.76 | -3832.43 | 513.89 | 309.961 | -70.590 | 87.938 | -2.165 |
| 2012-05-01 22:31:00 | 81060 | -3800991.71 | 5668114.59 | 3684094.54 | -5853.97 | -4129.26 | 325.59 | 307.883 | -69.101 | 85.896 | -4.622 |
| 2012-05-01 22:32:00 | 81120 | -4146205.32 | 5411770.76 | 3697955.26 | -5650.21 | -4413.34 | 136.32 | 306.024 | -67.594 | 84.091 | -6.984 |
| 2012-05-01 22:33:00 | 81180 | -4478669.49 | 5138785.87 | 3700444.86 | -5429.10 | -4683.82 | -53.35 | 304.344 | -66.072 | 82.481 | -9.261 |
| 2012-05-01 22:34:00 | 81240 | -4797363.86 | 4850001.59 | 3691557.33 | -5191.33 | -4939.85 | -242.82 | 302.809 | -64.538 | 81.034 | -11.465 |
| 2012-05-01 22:35:00 | 81300 | -5101310.79 | 4546308.01 | 3671321.66 | -4937.64 | -5180.67 | -431.52 | 301.394 | -62.993 | 79.724 | -13.604 |
| 2012-05-01 22:36:00 | 81360 | -5389578.28 | 4228640.81 | 3639801.77 | -4668.82 | -5405.52 | -618.87 | 300.079 | -61.437 | 78.529 | -15.687 |
| 2012-05-01 22:37:00 | 81420 | -5661282.89 | 3897978.39 | 3597096.27 | -4385.69 | -5613.73 | -804.28 | 298.846 | -59.873 | 77.432 | -17.719 |
| 2012-05-01 22:38:00 | 81480 | -5915592.36 | 3555338.81 | 3543338.14 | -4089.13 | -5804.66 | -987.19 | 297.682 | -58.300 | 76.419 | -19.708 |
| 2012-05-01 22:39:00 | 81540 | -6151728.24 | 3201776.65 | 3478694.31 | -3780.06 | -5977.72 | -1167.04 | 296.576 | -56.719 | 75.477 | -21.657 |
| 2012-05-01 22:40:00 | 81600 | -6368968.13 | 2838379.70 | 3403365.12 | -3459.43 | -6132.40 | -1343.28 | 295.517 | -55.130 | 74.596 | -23.572 |
| 2012-05-01 22:41:00 | 81660 | -6566647.97 | 2466265.62 | 3317583.68 | -3128.22 | -6268.22 | -1515.36 | 294.498 | -53.533 | 73.767 | -25.456 |
| 2012-05-01 22:42:00 | 81720 | -6744163.98 | 2086578.49 | 3221615.10 | -2787.47 | -6384.77 | -1682.77 | 293.511 | -51.928 | 72.984 | -27.313 |
| 2012-05-01 22:43:00 | 81780 | -6900974.52 | 1700485.25 | 3115755.73 | -2438.22 | -6481.71 | -1844.97 | 292.551 | -50.316 | 72.238 | -29.145 |
| 2012-05-01 22:44:00 | 81840 | -7036601.68 | 1309172.12 | 3000332.15 | -2081.54 | -6558.73 | -2001.49 | 291.611 | -48.695 | 71.525 | -30.955 |
| 2012-05-01 22:45:00 | 81900 | -7150632.70 | 913840.96 | 2875700.19 | -1718.53 | -6615.61 | -2151.84 | 290.687 | -47.066 | 70.840 | -32.745 |
| 2012-05-01 22:46:00 | 81960 | -7242721.22 | 515705.55 | 2742243.82 | -1350.31 | -6652.18 | -2295.56 | 289.774 | -45.429 | 70.177 | -34.517 |
| 2012-05-01 22:47:00 | 82020 | -7312588.29 | 115987.86 | 2600373.93 | -978.01 | -6668.34 | -2432.22 | 288.868 | -43.782 | 69.532 | -36.273 |
| 2012-05-01 22:48:00 | 82080 | -7360023.16 | -284085.67 | 2450527.09 | -602.76 | -6664.04 | -2561.39 | 287.965 | -42.125 | 68.901 | -38.014 |
| 2012-05-01 22:49:00 | 82140 | -7384883.90 | -683287.90 | 2293164.15 | -225.73 | -6639.30 | -2682.69 | 287.061 | -40.458 | 68.281 | -39.741 |
| 2012-05-01 22:50:00 | 82200 | -7387097.79 | -1080394.76 | 2128768.87 | 151.94 | -6594.21 | -2795.75 | 286.152 | -38.780 | 67.668 | -41.455 |
| 2012-05-01 22:51:00 | 82260 | -7366661.50 | -1474188.96 | 1957846.39 | 529.09 | -6528.91 | -2900.21 | 285.234 | -37.090 | 67.058 | -43.158 |
| 2012-05-01 22:52:00 | 82320 | -7323641.03 | -1863463.74 | 1780921.67 | 904.55 | -6443.60 | -2995.77 | 284.304 | -35.387 | 66.447 | -44.850 |
| 2012-05-01 22:53:00 | 82380 | -7258171.53 | -2247026.49 | 1598537.90 | 1277.20 | -6338.56 | -3082.13 | 283.357 | -33.669 | 65.832 | -46.532 |
| 2012-05-01 22:54:00 | 82440 | -7170456.76 | -2623702.46 | 1411254.84 | 1645.87 | -6214.10 | -3159.04 | 282.389 | -31.936 | 65.208 | -48.204 |
| 2012-05-01 22:55:00 | 82500 | -7060768.52 | -2992338.26 | 1219647.05 | 2009.46 | -6070.62 | -3226.25 | 281.396 | -30.185 | 64.573 | -49.868 |
| 2012-05-01 22:56:00 | 82560 | -6929445.71 | -3351805.41 | 1024302.18 | 2366.84 | -5908.56 | -3283.58 | 280.372 | -28.416 | 63.920 | -51.523 |
| 2012-05-01 22:57:00 | 82620 | -6776893.33 | -3701003.73 | 825819.18 | 2716.93 | -5728.42 | -3330.83 | 279.312 | -26.627 | 63.244 | -53.169 |
| 2012-05-01 22:58:00 | 82680 | -6603581.15 | -4038864.72 | 624806.42 | 3058.66 | -5530.75 | -3367.88 | 278.210 | -24.814 | 62.542 | -54.808 |
| 2012-05-01 22:59:00 | 82740 | -6410042.31 | -4364354.76 | 421879.87 | 3390.98 | -5316.16 | -3394.61 | 277.058 | -22.975 | 61.804 | -56.439 |
| 2012-05-01 23:00:00 | 82800 | -6196871.61 | -4676478.23 | 217661.24 | 3712.89 | -5085.31 | -3410.94 | 275.849 | -21.109 | 61.025 | -58.063 |
| 2012-05-01 23:01:00 | 82860 | -5964723.74 | -4974280.58 | 12776.03 | 4023.39 | -4838.91 | -3416.82 | 274.573 | -19.210 | 60.196 | -59.678 |
| 2012-05-01 23:02:00 | 82920 | -5714311.22 | -5256851.14 | -192148.30 | 4321.56 | -4577.71 | -3412.24 | 273.221 | -17.276 | 59.305 | -61.285 |
| 2012-05-01 23:03:00 | 82980 | -5446402.22 | -5523325.92 | -396484.31 | 4606.46 | -4302.52 | -3397.22 | 271.778 | -15.303 | 58.341 | -62.885 |
| 2012-05-01 23:04:00 | 83040 | -5161818.24 | -5772890.24 | -599606.44 | 4877.25 | -4014.17 | -3371.79 | 270.230 | -13.285 | 57.287 | -64.475 |
| 2012-05-01 23:05:00 | 83100 | -4861431.58 | -6004781.11 | -800892.89 | 5133.08 | -3713.55 | -3336.05 | 268.559 | -11.217 | 56.125 | -66.055 |
| 2012-05-01 23:06:00 | 83160 | -4546162.65 | -6218289.61 | -999727.59 | 5373.20 | -3401.58 | -3290.09 | 266.744 | -9.092 | 54.830 | -67.624 |
| 2012-05-01 23:07:00 | 83220 | -4216977.23 | -6412762.99 | -1195501.98 | 5596.85 | -3079.21 | -3234.06 | 264.758 | -6.906 | 53.371 | -69.181 |
| 2012-05-01 23:08:00 | 83280 | -3874883.47 | -6587606.66 | -1387616.89 | 5803.37 | -2747.43 | -3168.14 | 262.570 | -4.649 | 51.710 | -70.723 |
| 2012-05-01 23:09:00 | 83340 | -3520928.87 | -6742285.97 | -1575484.37 | 5992.11 | -2407.24 | -3092.52 | 260.140 | -2.316 | 49.793 | -72.247 |
| 2012-05-01 23:10:00 | 83400 | -3156197.07 | -6876327.84 | -1758529.46 | 6162.51 | -2059.69 | -3007.43 | 257.421 | 0.100 | 47.551 | -73.750 |
| 2012-05-01 23:11:00 | 83460 | -2781804.56 | -6989322.18 | -1936191.92 | 6314.05 | -1705.83 | -2913.14 | 254.354 | 2.602 | 44.891 | -75.224 |
| 2012-05-01 23:12:00 | 83520 | -2398897.32 | -7080923.13 | -2107927.96 | 6446.27 | -1346.75 | -2809.93 | 250.866 | 5.190 | 41.683 | -76.662 |
| 2012-05-01 23:13:00 | 83580 | -2008647.32 | -7150850.12 | -2273211.84 | 6558.75 | -983.55 | -2698.12 | 246.873 | 7.854 | 37.749 | -78.051 |
| 2012-05-01 23:14:00 | 83640 | -1612248.94 | -7198888.72 | -2431537.54 | 6651.16 | -617.33 | -2578.05 | 242.273 | 10.572 | 32.846 | -79.373 |
| 2012-05-01 23:15:00 | 83700 | -1210915.43 | -7224891.28 | -2582420.22 | 6723.21 | -249.20 | -2450.09 | 236.955 | 13.294 | 26.646 | -80.599 |
| 2012-05-01 23:16:00 | 83760 | -805875.14 | -7228777.40 | -2725397.74 | 6774.69 | 119.70 | -2314.61 | 230.810 | 15.939 | 18.746 | -81.688 |
| 2012-05-01 23:17:00 | 83820 | -398367.84 | -7210534.20 | -2860032.04 | 6805.42 | 488.25 | -2172.05 | 223.756 | 18.378 | 8.751 | -82.579 |
| 2012-05-01 23:18:00 | 83880 | 10359.02 | -7170216.34 | -2985910.50 | 6815.33 | 855.34 | -2022.83 | 215.781 | 20.439 | 356.537 | -83.195 |
| 2012-05-01 23:19:00 | 83940 | 419054.15 | -7107945.87 | -3102647.18 | 6804.37 | 1219.82 | -1867.40 | 206.994 | 21.919 | 342.669 | -83.458 |
| 2012-05-01 23:20:00 | 84000 | 826466.18 | -7023911.90 | -3209884.00 | 6772.57 | 1580.60 | -1706.25 | 197.660 | 22.637 | 328.527 | -83.325 |
| 2012-05-01 23:21:00 | 84060 | 1231347.48 | -6918370.05 | -3307291.85 | 6720.03 | 1936.57 | -1539.85 | 188.177 | 22.490 | 315.657 | -82.819 |
| 2012-05-01 23:22:00 | 84120 | 1632457.93 | -6791641.63 | -3394571.55 | 6646.91 | 2286.64 | -1368.73 | 178.983 | 21.501 | 304.918 | -82.009 |
| 2012-05-01 23:23:00 | 84180 | 2028568.70 | -6644112.78 | -3471454.86 | 6553.42 | 2629.74 | -1193.39 | 170.439 | 19.806 | 296.368 | -80.977 |
| 2012-05-01 23:24:00 | 84240 | 2418466.01 | -6476233.25 | -3537705.22 | 6439.84 | 2964.82 | -1014.39 | 162.756 | 17.599 | 289.666 | -79.788 |
| 2012-05-01 23:25:00 | 84300 | 2800954.82 | -6288515.08 | -3593118.52 | 6306.53 | 3290.86 | -832.25 | 155.996 | 15.077 | 284.395 | -78.492 |
| 2012-05-01 23:26:00 | 84360 | 3174862.44 | -6081531.08 | -3637523.74 | 6153.87 | 3606.85 | -647.54 | 150.119 | 12.400 | 280.199 | -77.120 |
| 2012-05-01 23:27:00 | 84420 | 3539042.16 | -5855913.10 | -3670783.48 | 5982.35 | 3911.83 | -460.83 | 145.028 | 9.678 | 276.807 | -75.692 |
| 2012-05-01 23:28:00 | 84480 | 3892376.74 | -5612350.14 | -3692794.38 | 5792.46 | 4204.87 | -272.68 | 140.612 | 6.980 | 274.021 | -74.224 |
| 2012-05-01 23:29:00 | 84540 | 4233781.80 | -5351586.28 | -3703487.48 | 5584.80 | 4485.05 | -83.66 | 136.762 | 4.347 | 271.698 | -72.725 |
| 2012-05-01 23:30:00 | 84600 | 4562209.19 | -5074418.43 | -3702828.42 | 5359.98 | 4751.52 | 105.63 | 133.383 | 1.795 | 269.733 | -71.202 |
| 2012-05-01 23:31:00 | 84660 | 4876650.19 | -4781693.94 | -3690817.59 | 5118.70 | 5003.47 | 294.63 | 130.394 | -0.669 | 268.051 | -69.659 |
| 2012-05-01 23:32:00 | 84720 | 5176138.58 | -4474308.05 | -3667490.14 | 4861.69 | 5240.11 | 482.76 | 127.726 | -3.047 | 266.593 | -68.100 |
| 2012-05-01 23:33:00 | 84780 | 5459753.68 | -4153201.17 | -3632915.88 | 4589.73 | 5460.72 | 669.43 | 125.325 | -5.343 | 265.316 | -66.526 |
| 2012-05-01 23:34:00 | 84840 | 5726623.14 | -3819356.02 | -3587199.10 | 4303.64 | 5664.61 | 854.08 | 123.147 | -7.564 | 264.188 | -64.941 |
| 2012-05-01 23:35:00 | 84900 | 5975925.66 | -3473794.69 | -3530478.27 | 4004.31 | 5851.16 | 1036.14 | 121.154 | -9.717 | 263.181 | -63.345 |
| 2012-05-01 23:36:00 | 84960 | 6206893.52 | -3117575.50 | -3462925.65 | 3692.64 | 6019.78 | 1215.05 | 119.318 | -11.809 | 262.277 | -61.738 |
| 2012-05-01 23:37:00 | 85020 | 6418814.99 | -2751789.80 | -3384746.76 | 3369.59 | 6169.96 | 1390.26 | 117.612 | -13.846 | 261.457 | -60.123 |
| 2012-05-01 23:38:00 | 85080 | 6611036.53 | -2377558.65 | -3296179.78 | 3036.15 | 6301.22 | 1561.23 | 116.017 | -15.833 | 260.709 | -58.498 |
| 2012-05-01 23:39:00 | 85140 | 6782964.84 | -1996029.38 | -3197494.88 | 2693.32 | 6413.17 | 1727.43 | 114.516 | -17.777 | 260.022 | -56.865 |
| 2012-05-01 23:40:00 | 85200 | 6934068.71 | -1608372.11 | -3088993.33 | 2342.17 | 6505.44 | 1888.36 | 113.095 | -19.682 | 259.387 | -55.223 |
| 2012-05-01 23:41:00 | 85260 | 7063880.70 | -1215776.19 | -2971006.67 | 1983.77 | 6577.75 | 2043.52 | 111.740 | -21.551 | 258.796 | -53.572 |
| 2012-05-01 23:42:00 | 85320 | 7171998.62 | -819446.50 | -2843895.69 | 1619.22 | 6629.86 | 2192.43 | 110.443 | -23.388 | 258.242 | -51.912 |
| 2012-05-01 23:43:00 | 85380 | 7258086.80 | -420599.81 | -2708049.33 | 1249.64 | 6661.62 | 2334.63 | 109.194 | -25.197 | 257.721 | -50.243 |
| 2012-05-01 23:44:00 | 85440 | 7321877.18 | -20461.01 | -2563883.50 | 876.15 | 6672.92 | 2469.67 | 107.984 | -26.980 | 257.228 | -48.564 |
| 2012-05-01 23:45:00 | 85500 | 7363170.19 | 379740.62 | -2411839.82 | 499.92 | 6663.72 | 2597.15 | 106.807 | -28.740 | 256.757 | -46.875 |
| 2012-05-01 23:46:00 | 85560 | 7381835.35 | 778775.24 | -2252384.31 | 122.09 | 6634.03 | 2716.67 | 105.658 | -30.478 | 256.306 | -45.175 |
| 2012-05-01 23:47:00 | 85620 | 7377811.78 | 1175416.19 | -2086005.92 | -256.18 | 6583.95 | 2827.86 | 104.529 | -32.197 | 255.871 | -43.464 |
| 2012-05-01 23:48:00 | 85680 | 7351108.43 | 1568443.81 | -1913215.03 | -633.71 | 6513.62 | 2930.37 | 103.415 | -33.897 | 255.449 | -41.740 |
| 2012-05-01 23:49:00 | 85740 | 7301804.03 | 1956649.22 | -1734541.97 | -1009.35 | 6423.25 | 3023.88 | 102.313 | -35.581 | 255.036 | -40.003 |
| 2012-05-01 23:50:00 | 85800 | 7230046.96 | 2338838.00 | -1550535.29 | -1381.94 | 6313.11 | 3108.11 | 101.216 | -37.249 | 254.631 | -38.251 |
| 2012-05-01 23:51:00 | 85860 | 7136054.80 | 2713833.94 | -1361760.15 | -1750.33 | 6183.55 | 3182.79 | 100.122 | -38.902 | 254.229 | -36.483 |
| 2012-05-01 23:52:00 | 85920 | 7020113.67 | 3080482.70 | -1168796.54 | -2113.39 | 6034.94 | 3247.69 | 99.024 | -40.542 | 253.829 | -34.698 |
| 2012-05-01 23:53:00 | 85980 | 6882577.46 | 3437655.33 | -972237.52 | -2469.98 | 5867.76 | 3302.60 | 97.919 | -42.169 | 253.428 | -32.894 |
| 2012-05-01 23:54:00 | 86040 | 6723866.67 | 3784251.82 | -772687.37 | -2819.02 | 5682.50 | 3347.36 | 96.803 | -43.784 | 253.023 | -31.069 |
| 2012-05-01 23:55:00 | 86100 | 6544467.21 | 4119204.53 | -570759.75 | -3159.43 | 5479.73 | 3381.83 | 95.670 | -45.386 | 252.610 | -29.220 |
| 2012-05-01 23:56:00 | 86160 | 6344928.91 | 4441481.46 | -367075.79 | -3490.15 | 5260.08 | 3405.89 | 94.516 | -46.977 | 252.187 | -27.344 |
| 2012-05-01 23:57:00 | 86220 | 6125863.83 | 4750089.52 | -162262.14 | -3810.15 | 5024.22 | 3419.48 | 93.335 | -48.556 | 251.750 | -25.439 |
| 2012-05-01 23:58:00 | 86280 | 5887944.39 | 5044077.60 | 43050.91 | -4118.46 | 4772.87 | 3422.54 | 92.121 | -50.125 | 251.295 | -23.501 |
| 2012-05-01 23:59:00 | 86340 | 5631901.31 | 5322539.53 | 248231.44 | -4414.12 | 4506.81 | 3415.06 | 90.869 | -51.682 | 250.818 | -21.525 |
| 2012-05-02 00:00:00 | 86400 | 5358521.39 | 5584616.90 | 452647.83 | -4696.21 | 4226.86 | 3397.07 | 89.572 | -53.227 | 250.313 | -19.506 |

At the first burn  (2012-05-01 19:57:30.117992), DGSA EL is between -57.692 deg and -59.358 deg and the VTSA EL is between -24.598 deg and -22.937 deg so NEITHER sensor has visibility of chaser
At the second burn (2012-05-01 20:54:18.219434), DGSA EL is between -26.995 deg and -25.085 deg and the VTSA EL is between -48.890 deg and -50.441 deg so NEITHER sensor has visibility of chaser


## Part 4: Eclipse Computations

> Use UTC time as input to sun computations.

19. How long is an eclipse for the **target orbit** and for the **initial chaser orbit**?
20. State whether each burn occurs in **sunlight or eclipse**. Show eclipse start/stop times relative to each burn.

In [83]:
#19 and 20
### eclipses:
class Eclipse:
    def __init__(self,Start,Midpoint,End,Duration):
        self.Start = Start
        self.Midpoint = Midpoint
        self.End = End
        self.Duration = Duration

eclipses_Target: list[Eclipse] = []
print("Estimated First 15 eclipses for Target:")
print(f"{'#':<4} {'Start':>18} {'Midpoint':>36} {'End':>19} {'Duration':>29}")
temp_start_time = EPOCH

for i in range(15):
    if i == 0:
        jDate = compute_j_date(temp_start_time.year, temp_start_time.month, temp_start_time.day, temp_start_time.hour, temp_start_time.minute, temp_start_time.second)
        SUN_Vector = compute_sun_vector(jDate)
        SUN_Vector = SUN_Vector/SUN_Vector.magnitude()
        h = compute_angular_momentum_vector(POS_TARGET, VEL_TARGET)
        h = h/h.magnitude()
        R_perifocal_ECI = np.array([
            [math.cos(KEP_TARGET.raan)*math.cos(KEP_TARGET.argp)-math.sin(KEP_TARGET.raan)*math.sin(KEP_TARGET.argp)*math.cos(KEP_TARGET.inc), -math.cos(KEP_TARGET.raan)*math.sin(KEP_TARGET.argp)-math.sin(KEP_TARGET.raan)*math.cos(KEP_TARGET.argp)*math.cos(KEP_TARGET.inc), math.sin(KEP_TARGET.raan)*math.sin(KEP_TARGET.inc)],
            [math.sin(KEP_TARGET.raan)*math.cos(KEP_TARGET.argp)+math.cos(KEP_TARGET.raan)*math.sin(KEP_TARGET.argp)*math.cos(KEP_TARGET.inc), -math.sin(KEP_TARGET.raan)*math.sin(KEP_TARGET.argp)+math.cos(KEP_TARGET.raan)*math.cos(KEP_TARGET.argp)*math.cos(KEP_TARGET.inc), -math.cos(KEP_TARGET.raan)*math.sin(KEP_TARGET.inc)],
            [math.sin(KEP_TARGET.inc)*math.sin(KEP_TARGET.argp), math.sin(KEP_TARGET.inc)*math.cos(KEP_TARGET.argp), math.cos(KEP_TARGET.inc)]
        ])
        Sun_perifocal = np.linalg.inv(R_perifocal_ECI) @ SUN_Vector.get_np_vector()
        noon_vector = Vector3(Sun_perifocal[0][0], Sun_perifocal[1][0], 0)
        midnight_vector = -1*noon_vector
        ta_noon_rad = atan2(noon_vector.y, noon_vector.x)
        ta_midnight_rad = pi+ta_noon_rad
        if ta_midnight_rad < 0: ta_midnight_rad = ta_midnight_rad+2*pi
        k = 0
        if KEP_TARGET.ta > ta_midnight_rad: k = 1
        Orbital_Period = compute_period(KEP_TARGET.a, KEP_TARGET.mu_earth)
        E_SV_rad = compute_eccentric_anomaly(KEP_TARGET.ta, KEP_TARGET.ecc)
        E_Midnight_rad = compute_eccentric_anomaly(ta_midnight_rad, KEP_TARGET.ecc)
        Orbital_Period = compute_period(KEP_TARGET.a, KEP_TARGET.mu_earth)
        Mean_Motion = compute_mean_motion(KEP_TARGET.mu_earth, KEP_TARGET.a)
        time_from_sv_to_midnight = k*Orbital_Period + 1/Mean_Motion*(E_Midnight_rad-KEP_TARGET.ecc*sin(E_Midnight_rad)) - 1/Mean_Motion*(E_SV_rad-KEP_TARGET.ecc*sin(E_SV_rad))
        midnight_time = temp_start_time+timedelta(seconds = time_from_sv_to_midnight)
        Earth_radius = 6378137 #m
        p = asin(Earth_radius / POS_TARGET.magnitude())
        e = 0
        midnight_jDate = compute_j_date(midnight_time.year, midnight_time.month, midnight_time.day, midnight_time.hour, midnight_time.minute, midnight_time.second)
        SUN_Vector_midnight = compute_sun_vector(midnight_jDate)
        S_hat_midnight = SUN_Vector_midnight / SUN_Vector_midnight.magnitude()
        beta = asin(S_hat_midnight.dot(h))
        if beta < p: e = e*acos(cos(p)/cos(beta))
        umbral_eclipse_duration = Orbital_Period/pi * acos(cos(p)/cos(beta))
        entry = (midnight_time - timedelta(seconds=umbral_eclipse_duration)/2)
        exit = (midnight_time + timedelta(seconds=umbral_eclipse_duration)/2)
    else:
        p = asin(Earth_radius / POS_TARGET.magnitude())
        midnight_time = midnight_time + timedelta(seconds=Orbital_Period)
        midnight_jDate = compute_j_date(midnight_time.year, midnight_time.month, midnight_time.day, midnight_time.hour, midnight_time.minute, midnight_time.second)
        SUN_Vector_midnight = compute_sun_vector(midnight_jDate)
        S_hat_midnight = SUN_Vector_midnight / SUN_Vector_midnight.magnitude()
        beta = asin(S_hat_midnight.dot(h))
        if beta < p: e = e*acos(cos(p)/cos(beta))
        umbral_eclipse_duration = Orbital_Period/pi * acos(cos(p)/cos(beta))
        entry = midnight_time - timedelta(seconds=umbral_eclipse_duration/2)
        exit = midnight_time + timedelta(seconds=umbral_eclipse_duration/2)

    eclipses_Target.append(Eclipse(entry, midnight_time, exit, umbral_eclipse_duration))
    if i == 11: print(f"{i+1:<4} [BURN 1] {entry.strftime('%d-%b-%Y %H:%M:%S'):>24} [BURN 2] {midnight_time.strftime('%d-%b-%Y %H:%M:%S'):>24} {exit.strftime('%d-%b-%Y %H:%M:%S'):>24} {umbral_eclipse_duration/60:>10.3f} min")
    else: print(f"{i+1:<4} {entry.strftime('%d-%b-%Y %H:%M:%S'):>33} {midnight_time.strftime('%d-%b-%Y %H:%M:%S'):>33} {exit.strftime('%d-%b-%Y %H:%M:%S'):>24} {umbral_eclipse_duration/60:>10.3f} min")


eclipses_Chaser: list[Eclipse] = []
print("\nEstimated First 15 eclipses for Chaser:")
print(f"{'#':<4} {'Start':>18} {'Midpoint':>36} {'End':>19} {'Duration':>29}")
temp_start_time = EPOCH

for i in range(15):
    if i == 0:
        jDate = compute_j_date(temp_start_time.year, temp_start_time.month, temp_start_time.day, temp_start_time.hour, temp_start_time.minute, temp_start_time.second)
        SUN_Vector = compute_sun_vector(jDate)
        SUN_Vector = SUN_Vector/SUN_Vector.magnitude()
        h = compute_angular_momentum_vector(POS_CHASER, VEL_CHASER)
        h = h/h.magnitude()
        R_perifocal_ECI = np.array([
            [math.cos(KEP_CHASER.raan)*math.cos(KEP_CHASER.argp)-math.sin(KEP_CHASER.raan)*math.sin(KEP_CHASER.argp)*math.cos(KEP_CHASER.inc), -math.cos(KEP_CHASER.raan)*math.sin(KEP_CHASER.argp)-math.sin(KEP_CHASER.raan)*math.cos(KEP_CHASER.argp)*math.cos(KEP_CHASER.inc), math.sin(KEP_CHASER.raan)*math.sin(KEP_CHASER.inc)],
            [math.sin(KEP_CHASER.raan)*math.cos(KEP_CHASER.argp)+math.cos(KEP_CHASER.raan)*math.sin(KEP_CHASER.argp)*math.cos(KEP_CHASER.inc), -math.sin(KEP_CHASER.raan)*math.sin(KEP_CHASER.argp)+math.cos(KEP_CHASER.raan)*math.cos(KEP_CHASER.argp)*math.cos(KEP_CHASER.inc), -math.cos(KEP_CHASER.raan)*math.sin(KEP_CHASER.inc)],
            [math.sin(KEP_CHASER.inc)*math.sin(KEP_CHASER.argp), math.sin(KEP_CHASER.inc)*math.cos(KEP_CHASER.argp), math.cos(KEP_CHASER.inc)]
        ])
        Sun_perifocal = np.linalg.inv(R_perifocal_ECI) @ SUN_Vector.get_np_vector()
        noon_vector = Vector3(Sun_perifocal[0][0], Sun_perifocal[1][0], 0)
        midnight_vector = -1*noon_vector
        ta_noon_rad = atan2(noon_vector.y, noon_vector.x)
        ta_midnight_rad = pi+ta_noon_rad
        if ta_midnight_rad < 0: ta_midnight_rad = ta_midnight_rad+2*pi
        k = 0
        if KEP_CHASER.ta > ta_midnight_rad: k = 1
        Orbital_Period = compute_period(KEP_CHASER.a, KEP_CHASER.mu_earth)
        E_SV_rad = compute_eccentric_anomaly(KEP_CHASER.ta, KEP_CHASER.ecc)
        E_Midnight_rad = compute_eccentric_anomaly(ta_midnight_rad, KEP_CHASER.ecc)
        Orbital_Period = compute_period(KEP_CHASER.a, KEP_CHASER.mu_earth)
        Mean_Motion = compute_mean_motion(KEP_CHASER.mu_earth, KEP_CHASER.a)
        time_from_sv_to_midnight = k*Orbital_Period + 1/Mean_Motion*(E_Midnight_rad-KEP_CHASER.ecc*sin(E_Midnight_rad)) - 1/Mean_Motion*(E_SV_rad-KEP_CHASER.ecc*sin(E_SV_rad))
        midnight_time = temp_start_time+timedelta(seconds = time_from_sv_to_midnight)
        Earth_radius = 6378137 #m
        p = asin(Earth_radius / POS_CHASER.magnitude())
        e = 0
        midnight_jDate = compute_j_date(midnight_time.year, midnight_time.month, midnight_time.day, midnight_time.hour, midnight_time.minute, midnight_time.second)
        SUN_Vector_midnight = compute_sun_vector(midnight_jDate)
        S_hat_midnight = SUN_Vector_midnight / SUN_Vector_midnight.magnitude()
        beta = asin(S_hat_midnight.dot(h))
        if beta < p: e = e*acos(cos(p)/cos(beta))
        umbral_eclipse_duration = Orbital_Period/pi * acos(cos(p)/cos(beta))
        entry = (midnight_time - timedelta(seconds=umbral_eclipse_duration)/2)
        exit = (midnight_time + timedelta(seconds=umbral_eclipse_duration)/2)
    else:
        p = asin(Earth_radius / POS_CHASER.magnitude())
        midnight_time = midnight_time + timedelta(seconds=Orbital_Period)
        midnight_jDate = compute_j_date(midnight_time.year, midnight_time.month, midnight_time.day, midnight_time.hour, midnight_time.minute, midnight_time.second)
        SUN_Vector_midnight = compute_sun_vector(midnight_jDate)
        S_hat_midnight = SUN_Vector_midnight / SUN_Vector_midnight.magnitude()
        beta = asin(S_hat_midnight.dot(h))
        if beta < p: e = e*acos(cos(p)/cos(beta))
        umbral_eclipse_duration = Orbital_Period/pi * acos(cos(p)/cos(beta))
        entry = midnight_time - timedelta(seconds=umbral_eclipse_duration/2)
        exit = midnight_time + timedelta(seconds=umbral_eclipse_duration/2)
    eclipses_Chaser.append(Eclipse(entry, midnight_time, exit, umbral_eclipse_duration))
    if i == 11: print(f"{i+1:<4} [BURN 1] {entry.strftime('%d-%b-%Y %H:%M:%S'):>24} [BURN 2] {midnight_time.strftime('%d-%b-%Y %H:%M:%S'):>24} {exit.strftime('%d-%b-%Y %H:%M:%S'):>24} {umbral_eclipse_duration/60:>10.3f} min")
    else: print(f"{i+1:<4} {entry.strftime('%d-%b-%Y %H:%M:%S'):>33} {midnight_time.strftime('%d-%b-%Y %H:%M:%S'):>33} {exit.strftime('%d-%b-%Y %H:%M:%S'):>24} {umbral_eclipse_duration/60:>10.3f} min")

print(f"\nEclipse duration for target is about 34.039 minutes or {34.039*60:.2f} seconds")
print(f"Eclipse duration for chaser is about 34.062 minutes or {34.062*60:.2f} seconds\n")

# Burn 1 time: 2012-05-01 19:57:30.117992
# Burn 2 time: 2012-05-01 20:54:18.219434

print("Burn 1 occurs in the sunlight")
print("Burn 2 occurs during an eclipse")

Estimated First 15 eclipses for Target:
#                 Start                             Midpoint                 End                      Duration
1                 01-May-2012 00:00:42              01-May-2012 00:17:44     01-May-2012 00:34:45     34.039 min
2                 01-May-2012 01:54:32              01-May-2012 02:11:33     01-May-2012 02:28:34     34.041 min
3                 01-May-2012 03:48:21              01-May-2012 04:05:22     01-May-2012 04:22:24     34.043 min
4                 01-May-2012 05:42:10              01-May-2012 05:59:12     01-May-2012 06:16:13     34.045 min
5                 01-May-2012 07:36:00              01-May-2012 07:53:01     01-May-2012 08:10:03     34.046 min
6                 01-May-2012 09:29:49              01-May-2012 09:46:50     01-May-2012 10:03:52     34.048 min
7                 01-May-2012 11:23:38              01-May-2012 11:40:40     01-May-2012 11:57:41     34.050 min
8                 01-May-2012 13:17:28              01-May